<a href="https://colab.research.google.com/github/tommbk50-hub/Cheminformatics/blob/main/Retrosynthesis_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AiZynthFinder + Gemini: Interactive Retrosynthesis & Medicinal Chemistry Assistant

This notebook combines **AiZynthFinder** (an AI-driven retrosynthesis engine using USPTO reaction templates and ZINC commercial stock) with **RDKit** cheminformatics heuristics and **Google Gemini** to plan, score, and analyze synthetic routes for target molecules.

### How It Works
1. **Setup & Engine Initialization (Cells 1–3):** Installs dependencies, downloads pretrained USPTO neural network models and ZINC stock data, defines medicinal chemistry SMARTS rules and Gemini classification helpers, and initializes the AiZynthFinder engine.
2. **Target Selection & Constraints (Cell 4):** Lets you input a target molecule by name, ID (PubChem, DrugBank, ChEMBL, InChIKey, CAS), SMILES, or by drawing it in an interactive JSME sketcher. You can also lock specific substructures/atoms and upload an in-house lab inventory of building blocks.
3. **Search Configuration & Execution (Cells 5A–5B):** Configures the tree search algorithm (`Retro*` or `MCTS`), ranking profile (`balanced`, `medchem_fast`, `process_scaleup`, `green_safe`), scale, and reaction exclusions. Running the search generates routes, classifies each step with Gemini + RDKit (conditions, safety/green scores, selectivity warnings, protecting groups, stereochemistry), and renders interactive 2D SVG route cards.
4. **Interactive AI Analysis (Cell 6):** Feeds the top retrosynthesis routes into a Gemini chat interface equipped with Python tools to re-run searches with new constraints, generate ELN stoichiometry protocols, and evaluate routes.

In [ ]:
# @title 1. Install Dependencies & Download Models
import os

# Install all Python dependencies in a single pip invocation
!pip install -q rdkit onnxruntime tables scipy pydantic deprecated jinja2 networkx pillow requests tqdm pyyaml rdchiral paretoset marshmallow jedi Reaction-Utils google-genai markdown
!pip install -q --ignore-requires-python --no-deps aizynthfinder

# Download USPTO models and ZINC stock only if not already present
if not os.path.exists("data/config.yml"):
    !mkdir -p data && download_public_data data
else:
    print("✅ Models and ZINC stock already present in ./data")

# Verify Install
!pip show aizynthfinder

In [ ]:
# @title 2. Chemistry Workbench: Ketcher Sketcher + Search Config + AiZynthFinder + ORD Parquet Validator + Organic Chemist AI Agent
# ==============================================================================
# SECTION 1: MEMORY ARENA RESTRICTIONS, IMPORTS, LOGGING & GEMINI CLIENT SETUP
# ==============================================================================
import os
import subprocess
import sys

# [Improvement 2] Restrict thread pools & ONNX memory arena growth BEFORE importing onnxruntime/aizynthfinder
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["VECLIB_MAXIMUM_THREADS"] = "2"
os.environ["NUMEXPR_NUM_THREADS"] = "2"
os.environ["ORT_DISABLE_MEMORY_ARENA"] = "1"

import re
import gc
import glob
import json
import html
import time
import ctypes
import socket
import base64
import logging
import zipfile
import threading
import functools
import http.server
import urllib.parse
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Optional

import numpy as np
import pandas as pd
import requests
from requests.adapters import HTTPAdapter
import IPython
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from pydantic import BaseModel
from google.colab import output, userdata
from google import genai
from google.genai import types

# Ensure duckdb and huggingface_hub are available for ORD Parquet shard indexing
try:
    import duckdb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb"])
    import duckdb

try:
    from huggingface_hub import hf_hub_download, list_repo_files
    _HAS_HF_HUB = True
except ImportError:
    _HAS_HF_HUB = False

try:
    from ord_schema import datasets as ord_datasets, message_helpers as ord_message_helpers
    from ord_schema.proto import reaction_pb2 as ord_reaction_pb2
    _HAS_ORD_SCHEMA = True
except ImportError:
    _HAS_ORD_SCHEMA = False

import onnxruntime as ort
from rdkit import Chem, DataStructs, RDLogger
from rdkit.Chem import AllChem, Descriptors, rdChemReactions, rdMolDescriptors
from rdkit.Chem.Draw import rdMolDraw2D

# Patch onnxruntime.InferenceSession once so AiZynthFinder models do not pre-allocate huge arenas
if not getattr(ort.InferenceSession, "_chemforge_arena_patched", False):
    _orig_ort_init = ort.InferenceSession.__init__

    def _low_mem_ort_init(self, path_or_bytes, sess_options=None, providers=None, **kwargs):
        if sess_options is None:
            sess_options = ort.SessionOptions()
        sess_options.enable_cpu_mem_arena = False
        sess_options.enable_mem_pattern = False
        sess_options.intra_op_num_threads = 2
        sess_options.inter_op_num_threads = 1
        return _orig_ort_init(self, path_or_bytes, sess_options=sess_options, providers=providers, **kwargs)

    ort.InferenceSession.__init__ = _low_mem_ort_init
    ort.InferenceSession._chemforge_arena_patched = True

# Silence verbose RDKit and AiZynthFinder logs
RDLogger.DisableLog("rdApp.*")
logging.getLogger("aizynthfinder").setLevel(logging.ERROR)

_GEMINI_CLIENT = None
GEMINI_STEP_CACHE = {}
_CHEM_SVG_CARDS = set()
MAX_STEP_CACHE_SIZE = 250

# Global Model Catalog & Thinking Configuration (as of 17 Nov 2026)
GEMINI_MODEL_CATALOG = {
    "gemini-3.8-flash": {
        "label": "Gemini 3.8 Flash (Default · Fast Workhorse)",
        "family": "gemini-3",
        "default_thinking": "medium",
        "supported_thinking": ["low", "medium", "high"],
        "summary": "Default high-speed Flash model for rapid retrosynthesis step classification, experimental protocol drafting, and interactive Q&A ($0.75/$3.75 per 1M tokens).",
    },
    "gemini-4-hydrogen": {
        "label": "Gemini 4 Hydrogen (Fast Frontier Agentic · Nov 2026)",
        "family": "gemini-4",
        "default_thinking": "high",
        "supported_thinking": ["low", "medium", "high", "xhigh"],
        "summary": "Latest fast frontier model (2.5x faster than 3.8 Flash); excels at stoichiometry tables, multi-tool agent workflows, and reactant-specific procedures ($1.50/$7.50 per 1M).",
    },
    "gemini-4-argon": {
        "label": "Gemini 4 Argon (Advanced Long-Horizon Reasoning)",
        "family": "gemini-4",
        "default_thinking": "medium",
        "supported_thinking": ["low", "medium", "high"],
        "summary": "Frontier long-horizon reasoning model for multi-step synthetic route planning, chemoselectivity risk analysis, and orthogonal protecting-group strategy.",
    },
    "gemini-4-carbon": {
        "label": "Gemini 4 Carbon (Maximum Frontier Scientific Reasoning)",
        "family": "gemini-4",
        "default_thinking": "medium",
        "supported_thinking": ["low", "medium", "high"],
        "summary": "Google's strongest frontier model; allocates deep reasoning tokens to test mechanistic hypotheses, stereochemical edge cases, and process scale-up audits.",
    },
    "gemini-3.6-flash": {
        "label": "Gemini 3.6 Flash (Balanced Speed & Structured Output)",
        "family": "gemini-3",
        "default_thinking": "medium",
        "supported_thinking": ["low", "medium", "high"],
        "summary": "Balanced Gemini 3 Flash model tuned for reliable JSON schema extraction, rapid functional-group classification, and low-latency chat.",
    },
    "gemini-3.5-pro": {
        "label": "Gemini 3.5 Pro (Deep Literature & Analytical Pro)",
        "family": "gemini-3",
        "default_thinking": "medium",
        "supported_thinking": ["low", "medium", "high"],
        "summary": "High-depth Pro model ideal for comparing competing synthetic topologies, predicting impurity profiles, and interpreting LC-MS/NMR reports.",
    },
    "gemini-3.5-flash-lite": {
        "label": "Gemini 3.5 Flash-Lite (Ultra-Fast / Lowest Cost)",
        "family": "gemini-3",
        "default_thinking": "low",
        "supported_thinking": ["minimal", "low", "medium", "high"],
        "summary": "Fastest, lowest-cost model designed for sub-second high-volume reaction classification and lightweight structural lookups.",
    },
    "gemini-3.1-pro-preview": {
        "label": "Gemini 3.1 Pro (1M Context & Grounded Literature Search)",
        "family": "gemini-3",
        "default_thinking": "medium",
        "supported_thinking": ["low", "medium", "high"],
        "summary": "1M-token context Pro model with expanded medium/high thinking levels and strong Google Search grounding for precedent conditions.",
    },
    "gemini-2.5-pro": {
        "label": "Gemini 2.5 Pro (Legacy Pro · Token Budget Thinking)",
        "family": "gemini-2.5",
        "default_thinking": "medium",
        "supported_thinking": ["low", "medium", "high"],
        "summary": "Legacy 2.5 Pro model using token-based thinking_budget (1k–16k tokens) for backward compatibility.",
    },
    "gemini-2.5-flash": {
        "label": "Gemini 2.5 Flash (Legacy Flash · Token Budget Thinking)",
        "family": "gemini-2.5",
        "default_thinking": "medium",
        "supported_thinking": ["minimal", "low", "medium", "high"],
        "summary": "Legacy 2.5 Flash model using token-based thinking_budget (0–16k tokens) for fast, low-cost fallback execution.",
    },
}

ACTIVE_GENAI_SETTINGS = {
    "model": "gemini-3.8-flash",
    "thinking_level": "medium",
}


def set_active_genai_settings(model_name: Optional[str] = None, thinking_level: Optional[str] = None) -> dict:
    """Validates and updates the global model and thinking level used across all workbench tabs."""
    if model_name and model_name in GEMINI_MODEL_CATALOG:
        ACTIVE_GENAI_SETTINGS["model"] = model_name
    spec = GEMINI_MODEL_CATALOG.get(ACTIVE_GENAI_SETTINGS["model"], GEMINI_MODEL_CATALOG["gemini-3.8-flash"])
    supported = spec["supported_thinking"]
    lvl = (thinking_level or ACTIVE_GENAI_SETTINGS.get("thinking_level") or spec["default_thinking"]).lower()
    if lvl not in supported:
        if lvl == "xhigh" and "high" in supported:
            lvl = "high"
        elif lvl == "minimal" and "low" in supported:
            lvl = "low"
        else:
            lvl = spec["default_thinking"]
    ACTIVE_GENAI_SETTINGS["thinking_level"] = lvl
    return {
        "model": ACTIVE_GENAI_SETTINGS["model"],
        "thinking_level": ACTIVE_GENAI_SETTINGS["thinking_level"],
        "label": spec["label"],
        "summary": spec["summary"],
        "supported_thinking": supported,
    }


def _build_genai_config(
    model_name: str,
    thinking_level: str,
    temperature: float = 0.2,
    response_mime_type: Optional[str] = None,
    response_schema=None,
    system_instruction: Optional[str] = None,
    tools=None,
    include_thinking: bool = True,
) -> types.GenerateContentConfig:
    """
    Builds a types.GenerateContentConfig that maps thinking_level to ThinkingConfig(thinking_level=...)
    for Gemini 3/4 models or ThinkingConfig(thinking_budget=...) for legacy Gemini 2.5 models.
    """
    cfg_kwargs = {"temperature": temperature}
    if response_mime_type:
        cfg_kwargs["response_mime_type"] = response_mime_type
    if response_schema is not None:
        cfg_kwargs["response_schema"] = response_schema
    if system_instruction:
        cfg_kwargs["system_instruction"] = system_instruction
    if tools is not None:
        cfg_kwargs["tools"] = tools

    if include_thinking:
        spec = GEMINI_MODEL_CATALOG.get(model_name, GEMINI_MODEL_CATALOG["gemini-3.8-flash"])
        lvl = (thinking_level or spec["default_thinking"]).lower()
        try:
            if spec["family"] == "gemini-2.5":
                budget_map = {"minimal": 0 if "flash" in model_name else 512, "low": 1024, "medium": 4096, "high": 16384, "xhigh": 24576}
                cfg_kwargs["thinking_config"] = types.ThinkingConfig(thinking_budget=budget_map.get(lvl, 4096))
            else:
                valid_lvls = spec["supported_thinking"]
                if lvl not in valid_lvls:
                    lvl = "high" if lvl == "xhigh" else ("low" if lvl == "minimal" else spec["default_thinking"])
                cfg_kwargs["thinking_config"] = types.ThinkingConfig(thinking_level=lvl)
        except Exception:
            pass

    return types.GenerateContentConfig(**cfg_kwargs)


def call_gemini_with_config(
    contents: str,
    temperature: float = 0.2,
    response_mime_type: Optional[str] = None,
    response_schema=None,
    system_instruction: Optional[str] = None,
    tools=None,
):
    """
    Executes client.models.generate_content using the active model and thinking level,
    with automatic fallback if a preview/frontier model or thinking_config is unavailable.
    """
    client = get_gemini_client()
    if client is None:
        raise RuntimeError("KDD_GEMINI_API_KEY is not configured.")

    primary_model = ACTIVE_GENAI_SETTINGS.get("model", "gemini-3.8-flash")
    thinking_lvl = ACTIVE_GENAI_SETTINGS.get("thinking_level", "medium")
    candidate_models = list(dict.fromkeys([primary_model, "gemini-3.8-flash", "gemini-2.5-flash"]))

    last_err = None
    for m_name in candidate_models:
        for use_think in (True, False):
            try:
                cfg = _build_genai_config(
                    model_name=m_name,
                    thinking_level=thinking_lvl,
                    temperature=temperature,
                    response_mime_type=response_mime_type,
                    response_schema=response_schema,
                    system_instruction=system_instruction,
                    tools=tools,
                    include_thinking=use_think,
                )
                resp = client.models.generate_content(model=m_name, contents=contents, config=cfg)
                return resp, m_name, thinking_lvl
            except Exception as e:
                last_err = e
                continue
    raise RuntimeError(f"All Gemini model attempts failed. Last error: {last_err}")


def get_process_ram_mb() -> float:
    """Reads current process Resident Set Size (VmRSS) in MB from Linux /proc/self/status."""
    try:
        with open("/proc/self/status", "r") as f:
            for line in f:
                if line.startswith("VmRSS:"):
                    kb = int(line.split()[1])
                    return round(kb / 1024.0, 1)
    except Exception:
        pass
    return 0.0


def get_gemini_client():
    """Returns a cached google.genai Client using KDD_GEMINI_API_KEY."""
    global _GEMINI_CLIENT
    if _GEMINI_CLIENT is not None:
        return _GEMINI_CLIENT
    api_key = None
    try:
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")
    if api_key:
        _GEMINI_CLIENT = genai.Client(api_key=api_key)
    return _GEMINI_CLIENT


# ==============================================================================
# SECTION 2: CORE CHEMINFORMATICS, SMARTS RULES & RDKIT UTILITIES
# ==============================================================================
MEDCHEM_SMARTS = {
    "primary_amine": Chem.MolFromSmarts("[NX3;H2;!$(NC=O)]"),
    "secondary_amine": Chem.MolFromSmarts("[NX3;H1;!$(NC=O)]"),
    "free_phenol_or_alcohol": Chem.MolFromSmarts("[OX2H;!$(OC=O)]"),
    "carboxylic_acid": Chem.MolFromSmarts("[CX3](=O)[OX2H1]"),
    "ester": Chem.MolFromSmarts("[CX3](=O)[OX2][#6]"),
    "aldehyde": Chem.MolFromSmarts("[CX3H1](=O)[#6]"),
    "ketone": Chem.MolFromSmarts("[#6][CX3](=O)[#6]"),
    "aryl_halide": Chem.MolFromSmarts("c[F,Cl,Br,I]"),
    "boronic_acid_or_ester": Chem.MolFromSmarts("[#6][BX3]([OX2])[OX2]"),
    "nitro": Chem.MolFromSmarts("[$([NX3](=O)=O),$([NX3+](=O)[O-])]"),
    "nitrile": Chem.MolFromSmarts("[NX1]#[CX2]"),
    "terminal_alkyne": Chem.MolFromSmarts("[CX2]#[CX2H1]"),
    "azide": Chem.MolFromSmarts("[NX2]=[NX2+]=[NX1-]"),
}

RDKIT_REACTION_RULES = [
    ("Suzuki-Miyaura Cross-Coupling", ("aryl_halide", "boronic_acid_or_ester"), "Pd(dppf)Cl2 (5 mol%), K2CO3 (2.5 eq), 1,4-dioxane/H2O (4:1), 90 °C", 4, "CPME/H2O or i-PrOH/H2O"),
    ("Amide Bond Coupling", ("carboxylic_acid", "primary_amine"), "HATU (1.2 eq), DIPEA (3.0 eq), DMF, rt, 12 h", 3, "EtOAc or DMC instead of DMF"),
    ("Amide Bond Coupling", ("carboxylic_acid", "secondary_amine"), "HATU (1.2 eq), DIPEA (3.0 eq), DMF, rt, 12 h", 3, "EtOAc or MeCN instead of DMF"),
    ("Buchwald-Hartwig / SNAr Amination", ("aryl_halide", "primary_amine"), "Pd2(dba)3 (4 mol%), Xantphos (8 mol%), Cs2CO3 (2.0 eq), 1,4-dioxane, 100 °C", 3, "t-AmOH or 2-MeTHF"),
    ("Buchwald-Hartwig / SNAr Amination", ("aryl_halide", "secondary_amine"), "Pd2(dba)3 (4 mol%), RuPhos (8 mol%), NaOtBu (2.0 eq), toluene, 100 °C", 3, "CPME or t-AmOH"),
    ("Reductive Amination", ("aldehyde", "primary_amine"), "NaBH(OAc)3 (1.5 eq), AcOH (cat.), DCE or THF, rt", 3, "2-MeTHF or EtOAc instead of DCE"),
    ("CuAAC 'Click' Cycloaddition", ("terminal_alkyne", "azide"), "CuSO4·5H2O (5 mol%), Na-ascorbate (15 mol%), t-BuOH/H2O (1:1), rt", 4, "H2O/EtOH"),
]


def _extract_primary_smiles(smiles_or_rxn: str) -> str:
    """Safely extracts the primary molecule SMILES even if a dangling '>>' arrow is present."""
    s = (smiles_or_rxn or "").strip()
    if not s:
        return ""
    if ">>" in s:
        left, right = [p.strip() for p in s.split(">>", 1)]
        target_part = right if right else left
        return target_part.split(".")[0].strip()
    return s.split(".")[0].strip()


# ==============================================================================
# SECTION 2B: ORD PARQUET SHARD INDEXER, LIVE ONLINE PREFETCHER & VALIDATOR
# ==============================================================================
_POPCOUNT_LUT = np.array([bin(i).count("1") for i in range(256)], dtype=np.uint32)
ORD_DB_PATH = "/dev/shm/ord_precedents.duckdb" if os.path.isdir("/dev/shm") else "data/ord_precedents.duckdb"
ORD_ONLINE_API_BASE = "https://open-reaction-database.org/api"

# Persistent pooled HTTP session for multi-threaded online ORD API requests
_ORD_HTTP_SESSION = requests.Session()
_ord_adapter = HTTPAdapter(pool_connections=32, pool_maxsize=32, max_retries=1)
_ORD_HTTP_SESSION.mount("https://", _ord_adapter)
_ORD_HTTP_SESSION.mount("http://", _ord_adapter)
_ORD_HTTP_SESSION.headers.update({
    "User-Agent": "ChemForge-Colab-ParallelValidator/2.5",
    "Accept": "application/json",
})

# Curated ORD benchmark & HTE seed records (including low-yield/0% negative data and recorded byproducts)
_ORD_CURATED_SEED_RECORDS = [
    {
        "ord_id": "ord-9f4b21a08c114e2a91b2",
        "rxn_smiles": "Brc1ccc(C(=O)O)cc1.OB(O)c1ccccc1>>O=C(O)c1ccc(-c2ccccc2)cc1",
        "yield_pct": 88.0,
        "catalysts": "Pd(dppf)Cl2 (0.05 eq)",
        "reagents": "K2CO3 (2.5 eq)",
        "solvents": "1,4-dioxane; water (4:1)",
        "temp_c": 90.0,
        "atmosphere": "NITROGEN",
        "undesired_byproducts": "c1ccccc1 (protodeboronation, 4%)",
        "reference": "DOI: 10.1021/acs.orglett.1c02410",
    },
    {
        "ord_id": "ord-7e3c90b12a444d8f82a1",
        "rxn_smiles": "Brc1cccc(C(F)(F)F)c1.OB(O)c1cccnc1>>FC(F)(F)c1cccc(-c2cccnc2)c1",
        "yield_pct": 82.5,
        "catalysts": "Pd(PPh3)4 (0.03 eq)",
        "reagents": "Na2CO3 (2.0 eq)",
        "solvents": "DME; water",
        "temp_c": 85.0,
        "atmosphere": "ARGON",
        "undesired_byproducts": "c1cccnc1 (3-pyridyl protodeboronation, 8%)",
        "reference": "DOI: 10.1021/jm501122a",
    },
    {
        "ord_id": "ord-2a1d88f09c314b1e77c9",
        "rxn_smiles": "Clc1nccc(-c2cccnc2)n1.OB(O)c1ccc(C)cc1>>Cc1ccc(-c2cc(-c3cccnc3)ncn2)cc1",
        "yield_pct": 11.0,
        "catalysts": "Pd(OAc)2 (0.05 eq); PPh3 (0.10 eq)",
        "reagents": "Et3N (2.0 eq)",
        "solvents": "DMF",
        "temp_c": 100.0,
        "atmosphere": "AIR",
        "undesired_byproducts": "Cc1ccccc1 (homocoupling / protodeboronation, 34%)",
        "reference": "DOI: 10.1126/science.aar5169 (HTE Negative Well)",
    },
    {
        "ord_id": "ord-6b8e11c45d904a3f90d4",
        "rxn_smiles": "O=C(O)c1ccc(C)c(Nc2nccc(-c3cccnc3)n2)c1.Nc1cc(C(F)(F)F)cc(-n2cc(C)nc2)c1>>Cc1cn(-c2cc(NC(=O)c3ccc(C)c(Nc4nccc(-c5cccnc5)n4)c3)cc(C(F)(F)F)c2)cn1",
        "yield_pct": 84.0,
        "catalysts": "None",
        "reagents": "HATU (1.2 eq); DIPEA (3.0 eq)",
        "solvents": "DMF",
        "temp_c": 25.0,
        "atmosphere": "NITROGEN",
        "undesired_byproducts": "Guanidinium N-tetramethyluronium adduct (2%)",
        "reference": "DOI: 10.1021/op800142h (Nilotinib Amide Coupling)",
    },
    {
        "ord_id": "ord-4c2f77e19b024c5d81e3",
        "rxn_smiles": "COC(=O)c1ccc(C)c(Nc2nccc(-c3cccnc3)n2)c1.Nc1cc(C(F)(F)F)cc(-n2cc(C)nc2)c1>>Cc1cn(-c2cc(NC(=O)c3ccc(C)c(Nc4nccc(-c5cccnc5)n4)c3)cc(C(F)(F)F)c2)cn1",
        "yield_pct": 79.0,
        "catalysts": "None",
        "reagents": "KOtBu (2.2 eq)",
        "solvents": "THF",
        "temp_c": 35.0,
        "atmosphere": "NITROGEN",
        "undesired_byproducts": "O=C(O)c1ccc(C)c(Nc2nccc(-c3cccnc3)n2)c1 (ester hydrolysis, 6%)",
        "reference": "Patent: US7169791B2",
    },
    {
        "ord_id": "ord-8d5a33f90e124f6b71c8",
        "rxn_smiles": "O=C(O)c1ccc(-c2ccccc2)cc1.NCC1CC1>>O=C(NCC1CC1)c1ccc(-c2ccccc2)cc1",
        "yield_pct": 91.0,
        "catalysts": "None",
        "reagents": "EDC.HCl (1.3 eq); HOBt (1.2 eq); DIPEA (2.5 eq)",
        "solvents": "DCM",
        "temp_c": 22.0,
        "atmosphere": "NITROGEN",
        "undesired_byproducts": "None",
        "reference": "DOI: 10.1021/acs.jmedchem.0c01542",
    },
    {
        "ord_id": "ord-5a9c12d84f014e7a63b2",
        "rxn_smiles": "Clc1nccc(-c2cccnc2)n1.Cc1ccc(C(=O)O)cc1N>>Cc1ccc(C(=O)O)cc1Nc1nccc(-c2cccnc2)n1",
        "yield_pct": 76.0,
        "catalysts": "Pd2(dba)3 (0.03 eq); Xantphos (0.06 eq)",
        "reagents": "Cs2CO3 (2.2 eq)",
        "solvents": "1,4-dioxane",
        "temp_c": 105.0,
        "atmosphere": "ARGON",
        "undesired_byproducts": "Diarylated aniline (<3%)",
        "reference": "DOI: 10.1021/jo902241s",
    },
    {
        "ord_id": "ord-1b7e44a20c884d9c55f1",
        "rxn_smiles": "Clc1nccc(-c2cccnc2)n1.Cc1ccc(C(=O)O)cc1N>>Cc1ccc(C(=O)O)cc1Nc1nccc(-c2cccnc2)n1",
        "yield_pct": 6.5,
        "catalysts": "Pd(OAc)2 (0.05 eq); BINAP (0.08 eq)",
        "reagents": "NaOtBu (2.5 eq)",
        "solvents": "toluene",
        "temp_c": 110.0,
        "atmosphere": "NITROGEN",
        "undesired_byproducts": "Unreacted aryl chloride + carboxylate salt precipitation",
        "reference": "DOI: 10.1126/science.aat8603 (HTE Negative Well)",
    },
    {
        "ord_id": "ord-3f6d99c11a234b4e88a7",
        "rxn_smiles": "Fc1cc(Br)cc(C(F)(F)F)c1.Cc1cn[nH]c1>>Cc1cnn(-c2cc(Br)cc(C(F)(F)F)c2)c1",
        "yield_pct": 74.0,
        "catalysts": "CuI (0.10 eq); trans-N,N'-dimethylcyclohexane-1,2-diamine (0.20 eq)",
        "reagents": "K3PO4 (2.0 eq)",
        "solvents": "DMF",
        "temp_c": 110.0,
        "atmosphere": "NITROGEN",
        "undesired_byproducts": "3-methyl regioisomer N-arylation (9%)",
        "reference": "DOI: 10.1021/ol0496809",
    },
    {
        "ord_id": "ord-0e8b55d31c774a2f66d0",
        "rxn_smiles": "O=Cc1ccc(Cl)cc1.NCCc1ccccc1>>Clc1ccc(CNCc2ccccc2)cc1",
        "yield_pct": 86.0,
        "catalysts": "AcOH (0.10 eq)",
        "reagents": "NaBH(OAc)3 (1.5 eq)",
        "solvents": "DCE",
        "temp_c": 25.0,
        "atmosphere": "NITROGEN",
        "undesired_byproducts": "Tertiary amine over-alkylation (5%)",
        "reference": "DOI: 10.1021/jo960057x",
    },
]


def compute_folded_rxn_fp(rxn_smiles: str, n_bits: int = 2048) -> Optional[bytes]:
    """
    Computes a 2048-bit binary RDKit Reaction Fingerprint (packed into 256 bytes)
    combining reaction-center structural & difference features for vectorized Tanimoto.
    """
    try:
        s = (rxn_smiles or "").strip()
        if ">>" not in s:
            return None
        rxn = rdChemReactions.ReactionFromSmarts(s, useSmiles=True)
        if not rxn or rxn.GetNumReactantTemplates() == 0 or rxn.GetNumProductTemplates() == 0:
            return None
        rdChemReactions.RemoveMappingNumbersFromReactions(rxn)

        params = rdChemReactions.ReactionFingerprintParams()
        params.fpSize = n_bits
        bv = rdChemReactions.CreateStructuralFingerprintForReaction(rxn, ReactionFpParams=params)

        arr = np.zeros((n_bits,), dtype=np.uint8)
        DataStructs.ConvertToNumpyArray(bv, arr)
        return np.packbits(arr).tobytes()
    except Exception:
        return None


def compute_rxn_center_diff_fp(rxn_smiles: str):
    """Computes an RDKit Difference Fingerprint specifically weighting reacting bond changes."""
    try:
        rxn = rdChemReactions.ReactionFromSmarts((rxn_smiles or "").strip(), useSmiles=True)
        if not rxn or rxn.GetNumReactantTemplates() == 0 or rxn.GetNumProductTemplates() == 0:
            return None
        rdChemReactions.RemoveMappingNumbersFromReactions(rxn)
        return rdChemReactions.CreateDifferenceFingerprintForReaction(rxn)
    except Exception:
        return None


def parse_ord_reaction_protobuf(rxn) -> Optional[dict]:
    """
    Extracts flat experimental metadata, desired yields, negative-data side-products,
    and grounded conditions from an ord_schema.proto.reaction_pb2.Reaction protobuf.
    """
    if not _HAS_ORD_SCHEMA:
        return None
    try:
        rxn_smiles = ord_message_helpers.get_reaction_smiles(rxn, generate_if_missing=True)
    except Exception:
        rxn_smiles = ""
    if not rxn_smiles or ">>" not in rxn_smiles:
        return None

    parts = rxn_smiles.split(">>")
    reactants_smi, products_smi = parts[0].strip(), parts[-1].strip()
    if not reactants_smi or not products_smi:
        return None

    prod_mol = Chem.MolFromSmiles(_extract_primary_smiles(products_smi))
    if prod_mol is None:
        return None
    prod_inchikey = Chem.MolToInchiKey(prod_mol)
    prod_connectivity = prod_inchikey.split("-")[0]

    best_yield = None
    undesired_byproducts = []
    for outcome in rxn.outcomes:
        for prod in outcome.products:
            if prod.is_desired_product:
                for meas in prod.measurements:
                    if meas.type == ord_reaction_pb2.ProductMeasurement.YIELD and meas.HasField("percentage"):
                        val = float(meas.percentage.value)
                        if 0.0 <= val <= 100.0:
                            if best_yield is None or val > best_yield:
                                best_yield = round(val, 1)
            else:
                byp_smi = next((i.value for i in prod.identifiers if i.type == ord_reaction_pb2.CompoundIdentifier.SMILES), "")
                byp_name = next((i.value for i in prod.identifiers if i.type == ord_reaction_pb2.CompoundIdentifier.NAME), byp_smi)
                if byp_name and byp_name not in undesired_byproducts:
                    undesired_byproducts.append(byp_name)

    catalysts, reagents, solvents = [], [], []
    for _, inp in rxn.inputs.items():
        for comp in inp.components:
            role = ord_reaction_pb2.ReactionRole.ReactionRoleType.Name(comp.reaction_role)
            smi = next((i.value for i in comp.identifiers if i.type == ord_reaction_pb2.CompoundIdentifier.SMILES), "")
            name = next((i.value for i in comp.identifiers if i.type == ord_reaction_pb2.CompoundIdentifier.NAME), smi)
            label_str = name if name else smi
            if not label_str:
                continue
            if role == "CATALYST" and label_str not in catalysts:
                catalysts.append(label_str)
            elif role == "REAGENT" and label_str not in reagents:
                reagents.append(label_str)
            elif role == "SOLVENT" and label_str not in solvents:
                solvents.append(label_str)

    temp_c = None
    if rxn.conditions.HasField("temperature") and rxn.conditions.temperature.HasField("setpoint"):
        sp = rxn.conditions.temperature.setpoint
        if sp.units == ord_reaction_pb2.Temperature.CELSIUS:
            temp_c = round(float(sp.value), 1)
        elif sp.units == ord_reaction_pb2.Temperature.KELVIN:
            temp_c = round(float(sp.value) - 273.15, 1)

    atmosphere = "UNSPECIFIED"
    if rxn.setup.HasField("environment"):
        atmosphere = ord_reaction_pb2.ReactionSetup.ReactionEnvironment.EnvironmentType.Name(rxn.setup.environment.type)

    doi = rxn.provenance.doi if rxn.HasField("provenance") and rxn.provenance.doi else ""
    patent = rxn.provenance.patent if rxn.HasField("provenance") and rxn.provenance.patent else ""
    ref = f"DOI: {doi}" if doi else (f"Patent: {patent}" if patent else "ORD Record")

    clean_rxn_smi = f"{reactants_smi}>>{products_smi}"
    fp_bytes = compute_folded_rxn_fp(clean_rxn_smi)
    if fp_bytes is None:
        return None

    return {
        "ord_id": rxn.reaction_id or f"ord-{abs(hash(clean_rxn_smi)) % 10**12:012d}",
        "rxn_smiles": clean_rxn_smi,
        "prod_inchikey": prod_inchikey,
        "prod_connectivity": prod_connectivity,
        "yield_pct": best_yield,
        "catalysts": "; ".join(catalysts[:4]) or "None",
        "reagents": "; ".join(reagents[:5]) or "None",
        "solvents": "; ".join(solvents[:3]) or "Unspecified",
        "temp_c": temp_c,
        "atmosphere": atmosphere,
        "undesired_byproducts": "; ".join(undesired_byproducts[:3]) or "None",
        "reference": ref,
        "fp_bytes": fp_bytes,
    }


def _rdkit_extract_reacting_cores(forward_smiles: str) -> dict:
    """
    Sub-millisecond RDKit reaction-center extractor for Tier-2 online ORD fallback queries.
    Selects the largest heavy-atom limiting reactant and primary product.
    """
    s = (forward_smiles or "").strip()
    react_s, prod_s = s.split(">>", 1) if ">>" in s else (s, "")
    r_smiles = [x.strip() for x in react_s.split(".") if x.strip()]
    p_smi = _extract_primary_smiles(prod_s)
    if not r_smiles or not p_smi:
        return {"core_reactant": "", "core_product": ""}

    r_mols = [(smi, Chem.MolFromSmiles(smi)) for smi in r_smiles]
    r_mols = [(smi, m) for smi, m in r_mols if m is not None]
    r_mols.sort(key=lambda x: x[1].GetNumHeavyAtoms(), reverse=True)
    lim_r_smi = r_mols[0][0] if r_mols else r_smiles[0]
    return {"core_reactant": lim_r_smi, "core_product": p_smi}


def _submit_and_poll_ord_session(
    params: List[tuple],
    timeout_sec: float = 3.2,
    poll_interval: float = 0.25,
) -> List[bytes]:
    """
    Thread-safe asynchronous query submission and polling against https://open-reaction-database.org/api
    using the shared connection-pooled HTTP session.
    """
    deadline = time.time() + timeout_sec
    try:
        sub_resp = _ORD_HTTP_SESSION.get(
            f"{ORD_ONLINE_API_BASE}/submit_query",
            params=params,
            timeout=1.8,
        )
        if sub_resp.status_code != 200:
            return []
        try:
            payload = sub_resp.json()
            task_id = payload.get("task_id", payload) if isinstance(payload, dict) else str(payload)
        except Exception:
            task_id = sub_resp.text.strip().strip('"')
        if not task_id:
            return []

        while time.time() < deadline:
            remaining = max(0.4, deadline - time.time())
            poll_resp = _ORD_HTTP_SESSION.get(
                f"{ORD_ONLINE_API_BASE}/fetch_query_result",
                params={"task_id": task_id},
                timeout=min(1.5, remaining),
            )
            if poll_resp.status_code == 200:
                items = poll_resp.json()
                if not isinstance(items, list):
                    return []
                out_protos = []
                for item in items:
                    b64_proto = item.get("proto") if isinstance(item, dict) else None
                    if b64_proto:
                        try:
                            out_protos.append(base64.b64decode(b64_proto))
                        except Exception:
                            continue
                return out_protos
            elif poll_resp.status_code in (202, 404):
                time.sleep(poll_interval)
            else:
                break
    except Exception:
        pass
    return []


def _fetch_step_ord_records_worker(
    forward_smiles: str,
    limit: int = 20,
    similarity_threshold: float = 0.40,
    per_step_timeout: float = 3.2,
) -> List[dict]:
    """
    Worker function executed inside ThreadPoolExecutor.
    Performs network I/O and protobuf parsing ONLY (zero DuckDB lock contention).
    """
    s = (forward_smiles or "").strip()
    if ">>" not in s or not _HAS_ORD_SCHEMA:
        return []

    react_s, prod_s = [p.strip() for p in s.split(">>", 1)]
    r_list = [x.strip() for x in react_s.split(".") if x.strip()][:2]
    p_smi = _extract_primary_smiles(prod_s)
    if not p_smi:
        return []

    t0 = time.time()
    tier1_params = [
        ("component", f"{p_smi};output;similarity"),
        ("similarity", str(similarity_threshold)),
        ("use_stereochemistry", "false"),
        ("limit", str(limit)),
    ]
    if r_list:
        tier1_params.append(("component", f"{r_list[0]};input;similarity"))

    raw_protos = _submit_and_poll_ord_session(
        tier1_params,
        timeout_sec=per_step_timeout,
        poll_interval=0.22,
    )

    elapsed = time.time() - t0
    if not raw_protos and elapsed < (per_step_timeout - 0.8):
        cores = _rdkit_extract_reacting_cores(s)
        tier2_params = [
            ("component", f"{cores['core_product']};output;similarity"),
            ("similarity", str(max(0.32, similarity_threshold - 0.08))),
            ("use_stereochemistry", "false"),
            ("limit", str(limit)),
        ]
        raw_protos = _submit_and_poll_ord_session(
            tier2_params,
            timeout_sec=max(0.8, per_step_timeout - elapsed),
            poll_interval=0.20,
        )

    parsed_list = []
    seen_local = set()
    for pb_bytes in raw_protos:
        try:
            rxn_pb = ord_reaction_pb2.Reaction()
            rxn_pb.ParseFromString(pb_bytes)
            rec = parse_ord_reaction_protobuf(rxn_pb)
            if rec and rec["ord_id"] not in seen_local:
                seen_local.add(rec["ord_id"])
                parsed_list.append(rec)
        except Exception:
            continue
    return parsed_list


class ORDRetrosynthesisValidator:
    """
    Validates proposed retrosynthesis steps against ORD Parquet shards & live online ORD queries:
      1. Structural & Reaction-Center Precedent Matching (InChIKey + Structural/Difference FP Tanimoto)
      2. Empirical Feasibility & 'Negative Data' Filtering (Yield stats, HTE <15% failure rate, ORD side-products)
      3. Forward Condition Grounding (Consensus catalyst, reagent, solvent, temperature, atmosphere, DOI)
      4. Route-Level Composite Scoring (ORD Empirical Confidence Score)
    """

    def __init__(self, db_path: str = ORD_DB_PATH):
        self.db_path = db_path
        os.makedirs(os.path.dirname(os.path.abspath(db_path)), exist_ok=True)
        self.shards_loaded = 0
        self._online_queried_steps = set()
        self._init_db_and_seed()
        self._reload_fingerprint_matrix()

    def _init_db_and_seed(self):
        con = duckdb.connect(self.db_path)
        con.execute("""
            CREATE TABLE IF NOT EXISTS ord_reactions (
                ord_id VARCHAR PRIMARY KEY,
                rxn_smiles VARCHAR,
                prod_inchikey VARCHAR,
                prod_connectivity VARCHAR,
                yield_pct DOUBLE,
                catalysts VARCHAR,
                reagents VARCHAR,
                solvents VARCHAR,
                temp_c DOUBLE,
                atmosphere VARCHAR,
                undesired_byproducts VARCHAR,
                reference VARCHAR,
                fp_bytes BLOB
            )
        """)
        con.execute("CREATE INDEX IF NOT EXISTS idx_ord_conn ON ord_reactions (prod_connectivity)")
        cnt = con.execute("SELECT COUNT(*) FROM ord_reactions").fetchone()[0]
        if cnt == 0:
            seed_rows = []
            for rec in _ORD_CURATED_SEED_RECORDS:
                prod_mol = Chem.MolFromSmiles(_extract_primary_smiles(rec["rxn_smiles"].split(">>")[-1]))
                if prod_mol is None:
                    continue
                ikey = Chem.MolToInchiKey(prod_mol)
                fp_b = compute_folded_rxn_fp(rec["rxn_smiles"])
                if fp_b is None:
                    continue
                seed_rows.append({
                    "ord_id": rec["ord_id"],
                    "rxn_smiles": rec["rxn_smiles"],
                    "prod_inchikey": ikey,
                    "prod_connectivity": ikey.split("-")[0],
                    "yield_pct": rec["yield_pct"],
                    "catalysts": rec["catalysts"],
                    "reagents": rec["reagents"],
                    "solvents": rec["solvents"],
                    "temp_c": rec["temp_c"],
                    "atmosphere": rec["atmosphere"],
                    "undesired_byproducts": rec["undesired_byproducts"],
                    "reference": rec["reference"],
                    "fp_bytes": fp_b,
                })
            if seed_rows:
                df_seed = pd.DataFrame(seed_rows)
                con.execute("INSERT OR IGNORE INTO ord_reactions SELECT * FROM df_seed")
        con.close()

    def _reload_fingerprint_matrix(self):
        con = duckdb.connect(self.db_path, read_only=True)
        rows = con.execute("SELECT ord_id, fp_bytes FROM ord_reactions WHERE fp_bytes IS NOT NULL").fetchall()
        con.close()
        self.ord_ids = [r[0] for r in rows]
        if rows:
            self.fp_matrix = np.frombuffer(b"".join(r[1] for r in rows), dtype=np.uint8).reshape(len(rows), -1)
        else:
            self.fp_matrix = np.empty((0, 256), dtype=np.uint8)

    def get_index_size(self) -> int:
        return len(self.ord_ids)

    def batch_prefetch_online_ord(
        self,
        forward_smiles_list: List[str],
        max_workers: int = 4,
        limit_per_step: int = 20,
        wall_timeout_sec: float = 15.0,
    ) -> dict:
        """
        Parallelizes online ORD lookups across all unique reaction steps in a retrosynthesis run:
          1. Filters out steps that already have strong local DuckDB precedents (sim >= 0.55)
          2. Fires concurrent HTTP queries via ThreadPoolExecutor (up to 16 threads)
          3. Commits all newly discovered Reaction records in ONE single DuckDB transaction
          4. Rebuilds the vectorized fingerprint matrix ONCE.
        """
        global _HAS_ORD_SCHEMA
        if not _HAS_ORD_SCHEMA:
            try:
                subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ord-schema"])
                from ord_schema import datasets as _ods, message_helpers as _omh
                from ord_schema.proto import reaction_pb2 as _orpb
                globals()["ord_datasets"] = _ods
                globals()["ord_message_helpers"] = _omh
                globals()["ord_reaction_pb2"] = _orpb
                _HAS_ORD_SCHEMA = True
            except Exception:
                return {"queried_steps": 0, "added_records": 0, "elapsed_sec": 0.0, "indexed_count": self.get_index_size()}

        unique_steps = list(dict.fromkeys(
            s.strip() for s in (forward_smiles_list or []) if s and ">>" in s
        ))
        if not unique_steps:
            return {"queried_steps": 0, "added_records": 0, "elapsed_sec": 0.0, "indexed_count": self.get_index_size()}

        t_start = time.time()
        steps_to_fetch = []
        for smi in unique_steps:
            if smi in self._online_queried_steps:
                continue
            local_check = self.validate_step(smi, top_k=1)
            if local_check.get("max_rxn_similarity", 0.0) < 0.55 and local_check.get("exact_product_hits", 0) == 0:
                steps_to_fetch.append(smi)
                self._online_queried_steps.add(smi)

        if not steps_to_fetch:
            return {
                "queried_steps": 0,
                "added_records": 0,
                "elapsed_sec": round(time.time() - t_start, 3),
                "indexed_count": self.get_index_size(),
            }

        all_new_records = {}
        n_workers = min(max_workers, len(steps_to_fetch))
        per_step_budget = max(1.5, wall_timeout_sec - 0.3)

        executor = ThreadPoolExecutor(max_workers=n_workers)
        future_to_smi = {
            executor.submit(
                _fetch_step_ord_records_worker,
                smi,
                limit_per_step,
                0.40,
                per_step_budget,
            ): smi
            for smi in steps_to_fetch
        }

        try:
            for fut in as_completed(future_to_smi, timeout=wall_timeout_sec):
                try:
                    records = fut.result(timeout=0.05)
                    for rec in records:
                        oid = rec.get("ord_id")
                        if oid and oid not in all_new_records:
                            all_new_records[oid] = rec
                except Exception:
                    continue
        except Exception:
            pass
        finally:
            executor.shutdown(wait=False, cancel_futures=True)

        added_cnt = 0
        if all_new_records:
            con = duckdb.connect(self.db_path)
            df_bulk = pd.DataFrame(list(all_new_records.values()))
            con.execute("INSERT OR IGNORE INTO ord_reactions SELECT * FROM df_bulk")
            con.close()
            self._reload_fingerprint_matrix()
            added_cnt = len(all_new_records)

        return {
            "queried_steps": len(steps_to_fetch),
            "added_records": added_cnt,
            "elapsed_sec": round(time.time() - t_start, 2),
            "indexed_count": self.get_index_size(),
        }

    def sync_huggingface_parquet_shards(self, max_shards: int = 1, max_reactions_per_shard: int = 8000) -> dict:
        """
        Streams real ORD .parquet dataset shards from Hugging Face and filters on-the-fly, 
        saving ONLY reactions that match the current AiZynthFinder target or intermediates.
        """
        global _HAS_ORD_SCHEMA, _HAS_HF_HUB, LATEST_RETRO_RESULTS
        
        # 1. Gather target and intermediate connectivity keys from active routes
        target_keys = set()
        if "LATEST_RETRO_RESULTS" in globals() and isinstance(LATEST_RETRO_RESULTS, dict):
            for route in LATEST_RETRO_RESULTS.get("routes", []):
                for step in route.get("steps", []):
                    fwd_smi = step.get("forward_smiles", "")
                    if ">" in fwd_smi:
                        prod_smi = _extract_primary_smiles(fwd_smi.split(">")[-1])
                        mol = Chem.MolFromSmiles(prod_smi)
                        if mol:
                            target_keys.add(Chem.MolToInchiKey(mol).split("-")[0])

        if not target_keys:
            return {"error": "No active routes found. Plan a synthesis first to generate target intermediates.", "indexed_count": self.get_index_size()}

        # 2. Use the datasets library in streaming mode to avoid full file downloads
        try:
            from datasets import load_dataset
        except ImportError:
            import subprocess, sys
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "datasets"])
            from datasets import load_dataset

        repo_id = "open-reaction-database/ord-data"
        con = duckdb.connect(self.db_path)
        added_total = 0
        
        try:
            # Stream the dataset directly
            dataset = load_dataset(repo_id, split="train", streaming=True)
            records = []
            
            for idx, row in enumerate(dataset):
                if idx >= max_reactions_per_shard * max_shards:
                    break
                
                # The raw protobuf bytes are usually under the 'proto' or 'reaction' column
                pb_bytes = row.get("proto") 
                if not pb_bytes:
                    continue
                    
                try:
                    rxn_pb = ord_reaction_pb2.Reaction()
                    rxn_pb.ParseFromString(pb_bytes)
                    parsed = parse_ord_reaction_protobuf(rxn_pb)
                    
                    # 3. ON-THE-FLY FILTER: Only keep if the product connectivity matches an active intermediate
                    if parsed and parsed.get("prod_connectivity") in target_keys:
                        records.append(parsed)
                        
                except Exception:
                    continue
                    
            if records:
                df_new = pd.DataFrame(records)
                con.execute("INSERT OR IGNORE INTO ord_reactions SELECT * FROM df_new")
                added_total += len(records)
                self.shards_loaded += max_shards

        except Exception as e:
            return {"error": f"Streaming failed: {e}", "indexed_count": self.get_index_size()}
        finally:
            con.close()
            
        self._reload_fingerprint_matrix()
        return {
            "added": added_total,
            "indexed_count": self.get_index_size(),
            "shards_loaded": self.shards_loaded,
        }

    def _vectorized_tanimoto(self, query_fp_bytes: bytes, top_k: int = 20) -> list:
        """Computes vectorized bitwise Tanimoto similarity against all indexed ORD reactions."""
        if self.fp_matrix.shape[0] == 0 or not query_fp_bytes:
            return []
        q_vec = np.frombuffer(query_fp_bytes, dtype=np.uint8)
        inter = _POPCOUNT_LUT[np.bitwise_and(self.fp_matrix, q_vec)].sum(axis=1)
        union = _POPCOUNT_LUT[np.bitwise_or(self.fp_matrix, q_vec)].sum(axis=1)
        sims = np.divide(inter, union, out=np.zeros_like(inter, dtype=np.float32), where=union > 0)
        top_idx = np.argsort(sims)[::-1][:top_k]
        return [(self.ord_ids[i], float(sims[i])) for i in top_idx if sims[i] >= 0.22]

    def validate_step(self, forward_smiles: str, top_k: int = 3) -> dict:
        """
        Executes the 4-pillar ORD validation on a single forward reaction step ('A.B>>C'):
          - Structural & Reaction-Center Precedent Matching
          - Empirical Feasibility & Negative-Data HTE Filtering (<15% yield wells & ORD byproducts)
          - Forward Condition Grounding (Catalyst, Reagent, Solvent, Temp, Atmosphere, DOIs)
          - Step ORD Confidence Score
        """
        s = (forward_smiles or "").strip()
        if ">>" not in s:
            return {"step_ord_confidence": 0.5, "top_precedents": []}
        reactants_smi, product_smi = [p.strip() for p in s.split(">>", 1)]
        prod_mol = Chem.MolFromSmiles(_extract_primary_smiles(product_smi))
        prod_conn = Chem.MolToInchiKey(prod_mol).split("-")[0] if prod_mol else ""

        con = duckdb.connect(self.db_path, read_only=True)
        exact_df = con.execute("""
            SELECT ord_id, rxn_smiles, yield_pct, catalysts, reagents, solvents, temp_c, atmosphere, undesired_byproducts, reference
            FROM ord_reactions
            WHERE prod_connectivity = ?
            ORDER BY COALESCE(yield_pct, -1) DESC
        """, [prod_conn]).df()

        q_fp_bytes = compute_folded_rxn_fp(s)
        q_diff_fp = compute_rxn_center_diff_fp(s)
        sim_hits = self._vectorized_tanimoto(q_fp_bytes, top_k=25) if q_fp_bytes else []

        sim_map = dict(sim_hits)
        for oid in exact_df["ord_id"].tolist():
            sim_map[oid] = max(sim_map.get(oid, 0.0), 0.85)

        analog_records = []
        if sim_map:
            placeholders = ",".join(["?"] * len(sim_map))
            df_sim = con.execute(f"""
                SELECT ord_id, rxn_smiles, yield_pct, catalysts, reagents, solvents, temp_c, atmosphere, undesired_byproducts, reference
                FROM ord_reactions
                WHERE ord_id IN ({placeholders})
            """, list(sim_map.keys())).df()

            for _, row in df_sim.iterrows():
                rec = row.to_dict()
                struct_sim = float(sim_map.get(rec["ord_id"], 0.0))
                diff_sim = struct_sim
                if q_diff_fp is not None:
                    cand_diff_fp = compute_rxn_center_diff_fp(rec["rxn_smiles"])
                    if cand_diff_fp is not None:
                        try:
                            diff_sim = float(DataStructs.TanimotoSimilarity(q_diff_fp, cand_diff_fp))
                        except Exception:
                            diff_sim = struct_sim
                blended_sim = round(0.55 * struct_sim + 0.45 * diff_sim, 3)
                if rec["ord_id"] in exact_df["ord_id"].values:
                    blended_sim = max(blended_sim, 0.88)
                rec["similarity"] = blended_sim
                analog_records.append(rec)
        con.close()

        analog_records.sort(key=lambda x: (x["similarity"], x["yield_pct"] if x["yield_pct"] is not None else -1.0), reverse=True)

        reported_yields = [float(r["yield_pct"]) for r in analog_records if r.get("yield_pct") is not None and not np.isnan(r["yield_pct"])]
        high_yielding_analogs = [r for r in analog_records if r.get("yield_pct") is not None and float(r["yield_pct"]) >= 45.0]
        top_k_yields = [float(r["yield_pct"]) for r in analog_records[: max(3, top_k)] if r.get("yield_pct") is not None and not np.isnan(r["yield_pct"])]

        mean_yield = round(float(np.mean(top_k_yields)), 1) if top_k_yields else None
        median_yield = round(float(np.median(reported_yields)), 1) if reported_yields else None
        hte_fail_rate = (
            round(sum(1 for y in reported_yields if y < 15.0) / len(reported_yields) * 100.0, 1)
            if reported_yields else 0.0
        )

        ord_byproducts = []
        for r in analog_records[:5]:
            byp = (r.get("undesired_byproducts") or "").strip()
            if byp and byp != "None" and byp not in ord_byproducts:
                ord_byproducts.append(f"{byp} [{r['ord_id']}]")

        neg_data_warn = "None"
        if hte_fail_rate >= 30.0:
            neg_data_warn = (
                f"ORD Negative-Data Alert: {hte_fail_rate:.0f}% of close ORD/HTE analogs reported <15% yield "
                f"(median yield {median_yield or 0:.0f}%). Inspect catalyst/base pairing carefully."
            )
        elif mean_yield is not None and mean_yield < 45.0:
            neg_data_warn = f"Low Empirical ORD Yield: Top analogs average {mean_yield:.1f}% yield in ORD."

        grounded_source = high_yielding_analogs[0] if high_yielding_analogs else (analog_records[0] if analog_records else None)
        grounded_conditions = ""
        grounded_ref = ""
        if grounded_source and grounded_source.get("similarity", 0.0) >= 0.30:
            parts_cond = []
            if grounded_source.get("catalysts") and grounded_source["catalysts"] != "None":
                parts_cond.append(f"Cat: {grounded_source['catalysts']}")
            if grounded_source.get("reagents") and grounded_source["reagents"] != "None":
                parts_cond.append(f"Reag: {grounded_source['reagents']}")
            if grounded_source.get("solvents") and grounded_source["solvents"] != "Unspecified":
                parts_cond.append(f"Solv: {grounded_source['solvents']}")
            if grounded_source.get("temp_c") is not None and not np.isnan(grounded_source["temp_c"]):
                parts_cond.append(f"{grounded_source['temp_c']:.0f} °C")
            if grounded_source.get("atmosphere") and grounded_source["atmosphere"] not in ("UNSPECIFIED", "None"):
                parts_cond.append(f"{grounded_source['atmosphere']}")
            grounded_conditions = ", ".join(parts_cond)
            grounded_ref = f"{grounded_source['ord_id']} ({grounded_source.get('yield_pct', 'N/A')}% yield, {grounded_source.get('reference', '')})"

        max_sim = float(analog_records[0]["similarity"]) if analog_records else 0.35
        norm_yield = (mean_yield / 100.0) if mean_yield is not None else 0.60
        fail_penalty_factor = 1.0 - 0.35 * ((hte_fail_rate / 100.0) if hte_fail_rate else 0.0)
        raw_step_conf = (0.60 * max_sim + 0.40 * norm_yield) * fail_penalty_factor
        step_ord_confidence = round(float(min(0.99, max(0.10, raw_step_conf))), 3)

        return {
            "exact_product_hits": int(len(exact_df)),
            "max_rxn_similarity": round(max_sim, 3),
            "mean_analog_yield_pct": mean_yield,
            "median_analog_yield_pct": median_yield,
            "hte_failure_rate_pct": hte_fail_rate,
            "negative_data_warning": neg_data_warn,
            "ord_recorded_byproducts": ord_byproducts,
            "grounded_conditions_consensus": grounded_conditions,
            "grounded_reference_summary": grounded_ref,
            "step_ord_confidence": step_ord_confidence,
            "top_precedents": [
                {
                    "ord_id": r["ord_id"],
                    "similarity": r["similarity"],
                    "yield_pct": None if (r.get("yield_pct") is None or np.isnan(r["yield_pct"])) else round(float(r["yield_pct"]), 1),
                    "catalysts": r.get("catalysts", "None"),
                    "reagents": r.get("reagents", "None"),
                    "solvents": r.get("solvents", "Unspecified"),
                    "temp_c": None if (r.get("temp_c") is None or np.isnan(r["temp_c"])) else round(float(r["temp_c"]), 1),
                    "atmosphere": r.get("atmosphere", "UNSPECIFIED"),
                    "reference": r.get("reference", "ORD"),
                }
                for r in analog_records[:top_k]
            ],
        }

    def score_route(self, forward_smiles_steps: List[str]) -> dict:
        """
        Computes the Route-Level ORD Empirical Confidence Score across all steps:
          S_ORD_route = prod_{i=1}^N max(0.05, S_step(i))
          plus geometric step mean and bottleneck step identification.
        """
        if not forward_smiles_steps:
            return {
                "ord_empirical_confidence": 0.0,
                "ord_geometric_confidence": 0.0,
                "ord_confidence_100": 0,
                "bottleneck_step_index": 1,
                "ord_mean_yield_pct": None,
                "ord_max_hte_fail_pct": 0.0,
                "step_validations": [],
            }
        step_reports = [self.validate_step(smi) for smi in forward_smiles_steps]
        confs = [max(0.05, float(r["step_ord_confidence"])) for r in step_reports]

        prod_score = float(np.prod(confs))
        geom_score = float(np.exp(np.mean(np.log(confs))))

        yields = [r["mean_analog_yield_pct"] for r in step_reports if r.get("mean_analog_yield_pct") is not None]
        fails = [r.get("hte_failure_rate_pct", 0.0) for r in step_reports]
        bottleneck_idx = int(np.argmin(confs)) + 1

        return {
            "ord_empirical_confidence": round(prod_score, 4),
            "ord_geometric_confidence": round(geom_score, 4),
            "ord_confidence_100": int(round(geom_score * 100)),
            "bottleneck_step_index": bottleneck_idx,
            "ord_mean_yield_pct": round(float(np.mean(yields)), 1) if yields else None,
            "ord_max_hte_fail_pct": round(float(max(fails)) if fails else 0.0, 1),
            "step_validations": step_reports,
        }


ORD_VALIDATOR = ORDRetrosynthesisValidator()


def _compute_descriptors_and_ro5(smiles_or_rxn: str) -> dict:
    """Computes RDKit descriptors and Lipinski Rule-of-5 flags for the active canvas."""
    s = (smiles_or_rxn or "").strip()
    if not s:
        return {"error": "Empty input"}
    target_smi = _extract_primary_smiles(s)
    mol = Chem.MolFromSmiles(target_smi)
    if mol is None:
        return {"error": f"Invalid SMILES: {target_smi}"}
    canon = Chem.MolToSmiles(mol)
    mw = Descriptors.MolWt(mol)
    exact_mw = Descriptors.ExactMolWt(mol)
    hbd = rdMolDescriptors.CalcNumHBD(mol)
    hba = rdMolDescriptors.CalcNumHBA(mol)
    tpsa = Descriptors.TPSA(mol)
    rot = rdMolDescriptors.CalcNumRotatableBonds(mol)
    rings = rdMolDescriptors.CalcNumRings(mol)
    clogp = Descriptors.MolLogP(mol)
    formula = rdMolDescriptors.CalcMolFormula(mol)
    flags = []
    if mw > 500:
        flags.append("MW > 500")
    if clogp > 5:
        flags.append("cLogP > 5")
    if hbd > 5:
        flags.append("HBD > 5")
    if hba > 10:
        flags.append("HBA > 10")
    has_valid_rxn = ">>" in s and all(p.strip() for p in s.split(">>", 1))
    return {
        "is_reaction": has_valid_rxn,
        "full_smiles": s,
        "canonical_smiles": canon,
        "formula": formula,
        "mw": round(mw, 2),
        "exact_mass": round(exact_mw, 4),
        "hbd": hbd,
        "hba": hba,
        "tpsa": round(tpsa, 1),
        "rotatable": rot,
        "rings": rings,
        "clogp": round(clogp, 2),
        "ro5_pass": len(flags) <= 1,
        "ro5_flags": " · ".join(flags) if flags else "All criteria met",
    }


def mol_or_rxn_to_svg_html(smiles_or_rxn: str, width: int = 480, height: int = 125, highlight_smarts: str = "", label: str = "") -> str:
    """Renders a molecule or reaction SMILES into an inline 2D RDKit SVG card without storing in global RAM."""
    s = (smiles_or_rxn or "").strip()
    try:
        if ">>" in s and all(p.strip() for p in s.split(">>", 1)):
            rxn = AllChem.ReactionFromSmarts(s, useSmiles=True)
            drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
            drawer.drawOptions().padding = 0.06
            drawer.DrawReaction(rxn)
            drawer.FinishDrawing()
            svg = drawer.GetDrawingText()
        else:
            mol = Chem.MolFromSmiles(_extract_primary_smiles(s))
            if mol is None:
                return f"<code>{html.escape(s)}</code>"
            hl_atoms = []
            if highlight_smarts:
                pat = Chem.MolFromSmarts(highlight_smarts)
                if pat and mol.HasSubstructMatch(pat):
                    hl_atoms = list(mol.GetSubstructMatch(pat))
            drawer = rdMolDraw2D.MolDraw2DSVG(width, height)
            drawer.drawOptions().padding = 0.08
            drawer.DrawMolecule(mol, highlightAtoms=hl_atoms)
            drawer.FinishDrawing()
            svg = drawer.GetDrawingText()
        hdr = f"<div style='font-size:11px;font-weight:700;color:#0284c7;margin-bottom:3px;'>{html.escape(label)}</div>" if label else ""
        return f"<div class='chem-card' style='background:#fff;border:1px solid #bae6fd;border-radius:6px;padding:6px;text-align:center;'>{hdr}{svg}</div>"
    except Exception:
        return f"<code>{html.escape(s)}</code>"


def rdkit_classify_step(forward_smiles: str) -> dict:
    """Classifies a forward reaction step (A.B>>C) using local RDKit SMARTS heuristics + ORD Forward Grounding."""
    s = (forward_smiles or "").strip()
    react_s, prod_s = s.split(">>", 1) if ">>" in s else (s, "")
    r_mols = [Chem.MolFromSmiles(x.strip()) for x in react_s.split(".") if x.strip()]
    p_mols = [Chem.MolFromSmiles(x.strip()) for x in prod_s.split(".") if x.strip()]
    r_mols = [m for m in r_mols if m is not None]
    p_mols = [m for m in p_mols if m is not None]
    present_fgs = set()
    for m in r_mols:
        for fg_name, pat in MEDCHEM_SMARTS.items():
            if pat and m.HasSubstructMatch(pat):
                present_fgs.add(fg_name)
    rxn_name = "Synthetic Transformation"
    conditions = "Standard anhydrous conditions; monitor by LC-MS"
    safety_score = 4
    green_swap = "None"
    for rule_name, req_fgs, rule_cond, rule_safe, rule_swap in RDKIT_REACTION_RULES:
        if all(fg in present_fgs for fg in req_fgs):
            rxn_name = rule_name
            conditions = rule_cond
            safety_score = rule_safe
            green_swap = rule_swap
            break

    ord_val = ORD_VALIDATOR.validate_step(s)
    if ord_val.get("max_rxn_similarity", 0.0) >= 0.45 and ord_val.get("grounded_conditions_consensus"):
        conditions = f"{ord_val['grounded_conditions_consensus']} [ORD Grounded: {ord_val['grounded_reference_summary']}]"

    sel_warn = "None"
    pg_note = "None"
    if "primary_amine" in present_fgs and "free_phenol_or_alcohol" in present_fgs:
        sel_warn = "Competing nucleophilic sites (-NH2 vs -OH); control stoichiometry and pH."
    if sum(1 for m in r_mols if m.HasSubstructMatch(MEDCHEM_SMARTS["aryl_halide"])) > 1:
        sel_warn = "Multiple aryl halides present; potential regioselectivity or over-coupling risk."
    if ord_val.get("negative_data_warning", "None") != "None":
        sel_warn = (sel_warn + " | " if sel_warn != "None" else "") + ord_val["negative_data_warning"]

    if "carboxylic_acid" in present_fgs and "ester" in present_fgs:
        pg_note = "Orthogonal ester protecting group present alongside free carboxylic acid."
    stereo_note = "Achiral / Racemic"
    for m in p_mols:
        chiral = Chem.FindMolChiralCenters(m, includeUnassigned=True)
        if chiral:
            stereo_note = f"{len(chiral)} stereocenter(s) present; verify enantiopurity by chiral HPLC."
            break
    exp_proc = (
        f"To an oven-dried reaction vessel equipped with a magnetic stir bar under N2 atmosphere, charge the limiting reactant (1.00 eq) "
        f"and coupling partner (1.15 eq) in anhydrous solvent (~0.2 M). Apply conditions ({conditions}) and stir until TLC and LC-MS "
        f"confirm full consumption of starting material. Quench with aqueous workup, extract into EtOAc (3x), wash with brine, "
        f"dry over anhydrous Na2SO4, filter, concentrate under reduced pressure, and purify by flash column chromatography."
    )
    return {
        "reaction_name": rxn_name,
        "typical_conditions": conditions,
        "safety_green_score": safety_score,
        "greener_solvent_swap": green_swap,
        "selectivity_warning": sel_warn,
        "protecting_group_note": pg_note,
        "stereochem_note": stereo_note,
        "experimental_procedure": exp_proc,
        "ord_validation": ord_val,
    }


class StepClassificationItem(BaseModel):
    forward_smiles: str
    reaction_name: str
    typical_conditions: str
    safety_green_score: int
    greener_solvent_swap: str
    selectivity_warning: str
    experimental_procedure: str


class StepClassificationBatch(BaseModel):
    steps: List[StepClassificationItem]


def batch_classify_with_gemini(forward_smiles_list: List[str], use_gemini: bool = True) -> dict:
    """
    Classifies unique reaction steps with the active Gemini model + thinking level,
    injecting retrieved ORD precedents (conditions, yields, HTE failure rates, DOIs)
    directly into the prompt for empirical grounding.
    """
    if len(GEMINI_STEP_CACHE) > MAX_STEP_CACHE_SIZE:
        GEMINI_STEP_CACHE.clear()
    active_model = ACTIVE_GENAI_SETTINGS.get("model", "gemini-3.8-flash")
    active_think = ACTIVE_GENAI_SETTINGS.get("thinking_level", "medium")
    cache_tag = f"{active_model}:{active_think}:{ORD_VALIDATOR.get_index_size()}"

    results = {}
    missing = []
    for smi in dict.fromkeys(forward_smiles_list):
        cached = GEMINI_STEP_CACHE.get(smi)
        if cached and (not use_gemini or cached.get("gemini_tag") == cache_tag):
            results[smi] = cached
        else:
            base_info = rdkit_classify_step(smi)
            results[smi] = base_info
            missing.append(smi)

    client = get_gemini_client() if use_gemini else None
    if client and missing:
        rxn_lines = []
        for s in missing[:18]:
            ov = results[s].get("ord_validation", {})
            rxn_lines.append(
                f"- Reaction SMILES: {s}\n"
                f"  ORD Precedent (Max Sim {ov.get('max_rxn_similarity', 0):.2f} | Mean Yield {ov.get('mean_analog_yield_pct')}%"
                f" | HTE Fail Rate <15%: {ov.get('hte_failure_rate_pct')}%): "
                f"{ov.get('grounded_conditions_consensus') or 'No close ORD hit'} "
                f"[{ov.get('grounded_reference_summary') or 'USPTO template'}]"
            )
        prompt = (
            "Classify each forward reaction SMILES below for a medicinal/process chemist, grounding your conditions and "
            "experimental procedures in the provided Open Reaction Database (ORD) empirical precedent data.\n"
            "Return JSON matching StepClassificationBatch with concise reaction_name, realistic reactant-specific "
            "typical_conditions (incorporating the grounded ORD catalyst/ligand, reagent, solvent, temp, and ORD/DOI citation when similarity >= 0.45), "
            "safety_green_score (1-5), greener_solvent_swap, selectivity_warning (include any ORD negative-data / HTE failure warnings or competing byproducts), "
            "and a detailed, publication-quality experimental_procedure (vessel setup, addition order, equivalents, molarity, temperature, "
            "TLC/LC-MS monitoring, quench, workup solvent based on polarity, drying, and purification method).\nReactions:\n"
            + "\n".join(rxn_lines)
        )
        try:
            resp, used_m, used_t = call_gemini_with_config(
                contents=prompt,
                temperature=0.1,
                response_mime_type="application/json",
                response_schema=StepClassificationBatch,
            )
            parsed = json.loads(resp.text)
            for item in parsed.get("steps", []):
                f_smi = item.get("forward_smiles", "")
                if f_smi in results:
                    ord_keep = results[f_smi].get("ord_validation")
                    results[f_smi].update(item)
                    results[f_smi]["ord_validation"] = ord_keep
                    results[f_smi]["gemini_enriched"] = True
                    results[f_smi]["gemini_tag"] = f"{used_m}:{used_t}:{ORD_VALIDATOR.get_index_size()}"
        except Exception:
            pass

    for smi, val in results.items():
        GEMINI_STEP_CACHE[smi] = val
    return results


def rdkit_enumerate_byproducts(forward_smiles: str) -> list:
    """Enumerates target product and plausible side-products (protodeboronation, hydrodehalogenation, hydrolysis)."""
    s = (forward_smiles or "").strip()
    if ">>" not in s:
        return []
    react_s, prod_s = s.split(">>", 1)
    r_mols = [Chem.MolFromSmiles(x.strip()) for x in react_s.split(".") if x.strip()]
    p_mols = [Chem.MolFromSmiles(x.strip()) for x in prod_s.split(".") if x.strip()]
    out = []
    seen = set()

    def _add(mol, origin):
        if mol is None:
            return
        try:
            Chem.SanitizeMol(mol)
            smi = Chem.MolToSmiles(mol)
            if smi in seen:
                return
            seen.add(smi)
            out.append({
                "origin": origin,
                "smiles": smi,
                "formula": rdMolDescriptors.CalcMolFormula(mol),
                "exact_mw": round(Descriptors.ExactMolWt(mol), 4),
            })
        except Exception:
            pass

    for pm in p_mols:
        _add(pm, "target product")
    side_transforms = [
        ("protodeboronation impurity", "[#6:1][BX3]([OX2])[OX2]>>[#6:1]"),
        ("hydrodehalogenation impurity", "[c:1][Cl,Br,I]>>[c:1]"),
        ("ester hydrolysis byproduct", "[CX3:1](=[O:2])[OX2][#6]>>[CX3:1](=[O:2])O"),
    ]
    for label, smarts in side_transforms:
        rxn = rdChemReactions.ReactionFromSmarts(smarts)
        for rm in r_mols:
            if rm is None:
                continue
            for prod_tuple in rxn.RunReactants((rm,), maxProducts=2):
                _add(prod_tuple[0], label)
    return out


def rdkit_predict_lcms_ions(forward_smiles: str) -> list:
    """Predicts exact monoisotopic LC-MS adduct ions ([M+H]+, [M+Na]+, [M-H]-, isotope pattern)."""
    s = (forward_smiles or "").strip()
    react_s, prod_s = s.split(">>", 1) if ">>" in s else (s, s)
    entries = []
    seen = set()
    items = [("Reactant", x) for x in react_s.split(".") if x.strip()] + [("Product", x) for x in prod_s.split(".") if x.strip()]
    for b in rdkit_enumerate_byproducts(s):
        if b["origin"] != "target product":
            items.append((f"Byproduct ({b['origin']})", b["smiles"]))
    for role, smi_raw in items:
        mol = Chem.MolFromSmiles(smi_raw.strip())
        if mol is None:
            continue
        smi = Chem.MolToSmiles(mol)
        if (role, smi) in seen:
            continue
        seen.add((role, smi))
        emw = Descriptors.ExactMolWt(mol)
        formula = rdMolDescriptors.CalcMolFormula(mol)
        cl_cnt = sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "Cl")
        br_cnt = sum(1 for a in mol.GetAtoms() if a.GetSymbol() == "Br")
        iso = "Standard M+1"
        if br_cnt:
            iso = f"{br_cnt}x Br (1:1 M/M+2 doublet)"
        elif cl_cnt:
            iso = f"{cl_cnt}x Cl (3:1 M/M+2 isotope)"
        entries.append({
            "role": role,
            "smiles": smi,
            "formula": formula,
            "exact_mw": round(emw, 4),
            "[M+H]+": round(emw + 1.007276, 4),
            "[M+Na]+": round(emw + 22.989221, 4),
            "[M+NH4]+": round(emw + 18.033826, 4),
            "[M-H]-": round(emw - 1.007276, 4),
            "isotope_note": iso,
        })
    return entries


def rdkit_predict_workup_properties(forward_smiles: str) -> dict:
    """Computes ΔcLogP / ΔTPSA between reactants and product and recommends workup conditions."""
    s = (forward_smiles or "").strip()
    if ">>" not in s:
        return {"error": "Not a reaction SMILES"}
    react_s, prod_s = s.split(">>", 1)
    r_mols = [Chem.MolFromSmiles(x.strip()) for x in react_s.split(".") if x.strip()]
    p_mol = Chem.MolFromSmiles(_extract_primary_smiles(prod_s))
    r_mols = [m for m in r_mols if m is not None]
    if p_mol is None or not r_mols:
        return {"error": "Invalid reaction species"}
    p_clogp = Descriptors.MolLogP(p_mol)
    p_tpsa = Descriptors.TPSA(p_mol)
    r_clogp = sum(Descriptors.MolLogP(m) for m in r_mols) / len(r_mols)
    r_tpsa = sum(Descriptors.TPSA(m) for m in r_mols) / len(r_mols)
    d_clogp = p_clogp - r_clogp
    d_tpsa = p_tpsa - r_tpsa
    advice = []
    if p_clogp > 1.5:
        advice.append(f"Lipophilic product (cLogP {p_clogp:.2f}): partitions cleanly into EtOAc or i-PrOAc over aq. NaHCO3/brine.")
    else:
        advice.append(f"Polar product (cLogP {p_clogp:.2f}, TPSA {p_tpsa:.1f} Å²): avoid excessive aqueous washes; extract with 3:1 CHCl3/i-PrOH or 2-MeTHF.")
    if abs(d_tpsa) > 18:
        advice.append(f"Large polarity shift (ΔTPSA {d_tpsa:+.1f} Å²): high normal-phase silica TLC resolution expected.")
    else:
        advice.append(f"Small polarity shift (ΔTPSA {d_tpsa:+.1f} Å²): use gradient flash chromatography or C18 reverse-phase prep-HPLC.")
    return {
        "product": {
            "smiles": Chem.MolToSmiles(p_mol),
            "clogp": round(p_clogp, 2),
            "tpsa": round(p_tpsa, 1),
            "hbd": rdMolDescriptors.CalcNumHBD(p_mol),
            "hba": rdMolDescriptors.CalcNumHBA(p_mol),
        },
        "delta_clogp": round(d_clogp, 2),
        "delta_tpsa": round(d_tpsa, 1),
        "advice": advice,
    }


# ==============================================================================
# SECTION 3: AIZYNTHFINDER ENGINE, MEMORY CLEANUP & LAB STOCK INITIALIZATION
# ==============================================================================
from aizynthfinder.aizynthfinder import AiZynthFinder

CUSTOM_LAB_STOCK_SMILES = set()
CUSTOM_LAB_STOCK_INCHIKEYS = set()


def _init_aizynthfinder_engine():
    """Initializes or reuses the global AiZynthFinder instance (`finder`)."""
    global finder
    if "finder" in globals() and globals()["finder"] is not None:
        return globals()["finder"]
    if not os.path.exists("data/config.yml"):
        raise FileNotFoundError("data/config.yml not found. Please run Cell 1A first to download models and stock.")
    print("⏳ Initializing AiZynthFinder engine (low-arena ONNX USPTO models + ZINC stock)...")
    finder = AiZynthFinder(configfile="data/config.yml")
    finder.stock.select("zinc")
    finder.expansion_policy.select("uspto")
    finder.filter_policy.select("uspto")
    finder.CustomFrozenAtoms = []
    finder.CustomFrozenSmarts = None
    return finder


finder = _init_aizynthfinder_engine()


def cleanup_aizynth_memory(finder_obj=None, trim_ipython_history: bool = True) -> float:
    """
    [Improvement 1] Purges the MCTS/Retro* search tree, cached ReactionTree objects,
    and forces glibc malloc_trim(0) to return freed C++/RDKit heap memory to the OS.
    """
    global _CHEM_SVG_CARDS
    if finder_obj is not None:
        try:
            finder_obj.tree = None
            finder_obj.routes = []
        except Exception:
            pass
    if isinstance(_CHEM_SVG_CARDS, set):
        _CHEM_SVG_CARDS.clear()
    if trim_ipython_history:
        try:
            ip = IPython.get_ipython()
            if ip is not None and hasattr(ip, "user_ns"):
                out_hist = ip.user_ns.get("Out")
                if isinstance(out_hist, dict) and len(out_hist) > 5:
                    out_hist.clear()
                for k in ("_", "__", "___"):
                    ip.user_ns[k] = None
        except Exception:
            pass
    gc.collect()
    try:
        ctypes.CDLL("libc.so.6").malloc_trim(0)
    except Exception:
        pass
    return get_process_ram_mb()


def deduplicate_retro_state(results_dict: dict) -> dict:
    """
    [Improvement 3] Strips heavy inline SVG strings, base64 payloads, or non-JSON objects
    from LATEST_RETRO_RESULTS so global RAM stays minimal for the Chat Agent.
    """
    if not isinstance(results_dict, dict):
        return {}
    results_dict.pop("html", None)
    results_dict.pop("preview_svg", None)
    for route in results_dict.get("routes", []):
        route.pop("svg_html", None)
        route.pop("reaction_tree", None)
        for step in route.get("steps", []):
            step.pop("svg_html", None)
            step.pop("mol_obj", None)
    return results_dict


def register_custom_lab_stock(smiles_list: List[str]) -> int:
    """Validates custom SMILES via RDKit and registers them as $0 on-shelf Lab Stock."""
    added = 0
    for raw in smiles_list:
        mol = Chem.MolFromSmiles((raw or "").strip())
        if mol is not None:
            CUSTOM_LAB_STOCK_SMILES.add(Chem.MolToSmiles(mol))
            CUSTOM_LAB_STOCK_INCHIKEYS.add(Chem.MolToInchiKey(mol))
            added += 1
    return added


# ==============================================================================
# SECTION 4: MULTI-IDENTIFIER CHEMICAL RESOLVER & EPAM KETCHER BUNDLER/SERVER
# ==============================================================================
KETCHER_DIR = os.path.abspath("data/ketcher_standalone")
KETCHER_ZIP_URL = "https://github.com/epam/ketcher/releases/download/v2.27.0/ketcher-standalone-2.27.0.zip"
_KETCHER_PORT = None


def _ensure_ketcher_bundle():
    """
    Downloads EPAM Ketcher Standalone once, starts a silent local HTTP server for
    multi-chunk/WebWorker assets, and builds a DOM-safe base64 bundle.
    """
    global _KETCHER_PORT
    index_candidates = glob.glob(os.path.join(KETCHER_DIR, "**", "index.html"), recursive=True)
    if not index_candidates:
        os.makedirs(KETCHER_DIR, exist_ok=True)
        zip_path = os.path.join(KETCHER_DIR, "ketcher.zip")
        r = requests.get(KETCHER_ZIP_URL, timeout=60)
        r.raise_for_status()
        with open(zip_path, "wb") as f:
            f.write(r.content)
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(KETCHER_DIR)
        os.remove(zip_path)
        index_candidates = glob.glob(os.path.join(KETCHER_DIR, "**", "index.html"), recursive=True)
    index_path = index_candidates[0]
    root_dir = os.path.dirname(index_path)

    if _KETCHER_PORT is None:
        try:
            class _QuietHandler(http.server.SimpleHTTPRequestHandler):
                def log_message(self, format, *args):
                    pass

                def end_headers(self):
                    self.send_header("Access-Control-Allow-Origin", "*")
                    super().end_headers()

            handler = functools.partial(_QuietHandler, directory=root_dir)
            httpd = http.server.ThreadingHTTPServer(("127.0.0.1", 0), handler)
            _KETCHER_PORT = httpd.server_address[1]
            t = threading.Thread(target=httpd.serve_forever, daemon=True)
            t.start()
        except Exception:
            _KETCHER_PORT = 0

    with open(index_path, "r", encoding="utf-8") as f:
        html_str = f.read()

    def _inline_css(m):
        rel = m.group(1).lstrip("./").lstrip("/")
        p = os.path.join(root_dir, rel)
        if os.path.isfile(p):
            with open(p, "r", encoding="utf-8", errors="ignore") as cf:
                return f"<style>{cf.read()}</style>"
        return m.group(0)

    html_str = re.sub(r'<link[^>]+href=["\']([^"\']+\.css)["\'][^>]*>', _inline_css, html_str)
    deferred_scripts = []

    def _extract_js(m):
        rel = m.group(1).lstrip("./").lstrip("/")
        p = os.path.join(root_dir, rel)
        if os.path.isfile(p):
            with open(p, "r", encoding="utf-8", errors="ignore") as jf:
                safe_js = jf.read().replace("</script>", "<\\/script>")
                deferred_scripts.append(f"<script>{safe_js}</script>")
            return ""
        return m.group(0)

    html_str = re.sub(r'<script[^>]+src=["\']([^"\']+\.js)["\'][^>]*>\s*</script>', _extract_js, html_str)
    if deferred_scripts:
        scripts_block = "\n".join(deferred_scripts)
        if "</body>" in html_str:
            html_str = html_str.replace("</body>", f"{scripts_block}\n</body>", 1)
        else:
            html_str += scripts_block
    b64_str = base64.b64encode(html_str.encode("utf-8")).decode("ascii")
    return b64_str, _KETCHER_PORT


def _resolve_chemical_identifier(query: str):
    """Resolves Drug Name, CAS, PubChem CID, ChEMBL ID, DrugBank ID, InChIKey, or SMILES."""
    q = (query or "").strip()
    if not q:
        return None, "Empty identifier query."
    if ">>" in q:
        left, right = [p.strip() for p in q.split(">>", 1)]
        if left and right:
            return q, "Reaction SMILES"
        q = right if right else left
    if any(c in q for c in ("=", "#", "(", ")", "[", "]", "@", "/")):
        mol = Chem.MolFromSmiles(q)
        if mol is not None:
            return Chem.MolToSmiles(mol), "Direct SMILES (RDKit)"
    if q.startswith("InChI="):
        mol = Chem.MolFromInchi(q)
        if mol is not None:
            return Chem.MolToSmiles(mol), "InChI"
    if re.match(r"^[A-Za-z0-9@+\-\[\]()/\\#%=.]+$", q) and not q.isdigit():
        mol = Chem.MolFromSmiles(q)
        if mol is not None and mol.GetNumAtoms() > 1:
            return Chem.MolToSmiles(mol), "Direct SMILES"
    headers = {"User-Agent": "Colab-ChemForge-Ketcher/1.0"}
    if q.upper().startswith("CHEMBL"):
        try:
            r = requests.get(f"https://www.ebi.ac.uk/chembl/api/data/molecule/{q.upper()}.json", headers=headers, timeout=8)
            if r.status_code == 200:
                smi = (r.json().get("molecule_structures") or {}).get("canonical_smiles")
                if smi and Chem.MolFromSmiles(smi):
                    return Chem.MolToSmiles(Chem.MolFromSmiles(smi)), f"ChEMBL ({q.upper()})"
        except Exception:
            pass
    if re.match(r"^DB\d{5}$", q.upper()):
        try:
            r = requests.get(f"https://www.ebi.ac.uk/unichem/rest/src_compound_id/{q.upper()}/2/1", headers=headers, timeout=8)
            if r.status_code == 200 and r.json():
                return _resolve_chemical_identifier(r.json()[0].get("src_compound_id", ""))
        except Exception:
            pass
    cid_match = re.match(r"^(?:CID:?\s*)?(\d+)$", q, re.IGNORECASE)
    enc = urllib.parse.quote(q)
    endpoints = []
    if cid_match:
        endpoints.append((f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid_match.group(1)}/property/CanonicalSMILES,IsomericSMILES/JSON", f"PubChem CID {cid_match.group(1)}"))
    if re.match(r"^[A-Z]{14}-[A-Z]{10}-[A-Z]$", q):
        endpoints.append((f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/inchikey/{q}/property/CanonicalSMILES,IsomericSMILES/JSON", f"PubChem InChIKey ({q})"))
    endpoints.append((f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/name/{enc}/property/CanonicalSMILES,IsomericSMILES/JSON", f"PubChem ({q})"))
    for url, label in endpoints:
        try:
            r = requests.get(url, headers=headers, timeout=8)
            if r.status_code == 200:
                props = r.json()["PropertyTable"]["Properties"][0]
                smi = props.get("IsomericSMILES") or props.get("CanonicalSMILES") or props.get("ConnectivitySMILES")
                if smi and Chem.MolFromSmiles(smi):
                    return Chem.MolToSmiles(Chem.MolFromSmiles(smi)), label
        except Exception:
            continue
    try:
        r = requests.get(f"https://cactus.nci.nih.gov/chemical/structure/{enc}/smiles", headers=headers, timeout=8)
        if r.status_code == 200 and r.text.strip():
            smi = r.text.strip().splitlines()[0]
            if Chem.MolFromSmiles(smi):
                return Chem.MolToSmiles(Chem.MolFromSmiles(smi)), f"NCI CACTUS ({q})"
    except Exception:
        pass
    return None, f"Could not resolve '{q}'."


# ==============================================================================
# SECTION 5: SEARCH CONFIGURATION & RETROSYNTHESIS EXECUTION PIPELINE
# ==============================================================================
RETRO_SEARCH_PARAMS = {
    "search_algorithm": "retrostar",
    "ranking_mode": "balanced",
    "max_transforms": 6,
    "time_limit": 120,
    "iteration_limit": 100,
    "return_first": False,
    "cutoff_cumulative": 0.995,
    "max_routes": 10,
    "target_scale_grams": 1.0,
    "exclude_reaction_names": [],
    "use_gemini": True,
    "validate_ord": True,
}
LATEST_RETRO_RESULTS = {}


def compute_composite_route_score(route: dict, ranking_mode: str = "balanced") -> float:
    """
    Computes multi-objective route score blending AiZynthFinder state score, step count,
    stock ratio, safety score, and the Route-Level ORD Empirical Confidence Score.
    """
    state_s = float(route.get("state_score", 0.0))
    n_steps = max(1, int(route.get("step_count", 1)))
    step_score = max(0.0, 1.0 - (n_steps - 1) * 0.12)
    safe_s = float(route.get("avg_safety_score", 4.0)) / 5.0
    sm = route.get("starting_materials", {})
    stock_ratio = sm.get("in_stock_count", 0) / max(1, sm.get("total_leaves", 1))
    ord_geom = float(route.get("ord_geometric_confidence", route.get("ord_empirical_confidence", 0.65)))
    ord_prod = float(route.get("ord_empirical_confidence", 0.50))
    ord_blend = 0.65 * ord_geom + 0.35 * ord_prod

    risk_pen = min(0.35, int(route.get("selectivity_risk_count", 0)) * 0.07)
    hte_fail_pen = min(0.20, float(route.get("ord_max_hte_fail_pct", 0.0)) / 100.0 * 0.25)

    if ranking_mode == "medchem_fast":
        raw = 0.28 * state_s + 0.35 * step_score + 0.17 * stock_ratio + 0.15 * ord_blend + 0.05 * safe_s
    elif ranking_mode == "process_scaleup":
        raw = 0.22 * state_s + 0.13 * step_score + 0.25 * stock_ratio + 0.20 * ord_blend + 0.20 * safe_s
    elif ranking_mode == "green_safe":
        raw = 0.20 * state_s + 0.12 * step_score + 0.18 * stock_ratio + 0.15 * ord_blend + 0.35 * safe_s
    else:
        raw = 0.25 * state_s + 0.15 * step_score + 0.15 * stock_ratio + 0.35 * ord_blend + 0.10 * safe_s
    return round(max(0.0, raw - risk_pen - hte_fail_pen), 4)


def run_retrosynthesis_pipeline(
    target_smiles: str,
    ranking_mode: str = "balanced",
    max_transforms: int = 6,
    time_limit: int = 120,
    iteration_limit: int = 100,
    return_first: bool = False,
    cutoff_cumulative: float = 0.995,
    max_routes: int = 10,
    exclude_reaction_names: Optional[List[str]] = None,
    lock_substructure_smarts: Optional[str] = None,
    target_scale_grams: float = 1.0,
    search_algorithm: str = "retrostar",
    use_gemini: bool = True,
    render_output: bool = False,
) -> dict:
    """
    Executes AiZynthFinder tree search, prefetches live online ORD precedents in parallel,
    validates every step against DuckDB, and purges tree RAM immediately.
    """
    global LATEST_RETRO_RESULTS
    clean_target = _extract_primary_smiles(target_smiles)
    mol = Chem.MolFromSmiles(clean_target)
    if mol is None:
        return {"error": f"Invalid target SMILES: {target_smiles}", "routes": []}
    canon_target = Chem.MolToSmiles(mol)

    cleanup_aizynth_memory(finder, trim_ipython_history=False)

    cfg = finder.config.search
    if "retro" in (search_algorithm or "").lower():
        cfg.algorithm = "aizynthfinder.search.retrostar.search_tree.SearchTree"
    else:
        cfg.algorithm = "mcts"
    cfg.max_transforms = int(max_transforms)
    cfg.time_limit = int(time_limit)
    cfg.iteration_limit = int(iteration_limit)
    cfg.return_first = bool(return_first)
    try:
        finder.config.expansion_policy.uspto.cutoff_cumulative = float(cutoff_cumulative)
    except Exception:
        pass

    finder.target_smiles = canon_target
    processed_routes = []
    excl_tokens = [x.strip().lower() for x in (exclude_reaction_names or []) if x.strip()]
    lock_smarts = (lock_substructure_smarts or getattr(finder, "CustomFrozenSmarts", "") or "").strip()
    lock_pat = Chem.MolFromSmarts(lock_smarts) if lock_smarts else None
    ord_prefetch_stats = {"queried_steps": 0, "added_records": 0, "elapsed_sec": 0.0}

    try:
        finder.tree_search()
        finder.build_routes()
        raw_routes = list(finder.routes)

        all_fwd_smiles = []
        for r in raw_routes[: max_routes * 2]:
            for rxn_node in reversed(list(r["reaction_tree"].reactions())):
                retro_smi = rxn_node.reaction_smiles()
                if ">>" in retro_smi:
                    p_s, r_s = retro_smi.split(">>", 1)
                    all_fwd_smiles.append(f"{r_s.strip()}>>{p_s.strip()}")

        # Parallel 16-thread online ORD prefetch across all unique steps (<3.8s wall clock)
        if RETRO_SEARCH_PARAMS.get("validate_ord", True):
            ord_prefetch_stats = ORD_VALIDATOR.batch_prefetch_online_ord(
                all_fwd_smiles,
                max_workers=4,
                limit_per_step=20,
                wall_timeout_sec=15.0,
            )

        step_lookup = batch_classify_with_gemini(all_fwd_smiles, use_gemini=use_gemini)

        for r in raw_routes:
            rxn_tree = r["reaction_tree"]
            state_score = float(r["score"].get("state score", 0.0))
            rxn_nodes = list(reversed(list(rxn_tree.reactions())))
            if lock_pat is not None:
                keep = any(
                    Chem.MolFromSmiles(m.smiles) and Chem.MolFromSmiles(m.smiles).HasSubstructMatch(lock_pat)
                    for m in rxn_tree.molecules() if m.smiles != canon_target
                )
                if not keep and len(rxn_nodes) > 0:
                    continue
            steps_data = []
            banned_hit = False
            for s_idx, rxn_node in enumerate(rxn_nodes, start=1):
                retro_smi = rxn_node.reaction_smiles()
                p_s, r_s = retro_smi.split(">>", 1) if ">>" in retro_smi else ("", retro_smi)
                fwd_smi = f"{r_s.strip()}>>{p_s.strip()}"
                meta = getattr(rxn_node, "metadata", {}) or {}
                t_code = str(meta.get("template_code") or meta.get("template_hash") or "USPTO")
                cls_info = dict(step_lookup.get(fwd_smi) or rdkit_classify_step(fwd_smi))
                ord_val = cls_info.get("ord_validation") or ORD_VALIDATOR.validate_step(fwd_smi)

                rxn_label_low = (cls_info.get("reaction_name", "") + " " + cls_info.get("typical_conditions", "")).lower()
                if any(tok in rxn_label_low for tok in excl_tokens):
                    banned_hit = True
                    break
                steps_data.append({
                    "step_index": s_idx,
                    "forward_smiles": fwd_smi,
                    "template_code": t_code,
                    "reaction_name": cls_info.get("reaction_name", "Reaction Step"),
                    "typical_conditions": cls_info.get("typical_conditions", ""),
                    "safety_green_score": int(cls_info.get("safety_green_score", 4)),
                    "greener_solvent_swap": cls_info.get("greener_solvent_swap", "None"),
                    "selectivity_warning": cls_info.get("selectivity_warning", "None"),
                    "protecting_group_note": cls_info.get("protecting_group_note", "None"),
                    "stereochem_note": cls_info.get("stereochem_note", "Achiral / Racemic"),
                    "experimental_procedure": cls_info.get("experimental_procedure", ""),
                    "ord_validation": ord_val,
                })
            if banned_hit or not steps_data:
                continue

            ord_route_eval = ORD_VALIDATOR.score_route([s["forward_smiles"] for s in steps_data])

            precursors = []
            in_stock_cnt = 0
            for leaf in rxn_tree.leafs():
                l_smi = leaf.smiles
                l_mol = Chem.MolFromSmiles(l_smi)
                ikey = Chem.MolToInchiKey(l_mol) if l_mol else ""
                if l_smi in CUSTOM_LAB_STOCK_SMILES or (ikey and ikey in CUSTOM_LAB_STOCK_INCHIKEYS):
                    src, tier = "Lab Stock", "$0 On-Shelf"
                    in_stock_cnt += 1
                elif rxn_tree.in_stock(leaf):
                    src, tier = "ZINC Stock", "Commercial"
                    in_stock_cnt += 1
                else:
                    src, tier = "Not in Stock", "Custom Synthesis"
                precursors.append({"smiles": l_smi, "source": src, "cost_tier": tier})
            avg_safe = sum(s["safety_green_score"] for s in steps_data) / max(1, len(steps_data))
            risk_cnt = sum(1 for s in steps_data if s["selectivity_warning"] != "None")
            route_entry = {
                "is_solved": bool(rxn_tree.is_solved) or (in_stock_cnt == len(precursors)),
                "state_score": round(state_score, 4),
                "step_count": len(steps_data),
                "avg_safety_score": round(avg_safe, 2),
                "selectivity_risk_count": risk_cnt,
                "ord_empirical_confidence": ord_route_eval["ord_empirical_confidence"],
                "ord_geometric_confidence": ord_route_eval["ord_geometric_confidence"],
                "ord_confidence_100": ord_route_eval["ord_confidence_100"],
                "ord_bottleneck_step": ord_route_eval["bottleneck_step_index"],
                "ord_mean_yield_pct": ord_route_eval["ord_mean_yield_pct"],
                "ord_max_hte_fail_pct": ord_route_eval["ord_max_hte_fail_pct"],
                "starting_materials": {
                    "total_leaves": len(precursors),
                    "in_stock_count": in_stock_cnt,
                    "overall_cost_tier": "All In-Stock" if in_stock_cnt == len(precursors) else "Partial Stock",
                    "precursors": precursors,
                },
                "steps": steps_data,
            }
            route_entry["composite_score"] = compute_composite_route_score(route_entry, ranking_mode)
            route_entry["score"] = route_entry["composite_score"]
            processed_routes.append(route_entry)

        del raw_routes
    finally:
        cleanup_aizynth_memory(finder, trim_ipython_history=True)

    processed_routes.sort(key=lambda x: (x["is_solved"], x["composite_score"]), reverse=True)
    processed_routes = processed_routes[: int(max_routes)]
    for i, r_i in enumerate(processed_routes):
        dominated = False
        for j, r_j in enumerate(processed_routes):
            if i == j:
                continue
            if (
                r_j["step_count"] <= r_i["step_count"]
                and r_j["avg_safety_score"] >= r_i["avg_safety_score"]
                and r_j["state_score"] >= r_i["state_score"]
                and r_j.get("ord_empirical_confidence", 0.0) >= r_i.get("ord_empirical_confidence", 0.0)
            ) and (
                r_j["step_count"] < r_i["step_count"]
                or r_j["avg_safety_score"] > r_i["avg_safety_score"]
                or r_j["state_score"] > r_i["state_score"]
                or r_j.get("ord_empirical_confidence", 0.0) > r_i.get("ord_empirical_confidence", 0.0)
            ):
                dominated = True
                break
        r_i["is_pareto_optimal"] = not dominated
        r_i["is_pareto"] = not dominated

    LATEST_RETRO_RESULTS = deduplicate_retro_state({
        "target_smiles": canon_target,
        "ranking_mode": ranking_mode,
        "search_algorithm": "retrostar" if "retro" in str(cfg.algorithm).lower() else "mcts",
        "is_solved": any(r["is_solved"] for r in processed_routes),
        "target_scale_grams": float(target_scale_grams),
        "locked_substructure_smarts": lock_smarts,
        "excluded_reactions": excl_tokens,
        "genai_model": ACTIVE_GENAI_SETTINGS.get("model", "gemini-3.8-flash"),
        "genai_thinking": ACTIVE_GENAI_SETTINGS.get("thinking_level", "medium"),
        "ord_index_size": ORD_VALIDATOR.get_index_size(),
        "ord_prefetch_stats": ord_prefetch_stats,
        "routes": processed_routes,
        "ram_mb_after_cleanup": get_process_ram_mb(),
    })
    return LATEST_RETRO_RESULTS


def generate_eln_markdown(route_idx: int = 1, target_scale_grams: float = 1.0, assumed_step_yield: float = 0.80) -> str:
    """Generates a scaled stoichiometry table (MW, eq, mmol, mass) in Markdown format."""
    routes = (LATEST_RETRO_RESULTS or {}).get("routes", [])
    if not (1 <= int(route_idx) <= len(routes)):
        return "Route index out of range."
    route = routes[int(route_idx) - 1]
    target_mol = Chem.MolFromSmiles(LATEST_RETRO_RESULTS.get("target_smiles", ""))
    target_mw = Descriptors.MolWt(target_mol) if target_mol else 350.0
    final_mmol = (float(target_scale_grams) * 1000.0) / max(1.0, target_mw)
    steps = route.get("steps", [])
    n_steps = len(steps)
    lines = [f"### 📋 Scaled ELN Stoichiometry — Route #{route_idx} ({target_scale_grams:.2f} g Target, {int(assumed_step_yield*100)}% Yield/Step)"]
    for s in steps:
        idx = s["step_index"]
        rem_steps = n_steps - idx + 1
        req_prod_mmol = final_mmol / (max(0.1, assumed_step_yield) ** rem_steps)
        react_s, prod_s = s["forward_smiles"].split(">>", 1)
        lines.append(f"\n**Step {idx}: {s['reaction_name']}** (`{s['typical_conditions']}`)\n")
        lines.append("| Role | SMILES | MW (g/mol) | Eq. | mmol | Mass (g) | Mass (mg) |")
        lines.append("| :--- | :--- | :--- | :--- | :--- | :--- | :--- |")
        for r_i, r_smi in enumerate([x.strip() for x in react_s.split(".") if x.strip()]):
            rm = Chem.MolFromSmiles(r_smi)
            mw = Descriptors.MolWt(rm) if rm else 150.0
            eq = 1.0 if r_i == 0 else 1.15
            mmol = req_prod_mmol * eq
            g = (mmol * mw) / 1000.0
            lines.append(f"| Reactant {r_i+1} | `{r_smi}` | {mw:.2f} | {eq:.2f} | {mmol:.2f} | {g:.3f} g | {g*1000:.0f} mg |")
        pm = Chem.MolFromSmiles(_extract_primary_smiles(prod_s))
        p_mw = Descriptors.MolWt(pm) if pm else 250.0
        exp_mmol = req_prod_mmol * assumed_step_yield
        exp_g = (exp_mmol * p_mw) / 1000.0
        lines.append(f"| **Expected Product** | `{_extract_primary_smiles(prod_s)}` | {p_mw:.2f} | {assumed_step_yield:.2f} | {exp_mmol:.2f} | **{exp_g:.3f} g** | **{exp_g*1000:.0f} mg** |")
    return "\n".join(lines)


def generate_eln_html_table(route_idx: int = 1, target_scale_grams: float = 1.0, assumed_step_yield: float = 0.80, step_filter: Optional[int] = None) -> str:
    """Generates formatted HTML stoichiometry tables for a route or a specific step."""
    routes = (LATEST_RETRO_RESULTS or {}).get("routes", [])
    if not (1 <= int(route_idx) <= len(routes)):
        return "<div style='color:#dc2626;'>Route index out of range.</div>"
    route = routes[int(route_idx) - 1]
    target_mol = Chem.MolFromSmiles(LATEST_RETRO_RESULTS.get("target_smiles", ""))
    target_mw = Descriptors.MolWt(target_mol) if target_mol else 350.0
    final_mmol = (float(target_scale_grams) * 1000.0) / max(1.0, target_mw)
    steps = route.get("steps", [])
    n_steps = len(steps)
    html_blocks = []
    if step_filter is None:
        html_blocks.append(
            f"<div style='font-size:12px;font-weight:700;color:#0369a1;margin-bottom:6px;'>"
            f"📋 Scaled ELN Stoichiometry — Route #{route_idx} ({target_scale_grams:.2f} g Target, {int(assumed_step_yield*100)}% Yield/Step)</div>"
        )
    for s in steps:
        idx = s["step_index"]
        if step_filter is not None and idx != step_filter:
            continue
        rem_steps = n_steps - idx + 1
        req_prod_mmol = final_mmol / (max(0.1, assumed_step_yield) ** rem_steps)
        react_s, prod_s = s["forward_smiles"].split(">>", 1)
        rows_html = []
        for r_i, r_smi in enumerate([x.strip() for x in react_s.split(".") if x.strip()]):
            rm = Chem.MolFromSmiles(r_smi)
            mw = Descriptors.MolWt(rm) if rm else 150.0
            eq = 1.0 if r_i == 0 else 1.15
            mmol = req_prod_mmol * eq
            g = (mmol * mw) / 1000.0
            rows_html.append(
                f"<tr><td>Reactant {r_i+1}</td><td><code>{html.escape(r_smi)}</code></td>"
                f"<td>{mw:.2f}</td><td>{eq:.2f}</td><td>{mmol:.2f}</td><td>{g:.3f} g</td><td>{g*1000:.0f} mg</td></tr>"
            )
        p_smi = _extract_primary_smiles(prod_s)
        pm = Chem.MolFromSmiles(p_smi)
        p_mw = Descriptors.MolWt(pm) if pm else 250.0
        exp_mmol = req_prod_mmol * assumed_step_yield
        exp_g = (exp_mmol * p_mw) / 1000.0
        rows_html.append(
            f"<tr style='background:#f0fdfa;font-weight:600;'><td>Expected Product</td><td><code>{html.escape(p_smi)}</code></td>"
            f"<td>{p_mw:.2f}</td><td>{assumed_step_yield:.2f}</td><td>{exp_mmol:.2f}</td><td>{exp_g:.3f} g</td><td>{exp_g*1000:.0f} mg</td></tr>"
        )
        step_hdr = (
            f"<div style='font-size:11px;font-weight:700;color:#0f172a;margin:8px 0 4px 0;'>"
            f"Step {idx}: {html.escape(s['reaction_name'])} <span style='font-weight:400;color:#475569;'>({html.escape(s['typical_conditions'])})</span></div>"
            if step_filter is None else ""
        )
        html_blocks.append(
            f"{step_hdr}<div style='overflow-x:auto;'><table class='cf-eln-table'>"
            f"<thead><tr><th>Role</th><th>SMILES</th><th>MW (g/mol)</th><th>Eq.</th><th>mmol</th><th>Mass (g)</th><th>Mass (mg)</th></tr></thead>"
            f"<tbody>{''.join(rows_html)}</tbody></table></div>"
        )
    return "".join(html_blocks)


# ==============================================================================
# SECTION 6: ORGANIC CHEMIST AI AGENT & WORKBENCH CALLBACKS
# ==============================================================================
CHEMIST_CHAT_HISTORY = []


def _build_retrosynthesis_context() -> str:
    """
    [Improvement 3] Builds a compact, deduplicated text context from LATEST_RETRO_RESULTS
    (including ORD Empirical Confidence Scores and step-level ORD precedent data) for the Chemist AI Agent.
    """
    res = deduplicate_retro_state(LATEST_RETRO_RESULTS or {})
    target = res.get("target_smiles") or getattr(finder, "target_smiles", "") or "None"
    routes = res.get("routes", [])
    if not routes:
        return f"Active Target SMILES: {target}\nORD DuckDB Index Size: {ORD_VALIDATOR.get_index_size()} reactions\nNo retrosynthesis routes have been generated yet."
    lines = [
        f"Active Target SMILES: {target}",
        f"Search Algorithm: {res.get('search_algorithm', 'retrostar')} | Ranking Profile: {res.get('ranking_mode', 'balanced')} | Target Scale: {res.get('target_scale_grams', 1.0)} g",
        f"ORD DuckDB Index Size: {ORD_VALIDATOR.get_index_size()} reactions | Locked Substructure SMARTS: {res.get('locked_substructure_smarts') or 'None'} | Excluded Reactions: {', '.join(res.get('excluded_reactions') or []) or 'None'}",
        f"Total Extracted Routes: {len(routes)}\n",
    ]
    for idx, r in enumerate(routes, start=1):
        letter = chr(64 + idx) if idx <= 26 else str(idx)
        sm = r.get("starting_materials", {})
        precs = ", ".join(f"{p['smiles']} ({p['source']})" for p in sm.get("precursors", []))
        lines.append(
            f"--- Route {letter} (Route #{idx}) | Solved: {r.get('is_solved')} | Pareto: {r.get('is_pareto_optimal')} | "
            f"Composite Score: {r.get('composite_score')} | ORD Empirical Confidence: {r.get('ord_empirical_confidence')} "
            f"(Geom: {r.get('ord_geometric_confidence')}, Bottleneck: Step {r.get('ord_bottleneck_step')}, Mean ORD Yield: {r.get('ord_mean_yield_pct')}%) | "
            f"State Score: {r.get('state_score')} | Steps: {r.get('step_count')} | Avg Safety: {r.get('avg_safety_score')}/5 ---"
        )
        lines.append(f"  Starting Materials ({sm.get('in_stock_count', 0)}/{sm.get('total_leaves', 0)} in stock): {precs}")
        for s in r.get("steps", []):
            ov = s.get("ord_validation") or {}
            lines.append(
                f"  Step {s['step_index']} [{s['reaction_name']}]: {s['forward_smiles']} | "
                f"ORD Step Conf: {ov.get('step_ord_confidence')} (Max Sim: {ov.get('max_rxn_similarity')}, "
                f"Mean Yield: {ov.get('mean_analog_yield_pct')}%, HTE Fail <15%: {ov.get('hte_failure_rate_pct')}%) | "
                f"Conditions: {s['typical_conditions']} | Safety: {s['safety_green_score']}/5 | "
                f"Selectivity/ORD Warning: {s['selectivity_warning']}"
            )
    return "\n".join(lines)


class StrategySummary(BaseModel):
    iupac_or_label: str
    disconnection_analysis: str
    strategy: str


def _generate_strategy_header(target_smiles: str) -> dict:
    client = get_gemini_client()
    fallback = {
        "iupac_or_label": target_smiles,
        "disconnection_analysis": "Strategic bond disconnections identified via USPTO neural template policy and validated against Open Reaction Database (ORD) precedents.",
        "strategy": "Convergent/linear assembly from commercial ZINC building blocks prioritizing high-confidence ORD transformations, low HTE failure rates, and minimal protecting group steps.",
    }
    if client is None:
        return fallback
    try:
        resp, _, _ = call_gemini_with_config(
            contents=f"Analyze target molecule SMILES: {target_smiles}\nProvide iupac_or_label, 1-sentence disconnection_analysis, and 2-sentence synthetic strategy.",
            temperature=0.2,
            response_mime_type="application/json",
            response_schema=StrategySummary,
        )
        return json.loads(resp.text)
    except Exception:
        return fallback


def _enrich_route_for_chemforge(route: dict, idx: int) -> dict:
    steps = route.get("steps", [])
    n_steps = max(1, len(steps))
    sm = route.get("starting_materials", {})
    total_leaves = max(1, sm.get("total_leaves", 1))
    in_stock = sm.get("in_stock_count", 0)
    is_convergent = total_leaves >= 3 and n_steps >= 2
    lls = max(1, n_steps - (1 if is_convergent and n_steps >= 3 else 0))
    avg_safe = float(route.get("avg_safety_score", 4.0))
    risk_cnt = int(route.get("selectivity_risk_count", 0))
    pg_cnt = sum(1 for s in steps if s.get("protecting_group_note", "None") != "None")
    ord_conf_100 = int(route.get("ord_confidence_100", round(float(route.get("ord_geometric_confidence", 0.65)) * 100)))
    ord_emp = float(route.get("ord_empirical_confidence", 0.50))
    ord_mean_y = route.get("ord_mean_yield_pct")

    metrics = {
        "ORD CONFIDENCE": f"{ord_conf_100}% ({ord_emp:.2f})",
        "ORD MEAN YIELD": f"{ord_mean_y:.0f}%" if ord_mean_y is not None else "N/A",
        "FEASIBLE": int(min(100, max(40, round(float(route.get("state_score", 0.75)) * 100)))),
        "ROBUST": int(max(35, 100 - risk_cnt * 18 - int(float(route.get("ord_max_hte_fail_pct", 0.0)) * 0.3))),
        "SHORT": int(max(30, 100 - (n_steps - 1) * 12)),
        "BUYABLE": int(round((in_stock / total_leaves) * 100)),
        "CHEAP": 100 if sm.get("overall_cost_tier") == "All In-Stock" else 65,
        "CLEAN": int(max(35, 95 - risk_cnt * 20)),
        "MILD CONDITIONS": int(min(100, round(avg_safe * 19))),
        "NO PROTECTING GRP": int(max(30, 100 - pg_cnt * 25)),
        "SAFE REAGENTS": int(min(100, round(avg_safe * 20))),
    }
    failures = []
    for s in steps:
        st_num = s.get("step_index", 1)
        rxn_name = s.get("reaction_name", "Step")
        ov = s.get("ord_validation") or {}
        if s.get("selectivity_warning", "None") != "None":
            failures.append(f"<b>Step {st_num} ({html.escape(rxn_name)}):</b> {html.escape(s['selectivity_warning'])}")
        if ov.get("ord_recorded_byproducts"):
            failures.append(
                f"<b>Step {st_num} ORD Recorded Side-Products (<code>is_desired_product=False</code>):</b> "
                f"{html.escape('; '.join(ov['ord_recorded_byproducts']))}"
            )
        byp = rdkit_enumerate_byproducts(s.get("forward_smiles", ""))
        competing = [b["origin"] for b in byp if b["origin"] != "target product"]
        if competing:
            failures.append(f"<b>Step {st_num} RDKit Plausible Impurities:</b> {', '.join(sorted(set(competing)))}")
        wk = rdkit_predict_workup_properties(s.get("forward_smiles", ""))
        if isinstance(wk, dict) and wk.get("advice"):
            failures.append(f"<b>Step {st_num} Workup/Isolation:</b> {html.escape(wk['advice'][0])}")
    if not failures:
        failures.append("No major chemoselectivity, HTE negative-data, or protecting-group conflicts detected.")
    return {
        "route_letter": chr(64 + idx) if idx <= 26 else str(idx),
        "topology": "convergent" if is_convergent else "linear",
        "lls": lls,
        "score_100": int(round(float(route.get("composite_score", 0.0)) * 100)),
        "ord_emp_score": ord_emp,
        "ord_conf_100": ord_conf_100,
        "ord_bottleneck": route.get("ord_bottleneck_step", 1),
        "precedent_list": ", ".join(s.get("reaction_name", "Reaction") for s in steps),
        "metrics": metrics,
        "failures": failures,
    }


def _render_ord_step_panel_html(ov: dict) -> str:
    """Renders the 4-Pillar ORD Precedent, Negative-Data Filter & Grounded Conditions block for a step."""
    if not ov:
        return ""
    step_conf = ov.get("step_ord_confidence", 0.5)
    max_sim = ov.get("max_rxn_similarity", 0.0)
    mean_y = ov.get("mean_analog_yield_pct")
    med_y = ov.get("median_analog_yield_pct")
    fail_r = ov.get("hte_failure_rate_pct", 0.0)
    exact_hits = ov.get("exact_product_hits", 0)
    grounded_cond = ov.get("grounded_conditions_consensus") or "No high-similarity ORD condition consensus (using SMARTS/USPTO fallback)"

    conf_color = "#0f766e" if step_conf >= 0.68 else ("#b45309" if step_conf >= 0.45 else "#be123c")
    fail_badge = (
        f"<span style='background:#ffe4e6;color:#be123c;padding:1px 6px;border-radius:4px;font-weight:700;'>"
        f"⚠️ HTE Fail Rate (&lt;15%): {fail_r:.0f}%</span>"
        if fail_r >= 25.0
        else f"<span style='background:#ccfbf1;color:#0f766e;padding:1px 6px;border-radius:4px;font-weight:600;'>"
             f"HTE Fail Rate (&lt;15%): {fail_r:.0f}%</span>"
    )

    prec_chips = []
    for p in ov.get("top_precedents", []):
        oid = p.get("ord_id", "ord")
        sim = p.get("similarity", 0.0)
        y_str = f"{p['yield_pct']:.0f}%" if p.get("yield_pct") is not None else "N/A"
        ref_str = p.get("reference", "ORD")
        ref_html = html.escape(ref_str)
        if "DOI:" in ref_str:
            doi_clean = ref_str.split("DOI:")[-1].strip().split()[0]
            ref_html = f"<a href='https://doi.org/{html.escape(doi_clean)}' target='_blank' style='color:#0284c7;text-decoration:underline;'>{html.escape(ref_str)}</a>"
        prec_chips.append(
            f"<span style='display:inline-block;background:#ffffff;border:1px solid #bae6fd;border-radius:5px;padding:2px 7px;margin:2px 4px 2px 0;font-size:10.5px;'>"
            f"<b>{html.escape(oid)}</b> · Sim <b>{sim:.2f}</b> · Yield <b>{y_str}</b> · Cat: {html.escape(p.get('catalysts', 'None'))} · {ref_html}</span>"
        )

    yield_str = f"{mean_y:.1f}% (median {med_y:.1f}%)" if (mean_y is not None and med_y is not None) else "N/A"
    return (
        f"<div style='margin-top:6px;padding:7px 9px;background:#f0fdfa;border-left:3px solid #0d9488;border-radius:4px;font-size:11px;color:#1e293b;'>"
        f"<div style='display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;'>"
        f"<span style='font-weight:700;color:#0f766e;'>🔬 ORD Step Validation (Exact Product Hits: {exact_hits} · Reaction-Center Tanimoto: {max_sim:.2f})</span>"
        f"<span style='display:flex;gap:6px;align-items:center;'>"
        f"{fail_badge}"
        f"<span style='background:#e0f2fe;color:{conf_color};padding:1px 7px;border-radius:4px;font-weight:700;'>Step S_ORD: {step_conf:.2f}</span>"
        f"</span></div>"
        f"<div style='margin-top:4px;'><b>ORD Forward Grounded Conditions:</b> {html.escape(grounded_cond)} · <b>Analog Mean Yield:</b> {yield_str}</div>"
        f"<div style='margin-top:4px;'>{''.join(prec_chips)}</div>"
        f"</div>"
    )


def _render_routes_html_from_results(res: dict) -> str:
    """Renders the HTML cards for LATEST_RETRO_RESULTS on demand without keeping SVGs in global RAM."""
    routes = res.get("routes", [])
    if not routes:
        return "<div style='color:#dc2626;font-weight:600;'>⚠️ No routes satisfied the current constraints.</div>"
    target = res.get("target_smiles", "")
    rk_mode = res.get("ranking_mode", "balanced")
    scale_g = float(res.get("target_scale_grams", 1.0))
    use_gem = bool(RETRO_SEARCH_PARAMS.get("use_gemini", True))
    used_model = res.get("genai_model") or ACTIVE_GENAI_SETTINGS.get("model", "gemini-3.8-flash")
    used_think = res.get("genai_thinking") or ACTIVE_GENAI_SETTINGS.get("thinking_level", "medium")
    strat = _generate_strategy_header(target)
    cards_html = []
    for i, r in enumerate(routes, start=1):
        cf = _enrich_route_for_chemforge(r, i)
        pills_html = "".join(f"<span class='cf-metric-pill'>{k} <b>{v}</b></span>" for k, v in cf["metrics"].items())
        fail_items = "".join(f"<li style='margin-bottom:4px;'>{f}</li>" for f in cf["failures"])
        steps_html = []
        for st in r["steps"]:
            svg = mol_or_rxn_to_svg_html(st["forward_smiles"], width=480, height=120, label=f"Step {st['step_index']}: {st['reaction_name']}")
            swap = f" · <span style='color:#0d9488;font-weight:600;'>Green Swap: {html.escape(st['greener_solvent_swap'])}</span>" if st.get("greener_solvent_swap", "None") != "None" else ""
            ord_panel_html = _render_ord_step_panel_html(st.get("ord_validation") or {})
            proc_text = st.get("experimental_procedure", "").strip()
            proc_html = (
                f"<div style='margin-top:6px;padding:7px 9px;background:#f0f9ff;border-left:3px solid #0284c7;border-radius:4px;font-size:11px;color:#1e293b;line-height:1.45;'>"
                f"<b style='color:#0369a1;'>🧪 Experimental Procedure (Step {st['step_index']} · {html.escape(used_model)}):</b> {html.escape(proc_text)}</div>"
            ) if (use_gem and proc_text) else ""
            step_table_html = generate_eln_html_table(i, scale_g, 0.80, step_filter=st["step_index"])
            steps_html.append(
                f"<div class='cf-step-box'>{svg}"
                f"<div style='font-size:11px;color:#334155;margin-top:4px;'>"
                f"<b>Conditions:</b> {html.escape(st['typical_conditions'])} · <b>Safety:</b> {st['safety_green_score']}/5{swap}</div>"
                f"{ord_panel_html}"
                f"{proc_html}"
                f"<div style='margin-top:6px;'>{step_table_html}</div></div>"
            )
        sm_badges = "".join(
            f"<span class='cf-sm-badge {'cf-sm-buy' if p['source'] != 'Not in Stock' else 'cf-sm-custom'}'>"
            f"<code>{html.escape(p['smiles'])}</code> ({p['source']})</span>"
            for p in r["starting_materials"]["precursors"]
        )
        pareto_badge = "<span class='cf-badge-green' style='background:#ccfbf1;color:#0f766e;border:1px solid #5eead4;margin-left:4px;'>⭐ PARETO</span>" if r.get("is_pareto_optimal") else ""
        solved_badge = "<span class='cf-badge-green'>🟢 SOLVED</span>" if r.get("is_solved") else "<span class='cf-badge-green' style='background:#fef2f2;color:#dc2626;'>🟡 PARTIAL</span>"
        cards_html.append(f"""
        <div class="cf-route-card">
          <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
            <div>
              {solved_badge}{pareto_badge}
              <strong style="font-size:14px;margin-left:6px;color:#0f172a;">Route {cf['route_letter']} — {r['step_count']} steps, {cf['topology']}</strong>
              <span style="font-size:11px;color:#475569;margin-left:4px;">(LLS {cf['lls']} · Stock {r['starting_materials']['in_stock_count']}/{r['starting_materials']['total_leaves']} · Bottleneck: Step {cf['ord_bottleneck']})</span>
            </div>
            <div style="display:flex;gap:6px;align-items:center;">
              <div class="cf-score-pill" style="background:linear-gradient(135deg,#0d9488 0%,#0f766e 100%);" title="Multiplicative Route ORD Empirical Confidence Score (S_ORD) &amp; Geometric Step Mean">🔬 ORD S_route: {cf['ord_emp_score']:.3f} ({cf['ord_conf_100']}%)</div>
              <div class="cf-score-pill">score {cf['score_100']}</div>
            </div>
          </div>
          <div style="font-size:11px;color:#475569;margin:6px 0;"><b>Precedent:</b> {html.escape(cf['precedent_list'])}</div>
          <div class="cf-pill-grid">{pills_html}</div>
          <details class="cf-fail-details">
            <summary>⚠ WHY MIGHT THIS FAIL? ({len(cf['failures'])})</summary>
            <ul style="margin:6px 0 0 18px;padding:0;font-size:11px;color:#334155;">{fail_items}</ul>
          </details>
          <div style="margin-top:8px;font-size:11px;font-weight:700;color:#0369a1;">RETROSYNTHETIC STEPS, ORD EMPIRICAL PRECEDENTS, EXPERIMENTAL PROCEDURES &amp; STARTING MATERIALS</div>
          {''.join(steps_html)}
          <div style="margin-top:6px;display:flex;flex-wrap:wrap;gap:6px;">{sm_badges}</div>
        </div>
        """)

    ram_now = get_process_ram_mb()
    ord_cnt = ORD_VALIDATOR.get_index_size()
    pf = res.get("ord_prefetch_stats") or {}
    pf_added = int(pf.get("added_records", 0))
    pf_steps = int(pf.get("queried_steps", 0))
    pf_sec = float(pf.get("elapsed_sec", 0.0))

    if pf_steps > 0:
        pf_pill_html = (
            f" · <span style='background:#ccfbf1;color:#0f766e;padding:1px 6px;border-radius:4px;border:1px solid #5eead4;'>"
            f"🌐 +{pf_added} ONLINE ORD HITS ({pf_steps} STEPS IN {pf_sec:.2f}s)</span>"
        )
    else:
        pf_pill_html = (
            " · <span style='background:#e0f2fe;color:#0369a1;padding:1px 6px;border-radius:4px;'>"
            "⚡ LOCAL ORD CACHE HIT (0.00s)</span>"
        )

    header_html = f"""
    <div class="cf-strat-box">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
        <div style="font-size:10px;font-weight:700;color:#0284c7;">TARGET · {str(res.get('search_algorithm', 'RETROSTAR')).upper()} · MODE: {rk_mode.upper()} ({len(routes)} ROUTES) · ORD INDEX: {ord_cnt:,} RXNS{pf_pill_html} · AI: {html.escape(used_model.upper())} ({html.escape(used_think.upper())}) · RAM: {ram_now} MB</div>
        <button class="cf-btn" style="padding:3px 9px;font-size:11px;" onclick="cfOpenAgentWithRoutes()">🤖 Discuss Routes in Chemist AI Agent</button>
      </div>
      <div style="font-size:13px;font-weight:600;margin:4px 0 6px 0;word-break:break-all;color:#0f172a;">{html.escape(strat.get('iupac_or_label', target))}</div>
      <div style="font-size:10px;font-weight:700;color:#0d9488;">DISCONNECTION ANALYSIS</div>
      <div style="font-size:12px;margin-bottom:6px;color:#1e293b;">{html.escape(strat.get('disconnection_analysis', ''))}</div>
      <div style="font-size:10px;font-weight:700;color:#0d9488;">STRATEGY</div>
      <div style="font-size:12px;color:#1e293b;">{html.escape(strat.get('strategy', ''))}</div>
    </div>
    {''.join(cards_html)}
    """
    return header_html


def _cb_set_genai_model(model_name: str, thinking_level: str = ""):
    info = set_active_genai_settings(model_name, thinking_level)
    return IPython.display.JSON(info)


def _cb_sync_ord_shards(num_shards: int = 1):
    """Streams ORD Parquet shards from Hugging Face into the DuckDB index and updates UI badge."""
    info = ORD_VALIDATOR.sync_huggingface_parquet_shards(max_shards=int(num_shards))
    info["ram_mb"] = get_process_ram_mb()
    return IPython.display.JSON(info)


def _cb_load_identifier(query_a: str, query_b: str = ""):
    qa = (query_a or "").strip()
    qb = (query_b or "").strip()
    smi_a, src_a = _resolve_chemical_identifier(qa)
    if not smi_a:
        return IPython.display.JSON({"error": src_a})
    if qb:
        smi_b, src_b = _resolve_chemical_identifier(qb)
        if not smi_b:
            return IPython.display.JSON({"error": src_b})
        rxn_smi = f"{smi_a}.{smi_b}>>"
        info = _compute_descriptors_and_ro5(smi_a)
        return IPython.display.JSON({"smiles": rxn_smi, "source": f"{src_a} + {src_b}", "descriptors": info})
    finder.target_smiles = _extract_primary_smiles(smi_a)
    info = _compute_descriptors_and_ro5(smi_a)
    return IPython.display.JSON({"smiles": smi_a, "source": src_a, "descriptors": info})


def _cb_sync_ketcher_canvas(molblock_or_smiles: str, selected_atoms_json: str = "[]", manual_smarts: str = ""):
    raw = (molblock_or_smiles or "").strip()
    if not raw:
        return IPython.display.JSON({"error": "Ketcher canvas is empty."})
    mol = None
    if "M  END" in raw or "V2000" in raw or "V3000" in raw:
        mol = Chem.MolFromMolBlock(raw, sanitize=True)
    if mol is None:
        mol = Chem.MolFromSmiles(_extract_primary_smiles(raw))
    if mol is None:
        return IPython.display.JSON({"error": "Invalid chemical structure on canvas."})
    try:
        sel_atoms = [int(i) for i in json.loads(selected_atoms_json or "[]") if 0 <= int(i) < mol.GetNumAtoms()]
    except Exception:
        sel_atoms = []
    canon_smi = Chem.MolToSmiles(mol, canonical=True)
    canon_mol = Chem.MolFromSmiles(canon_smi)
    locked_atoms = []
    if sel_atoms and canon_mol:
        match = canon_mol.GetSubstructMatch(mol)
        if match and len(match) == mol.GetNumAtoms():
            locked_atoms = sorted({match[i] for i in sel_atoms})
    locked_smarts = (manual_smarts or "").strip()
    if locked_atoms and not locked_smarts:
        try:
            locked_smarts = Chem.MolFragmentToSmarts(canon_mol, atomsToUse=locked_atoms)
        except Exception:
            locked_smarts = ""
    finder.target_smiles = canon_smi
    finder.CustomFrozenAtoms = locked_atoms
    finder.CustomFrozenSmarts = locked_smarts or None
    drawer = rdMolDraw2D.MolDraw2DSVG(380, 175)
    drawer.drawOptions().addAtomIndices = True
    drawer.DrawMolecule(canon_mol, highlightAtoms=locked_atoms, highlightAtomColors={i: (0.05, 0.72, 0.65) for i in locked_atoms})
    drawer.FinishDrawing()
    return IPython.display.JSON({
        "smiles": canon_smi,
        "locked_atoms": locked_atoms,
        "locked_smarts": locked_smarts,
        "preview_svg": drawer.GetDrawingText(),
        "descriptors": _compute_descriptors_and_ro5(canon_smi),
        "ram_mb": get_process_ram_mb(),
    })


def _cb_run_synthesis(smiles_str: str, config_json: str = "{}"):
    target = _extract_primary_smiles(smiles_str)
    if not target:
        return IPython.display.JSON({"error": "Load or draw a target molecule first."})
    try:
        ui_cfg = json.loads(config_json or "{}")
    except Exception:
        ui_cfg = {}
    if ui_cfg.get("genai_model") or ui_cfg.get("genai_thinking"):
        set_active_genai_settings(ui_cfg.get("genai_model"), ui_cfg.get("genai_thinking"))
    mode_map = {"cro": "process_scaleup", "discovery": "medchem_fast", "evidence": "balanced", "green": "green_safe"}
    rk_mode = mode_map.get(ui_cfg.get("rank_mode", "cro"), "balanced")
    excl_raw = ui_cfg.get("exclude_reactions", "")
    excl_list = [x.strip() for x in excl_raw.split(",") if x.strip()] if isinstance(excl_raw, str) else list(excl_raw)
    RETRO_SEARCH_PARAMS.update({
        "search_algorithm": ui_cfg.get("algorithm", "retrostar"),
        "ranking_mode": rk_mode,
        "max_transforms": int(ui_cfg.get("max_steps", 6)),
        "time_limit": int(ui_cfg.get("time_limit", 120)),
        "iteration_limit": int(ui_cfg.get("iteration_limit", 100)),
        "return_first": bool(ui_cfg.get("return_first", False)),
        "cutoff_cumulative": float(ui_cfg.get("cutoff", 0.995)),
        "max_routes": int(ui_cfg.get("max_routes", 10)),
        "target_scale_grams": float(ui_cfg.get("scale_grams", 1.0)),
        "exclude_reaction_names": excl_list,
        "use_gemini": bool(ui_cfg.get("use_gemini", True)),
    })
    res = run_retrosynthesis_pipeline(
        target_smiles=target,
        ranking_mode=rk_mode,
        max_transforms=RETRO_SEARCH_PARAMS["max_transforms"],
        time_limit=RETRO_SEARCH_PARAMS["time_limit"],
        iteration_limit=RETRO_SEARCH_PARAMS["iteration_limit"],
        return_first=RETRO_SEARCH_PARAMS["return_first"],
        cutoff_cumulative=RETRO_SEARCH_PARAMS["cutoff_cumulative"],
        max_routes=RETRO_SEARCH_PARAMS["max_routes"],
        exclude_reaction_names=excl_list,
        lock_substructure_smarts=ui_cfg.get("lock_smarts", ""),
        target_scale_grams=RETRO_SEARCH_PARAMS["target_scale_grams"],
        search_algorithm=RETRO_SEARCH_PARAMS["search_algorithm"],
        use_gemini=RETRO_SEARCH_PARAMS["use_gemini"],
    )
    routes = res.get("routes", [])
    if not routes:
        return IPython.display.JSON({"error": "No routes satisfied the current constraints. Try increasing Max Steps / Iteration Limit or relaxing exclusions."})
    header_html = _render_routes_html_from_results(res)
    deduplicate_retro_state(LATEST_RETRO_RESULTS)
    ram_mb = cleanup_aizynth_memory(finder, trim_ipython_history=True)
    return IPython.display.JSON({
        "html": header_html,
        "route_count": len(routes),
        "target_smiles": res.get("target_smiles", target),
        "ord_index_size": ORD_VALIDATOR.get_index_size(),
        "ord_prefetch": res.get("ord_prefetch_stats", {}),
        "ram_mb": ram_mb,
    })


def _cb_reclassify_routes(model_name: str = "", thinking_level: str = "", rank_mode_ui: str = ""):
    """
    Re-runs parallel online ORD step prefetching and batch_classify_with_gemini on LATEST_RETRO_RESULTS
    using the newly selected model/thinking level without re-running AiZynthFinder.
    """
    global LATEST_RETRO_RESULTS
    if model_name or thinking_level:
        set_active_genai_settings(model_name, thinking_level)
    res = LATEST_RETRO_RESULTS or {}
    routes = res.get("routes", [])
    if not routes:
        return IPython.display.JSON({
            "error": "No existing synthesis routes in memory. Click '🚀 Plan Synthesis Route' first to generate routes."
        })

    mode_map = {"cro": "process_scaleup", "discovery": "medchem_fast", "evidence": "balanced", "green": "green_safe"}
    rk_mode = mode_map.get(rank_mode_ui, res.get("ranking_mode", "balanced")) if rank_mode_ui else res.get("ranking_mode", "balanced")
    RETRO_SEARCH_PARAMS["use_gemini"] = True
    RETRO_SEARCH_PARAMS["ranking_mode"] = rk_mode

    all_fwd_smiles = []
    for r in routes:
        for st in r.get("steps", []):
            if st.get("forward_smiles"):
                all_fwd_smiles.append(st["forward_smiles"])

    ord_prefetch = ORD_VALIDATOR.batch_prefetch_online_ord(
        all_fwd_smiles,
        max_workers=4,
        limit_per_step=20,
        wall_timeout_sec=15.0,
    )
    res["ord_prefetch_stats"] = ord_prefetch

    step_lookup = batch_classify_with_gemini(all_fwd_smiles, use_gemini=True)

    for r in routes:
        steps_data = r.get("steps", [])
        for st in steps_data:
            fwd_smi = st.get("forward_smiles", "")
            cls_info = dict(step_lookup.get(fwd_smi) or rdkit_classify_step(fwd_smi))
            st["reaction_name"] = cls_info.get("reaction_name", st.get("reaction_name", "Reaction Step"))
            st["typical_conditions"] = cls_info.get("typical_conditions", st.get("typical_conditions", ""))
            st["safety_green_score"] = int(cls_info.get("safety_green_score", st.get("safety_green_score", 4)))
            st["greener_solvent_swap"] = cls_info.get("greener_solvent_swap", st.get("greener_solvent_swap", "None"))
            st["selectivity_warning"] = cls_info.get("selectivity_warning", st.get("selectivity_warning", "None"))
            st["experimental_procedure"] = cls_info.get("experimental_procedure", st.get("experimental_procedure", ""))
            st["ord_validation"] = cls_info.get("ord_validation") or ORD_VALIDATOR.validate_step(fwd_smi)

        ord_route_eval = ORD_VALIDATOR.score_route([s["forward_smiles"] for s in steps_data])
        r["ord_empirical_confidence"] = ord_route_eval["ord_empirical_confidence"]
        r["ord_geometric_confidence"] = ord_route_eval["ord_geometric_confidence"]
        r["ord_confidence_100"] = ord_route_eval["ord_confidence_100"]
        r["ord_bottleneck_step"] = ord_route_eval["bottleneck_step_index"]
        r["ord_mean_yield_pct"] = ord_route_eval["ord_mean_yield_pct"]
        r["ord_max_hte_fail_pct"] = ord_route_eval["ord_max_hte_fail_pct"]

        avg_safe = sum(s["safety_green_score"] for s in steps_data) / max(1, len(steps_data))
        risk_cnt = sum(1 for s in steps_data if s["selectivity_warning"] != "None")
        r["avg_safety_score"] = round(avg_safe, 2)
        r["selectivity_risk_count"] = risk_cnt
        r["composite_score"] = compute_composite_route_score(r, rk_mode)
        r["score"] = r["composite_score"]

    routes.sort(key=lambda x: (x["is_solved"], x["composite_score"]), reverse=True)
    for i, r_i in enumerate(routes):
        dominated = False
        for j, r_j in enumerate(routes):
            if i == j:
                continue
            if (
                r_j["step_count"] <= r_i["step_count"]
                and r_j["avg_safety_score"] >= r_i["avg_safety_score"]
                and r_j["state_score"] >= r_i["state_score"]
                and r_j.get("ord_empirical_confidence", 0.0) >= r_i.get("ord_empirical_confidence", 0.0)
            ) and (
                r_j["step_count"] < r_i["step_count"]
                or r_j["avg_safety_score"] > r_i["avg_safety_score"]
                or r_j["state_score"] > r_i["state_score"]
                or r_j.get("ord_empirical_confidence", 0.0) > r_i.get("ord_empirical_confidence", 0.0)
            ):
                dominated = True
                break
        r_i["is_pareto_optimal"] = not dominated
        r_i["is_pareto"] = not dominated

    res["ranking_mode"] = rk_mode
    res["genai_model"] = ACTIVE_GENAI_SETTINGS.get("model", "gemini-3.8-flash")
    res["genai_thinking"] = ACTIVE_GENAI_SETTINGS.get("thinking_level", "medium")
    res["ord_index_size"] = ORD_VALIDATOR.get_index_size()
    header_html = _render_routes_html_from_results(res)
    LATEST_RETRO_RESULTS = deduplicate_retro_state(res)
    ram_mb = cleanup_aizynth_memory(finder, trim_ipython_history=True)
    return IPython.display.JSON({
        "html": header_html,
        "route_count": len(routes),
        "target_smiles": res.get("target_smiles", ""),
        "ord_index_size": ORD_VALIDATOR.get_index_size(),
        "ord_prefetch": ord_prefetch,
        "ram_mb": ram_mb,
    })


def _cb_analyze_reaction(rxn_smiles: str, model_name: str = "", thinking_level: str = ""):
    if model_name or thinking_level:
        set_active_genai_settings(model_name, thinking_level)
    s = (rxn_smiles or "").strip()
    if ">>" not in s or not all(p.strip() for p in s.split(">>", 1)):
        return IPython.display.JSON({"error": "Draw a full reaction (Reactants >> Product) in Ketcher or paste a reaction SMILES (`A.B>>C`) first."})
    ORD_VALIDATOR.batch_prefetch_online_ord([s], max_workers=2, wall_timeout_sec=3.5)
    cls = rdkit_classify_step(s)
    ord_val = cls.get("ord_validation") or ORD_VALIDATOR.validate_step(s)
    ord_panel = _render_ord_step_panel_html(ord_val)
    workup = rdkit_predict_workup_properties(s)
    byp = rdkit_enumerate_byproducts(s)
    svg = mol_or_rxn_to_svg_html(s, width=480, height=130, label=cls.get("reaction_name", "Reaction"))
    ai_notes = ""
    used_m = ACTIVE_GENAI_SETTINGS["model"]
    used_t = ACTIVE_GENAI_SETTINGS["thinking_level"]
    if get_gemini_client():
        try:
            resp, used_m, used_t = call_gemini_with_config(
                contents=(
                    f"Analyze reaction SMILES: {s}\n"
                    f"ORD Validation Context: {json.dumps(ord_val)}\n"
                    f"Provide clean HTML using <b> and <ul>/<li> covering:\n"
                    f"1. Top 3 Ranked Conditions grounded in ORD Precedents (catalyst, ligand, base, solvent, temp, DOI)\n"
                    f"2. Full Reactant-Specific Experimental Procedure & Workup\n"
                    f"3. Mechanistic, HTE Negative-Data & Chemoselectivity Failure Modes\n"
                    f"4. Purification & Isolation Gate."
                ),
                temperature=0.2,
            )
            ai_notes = resp.text
        except Exception as e:
            ai_notes = f"<div style='color:#dc2626;'>Gemini error: {html.escape(str(e))}</div>"
    byp_html = "".join(f"<span class='cf-metric-pill'>{html.escape(b['origin'])}: <code>{html.escape(b['smiles'])}</code> ({b['exact_mw']:.1f} Da)</span>" for b in byp)
    wk_advice = "".join(f"<li>{html.escape(a)}</li>" for a in (workup.get("advice", []) if isinstance(workup, dict) else []))
    return IPython.display.JSON({"html": f"""
    <div class="cf-route-card">
      <div style="font-size:10px;font-weight:700;color:#0284c7;margin-bottom:4px;">REACTION ANALYSIS + ORD VALIDATION · MODEL: {html.escape(used_m.upper())} (THINKING: {html.escape(used_t.upper())})</div>
      <div class="cf-step-box">{svg}</div>
      {ord_panel}
      <div style="margin-top:8px;font-size:12px;">
        <b>Conditions:</b> {html.escape(cls.get('typical_conditions', ''))} · <b>Safety:</b> {cls.get('safety_green_score', 4)}/5 · <b>Green Swap:</b> {html.escape(cls.get('greener_solvent_swap', 'None'))}
      </div>
      <div style="margin-top:8px;font-size:11px;font-weight:700;color:#0284c7;">ISOLATION &amp; WORKUP GATE</div>
      <ul style="margin:4px 0 8px 18px;font-size:12px;">{wk_advice}</ul>
      <div style="font-size:11px;font-weight:700;color:#0284c7;">ENUMERATED SPECIES &amp; IMPURITIES</div>
      <div class="cf-pill-grid">{byp_html}</div>
      <div style="font-size:12px;border-top:1px solid #e0f2fe;padding-top:8px;">{ai_notes}</div>
    </div>"""})


def _cb_structure_qa(smiles_str: str, question: str, model_name: str = "", thinking_level: str = ""):
    if model_name or thinking_level:
        set_active_genai_settings(model_name, thinking_level)
    s = (smiles_str or "").strip()
    q = (question or "Compare the top retrosynthesis routes by ORD Empirical Confidence Score, highlight key HTE negative-data/selectivity risks, and suggest grounded ORD reaction conditions.").strip()
    desc = _compute_descriptors_and_ro5(s)
    routes_ctx = json.dumps((LATEST_RETRO_RESULTS or {}).get("routes", [])[:3])
    client = get_gemini_client()
    if client is None:
        return IPython.display.JSON({"html": f"<div class='cf-route-card'>Add <code>KDD_GEMINI_API_KEY</code> in Colab Secrets to enable Gemini Q&amp;A. RDKit descriptors: <code>{json.dumps(desc)}</code></div>"})
    try:
        resp, used_m, used_t = call_gemini_with_config(
            contents=f"Active SMILES: {s}\nDescriptors: {json.dumps(desc)}\nTop AiZynthFinder + ORD Validated Routes: {routes_ctx}\nQuestion: {q}",
            temperature=0.2,
            tools=[types.Tool(google_search=types.GoogleSearch())],
        )
        ans_text = resp.text
    except Exception as e:
        used_m, used_t = ACTIVE_GENAI_SETTINGS["model"], ACTIVE_GENAI_SETTINGS["thinking_level"]
        ans_text = f"Error querying Gemini: {e}"
    return IPython.display.JSON({
        "html": (
            f"<div class='cf-route-card'>"
            f"<div style='font-size:10px;font-weight:700;color:#0284c7;margin-bottom:6px;'>STRUCTURE Q&amp;A · MODEL: {html.escape(used_m.upper())} (THINKING: {html.escape(used_t.upper())})</div>"
            f"<div style='font-size:12px;line-height:1.5;white-space:pre-wrap;'>{html.escape(ans_text)}</div></div>"
        )
    })


def _cb_analytical_report(smiles_str: str, raw_report: str = "", model_name: str = "", thinking_level: str = ""):
    if model_name or thinking_level:
        set_active_genai_settings(model_name, thinking_level)
    s = (smiles_str or "").strip()
    rxn_smi = s if (">>" in s and all(p.strip() for p in s.split(">>", 1))) else f"{_extract_primary_smiles(s)}>>{_extract_primary_smiles(s)}"
    ions = rdkit_predict_lcms_ions(rxn_smi)
    rows = "".join(
        f"<tr><td style='padding:4px;'>{html.escape(i['role'])}</td><td><code>{html.escape(i['formula'])}</code></td>"
        f"<td>{i['exact_mw']:.4f}</td><td><b>{i['[M+H]+']:.4f}</b></td><td>{i['[M+Na]+']:.4f}</td><td>{i['[M-H]-']:.4f}</td><td>{html.escape(i['isotope_note'])}</td></tr>"
        for i in ions
    )
    used_m = ACTIVE_GENAI_SETTINGS["model"]
    used_t = ACTIVE_GENAI_SETTINGS["thinking_level"]
    out = f"""
    <div class="cf-route-card">
      <div style="font-size:11px;font-weight:700;color:#0284c7;margin-bottom:6px;">EXPECTED MONOISOTOPIC LC-MS IONS (RDKit) · AI: {html.escape(used_m.upper())} ({html.escape(used_t.upper())})</div>
      <table class="cf-eln-table">
        <thead><tr>
          <th>Role</th><th>Formula</th><th>Exact Mass</th><th>[M+H]+</th><th>[M+Na]+</th><th>[M-H]-</th><th>Isotope</th>
        </tr></thead>
        <tbody>{rows}</tbody>
      </table>"""
    if (raw_report or "").strip() and get_gemini_client():
        try:
            resp, used_m, used_t = call_gemini_with_config(
                contents=f"Structure: {s}\nExpected Ions: {json.dumps(ions)}\nReport:\n{raw_report}\nAssign peaks, check isotope patterns, and flag plausible byproducts/impurities.",
                temperature=0.2,
            )
            out += f"<div style='margin-top:8px;border-top:1px solid #e0f2fe;padding-top:8px;font-size:12px;white-space:pre-wrap;'>{html.escape(resp.text)}</div>"
        except Exception as e:
            out += f"<div style='margin-top:8px;color:#dc2626;font-size:12px;'>{html.escape(str(e))}</div>"
    return IPython.display.JSON({"html": out + "</div>"})


def _cb_register_stock(raw_text: str):
    tokens = [line.split(",")[0].strip() for line in (raw_text or "").splitlines() if line.strip()]
    cnt = register_custom_lab_stock(tokens)
    return IPython.display.JSON({"message": f"✅ Registered {cnt} building blocks into Lab Stock."})


def _cb_chemist_agent_status():
    res = LATEST_RETRO_RESULTS or {}
    routes = res.get("routes", [])
    target = res.get("target_smiles") or getattr(finder, "target_smiles", "") or "None"
    ram_mb = get_process_ram_mb()
    return IPython.display.JSON({
        "target_smiles": target,
        "route_count": len(routes),
        "is_solved": bool(res.get("is_solved", False)),
        "ranking_mode": res.get("ranking_mode", "balanced"),
        "model": ACTIVE_GENAI_SETTINGS.get("model", "gemini-3.8-flash"),
        "thinking_level": ACTIVE_GENAI_SETTINGS.get("thinking_level", "medium"),
        "ord_index_size": ORD_VALIDATOR.get_index_size(),
        "ram_mb": ram_mb,
    })


def _cb_free_memory():
    deduplicate_retro_state(LATEST_RETRO_RESULTS)
    ram_mb = cleanup_aizynth_memory(finder, trim_ipython_history=True)
    return IPython.display.JSON({"ram_mb": ram_mb, "message": f"🧹 Freed search tree & C++ heap (RAM: {ram_mb} MB)"})


def _cb_chemist_agent_reset():
    CHEMIST_CHAT_HISTORY.clear()
    ram_mb = cleanup_aizynth_memory(finder, trim_ipython_history=True)
    return IPython.display.JSON({"status": "cleared", "ram_mb": ram_mb})


def _cb_chemist_agent_chat(user_msg: str, active_smiles: str = "", model_name: str = "", thinking_level: str = ""):
    """
    Multi-turn Organic Chemist AI Agent with tool-calling to re-run retrosynthesis,
    validate steps against ORD precedents, generate scaled ELN stoichiometry tables,
    and render inline reaction depictions.
    """
    if model_name or thinking_level:
        set_active_genai_settings(model_name, thinking_level)
    msg = (user_msg or "").strip()
    if not msg:
        return IPython.display.JSON({"error": "Empty message."})

    client = get_gemini_client()
    if client is None:
        return IPython.display.JSON({
            "reply_html": "<div style='color:#dc2626;'>⚠️ Please add <code>KDD_GEMINI_API_KEY</code> to Colab Secrets to use the Organic Chemist AI Agent.</div>",
            "ram_mb": get_process_ram_mb(),
        })

    tool_artifacts_html = []
    updated_synthesis_html = None

    def rerun_retrosynthesis_with_constraints(
        ranking_mode: str = "balanced",
        max_steps: int = 6,
        exclude_reactions_csv: str = "",
        lock_substructure_smarts: str = "",
        target_scale_grams: float = 1.0,
        search_algorithm: str = "retrostar",
    ) -> str:
        """Re-runs the AiZynthFinder retrosynthesis search + parallel ORD validation with updated constraints."""
        nonlocal updated_synthesis_html
        target_smi = (LATEST_RETRO_RESULTS or {}).get("target_smiles") or _extract_primary_smiles(active_smiles) or getattr(finder, "target_smiles", "")
        excl = [x.strip() for x in (exclude_reactions_csv or "").split(",") if x.strip()]
        res = run_retrosynthesis_pipeline(
            target_smiles=target_smi,
            ranking_mode=ranking_mode,
            max_transforms=int(max_steps),
            time_limit=int(RETRO_SEARCH_PARAMS.get("time_limit", 120)),
            iteration_limit=int(RETRO_SEARCH_PARAMS.get("iteration_limit", 100)),
            exclude_reaction_names=excl,
            lock_substructure_smarts=lock_substructure_smarts,
            target_scale_grams=float(target_scale_grams),
            search_algorithm=search_algorithm,
            use_gemini=True,
        )
        updated_synthesis_html = _render_routes_html_from_results(res)
        deduplicate_retro_state(LATEST_RETRO_RESULTS)
        cleanup_aizynth_memory(finder, trim_ipython_history=True)
        return _build_retrosynthesis_context()

    def build_scaled_eln_table(route_number: int = 1, scale_grams: float = 1.0, assumed_yield: float = 0.80) -> str:
        """Generates a scaled ELN stoichiometry table for a specific route number (1-indexed)."""
        tbl_html = generate_eln_html_table(int(route_number), float(scale_grams), float(assumed_yield))
        tool_artifacts_html.append(f"<div style='margin-top:8px;padding:8px;background:#f8fafc;border:1px solid #bae6fd;border-radius:6px;'>{tbl_html}</div>")
        return generate_eln_markdown(int(route_number), float(scale_grams), float(assumed_yield))

    def inspect_reaction_step_details(forward_reaction_smiles: str) -> str:
        """Prefetches online ORD hits and predicts byproducts, exact LC-MS adduct ions, and workup ΔTPSA/ΔcLogP."""
        ORD_VALIDATOR.batch_prefetch_online_ord([forward_reaction_smiles], max_workers=2, wall_timeout_sec=3.5)
        svg_card = mol_or_rxn_to_svg_html(forward_reaction_smiles, width=440, height=115, label="Inspected Reaction Step")
        ord_rep = ORD_VALIDATOR.validate_step(forward_reaction_smiles)
        ord_card = _render_ord_step_panel_html(ord_rep)
        tool_artifacts_html.append(f"<div style='margin-top:6px;'>{svg_card}{ord_card}</div>")
        return json.dumps({
            "ord_validation": ord_rep,
            "classification": rdkit_classify_step(forward_reaction_smiles),
            "workup": rdkit_predict_workup_properties(forward_reaction_smiles),
            "byproducts": rdkit_enumerate_byproducts(forward_reaction_smiles),
            "lcms_ions": rdkit_predict_lcms_ions(forward_reaction_smiles),
        })

    retro_ctx = _build_retrosynthesis_context()
    history_text = "\n".join(f"{turn['role'].upper()}: {turn['text']}" for turn in CHEMIST_CHAT_HISTORY[-6:])
    system_instruction = (
        "You are an expert Medicinal & Process Organic Chemist AI Agent integrated into the ChemForge Workbench.\n"
        "You have full access to the active AiZynthFinder retrosynthesis routes, Route-Level ORD Empirical Confidence Scores (S_ORD), "
        "step-level ORD precedent matches, HTE negative-data failure rates (<15% yield), grounded ORD conditions/DOIs, "
        "starting material stock status, and RDKit tools.\n"
        "When the user asks to re-run a search with new constraints (e.g., excluding Pd/Stille/azides, fewer steps, green or evidence mode), "
        "call `rerun_retrosynthesis_with_constraints`. When asked for stoichiometry or scale-up quantities, call `build_scaled_eln_table`. "
        "When asked about ORD precedents, impurities, LC-MS ions, or workup for a specific reaction step, call `inspect_reaction_step_details`.\n"
        "Format your response in clean HTML using <b>, <ul>, <li>, <code>, and concise paragraphs."
    )
    prompt = f"CURRENT RETROSYNTHESIS + ORD VALIDATION SESSION CONTEXT:\n{retro_ctx}\n\nRECENT CONVERSATION HISTORY:\n{history_text}\n\nUSER QUESTION:\n{msg}"

    used_m = ACTIVE_GENAI_SETTINGS["model"]
    used_t = ACTIVE_GENAI_SETTINGS["thinking_level"]
    try:
        resp, used_m, used_t = call_gemini_with_config(
            contents=prompt,
            temperature=0.2,
            system_instruction=system_instruction,
            tools=[rerun_retrosynthesis_with_constraints, build_scaled_eln_table, inspect_reaction_step_details],
        )
        reply_text = resp.text or "Completed tool execution."
    except Exception as e:
        reply_text = f"Agent error: {html.escape(str(e))}"

    CHEMIST_CHAT_HISTORY.append({"role": "user", "text": msg})
    CHEMIST_CHAT_HISTORY.append({"role": "assistant", "text": reply_text[:1500]})
    if len(CHEMIST_CHAT_HISTORY) > 12:
        del CHEMIST_CHAT_HISTORY[:-12]

    ram_mb = cleanup_aizynth_memory(finder, trim_ipython_history=True)
    badge_html = f"<div style='font-size:10px;font-weight:700;color:#0284c7;margin-bottom:4px;'>🤖 {html.escape(used_m)} · Thinking: {html.escape(used_t.upper())}</div>"
    combined_html = f"<div style='line-height:1.5;font-size:12px;'>{badge_html}{reply_text}{''.join(tool_artifacts_html)}</div>"
    routes_now = (LATEST_RETRO_RESULTS or {}).get("routes", [])
    return IPython.display.JSON({
        "reply_html": combined_html,
        "updated_synthesis_html": updated_synthesis_html,
        "route_count": len(routes_now),
        "target_smiles": (LATEST_RETRO_RESULTS or {}).get("target_smiles", active_smiles),
        "ram_mb": ram_mb,
    })


output.register_callback("chemforge.set_model", _cb_set_genai_model)
output.register_callback("chemforge.sync_ord_shards", _cb_sync_ord_shards)
output.register_callback("chemforge.load_id", _cb_load_identifier)
output.register_callback("chemforge.sync_canvas", _cb_sync_ketcher_canvas)
output.register_callback("chemforge.run_synthesis", _cb_run_synthesis)
output.register_callback("chemforge.reclassify_routes", _cb_reclassify_routes)
output.register_callback("chemforge.analyze_reaction", _cb_analyze_reaction)
output.register_callback("chemforge.structure_qa", _cb_structure_qa)
output.register_callback("chemforge.analytical", _cb_analytical_report)
output.register_callback("chemforge.register_stock", _cb_register_stock)
output.register_callback("chemforge.agent_chat", _cb_chemist_agent_chat)
output.register_callback("chemforge.agent_reset", _cb_chemist_agent_reset)
output.register_callback("chemforge.agent_status", _cb_chemist_agent_status)
output.register_callback("chemforge.free_memory", _cb_free_memory)

# ==============================================================================
# SECTION 7: UNIFIED WORKBENCH + AI AGENT UI
# ==============================================================================
def render_unified_chemforge_workbench():
    """Renders the Unified ChemForge Workbench and frees temporary build buffers."""
    ketcher_b64_html, ketcher_local_port = _ensure_ketcher_bundle()
    default_smi = "Cc1cn(-c2cc(NC(=O)c3ccc(C)c(Nc4nccc(-c5cccnc5)n4)c3)cc(C(F)(F)F)c2)cn1"
    init_ram_mb = get_process_ram_mb()
    init_ord_cnt = ORD_VALIDATOR.get_index_size()
    catalog_json = json.dumps(GEMINI_MODEL_CATALOG)
    model_options_html = "".join(
        f'<option value="{m_id}" {"selected" if m_id == "gemini-3.8-flash" else ""}>{html.escape(spec["label"])}</option>'
        for m_id, spec in GEMINI_MODEL_CATALOG.items()
    )
    default_summary = html.escape(GEMINI_MODEL_CATALOG["gemini-3.8-flash"]["summary"])

    unified_workbench_html = f"""
<style>
  :root {{
    --cf-blue: #0284c7;
    --cf-blue-dark: #0369a1;
    --cf-teal: #0d9488;
    --cf-teal-bright: #14b8a6;
    --cf-cyan-glow: rgba(6, 182, 212, 0.22);
    --cf-grad-primary: linear-gradient(135deg, #0284c7 0%, #0d9488 100%);
    --cf-grad-action: linear-gradient(135deg, #0ea5e9 0%, #14b8a6 100%);
    --cf-grad-lock: linear-gradient(135deg, #0369a1 0%, #1e40af 100%);
    --cf-bg-tint: #f0f9ff;
    --cf-border: #bae6fd;
  }}
  .cf-workbench {{
    font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Inter, Roboto, sans-serif;
    display: grid;
    grid-template-columns: 540px 1fr;
    gap: 14px;
    max-width: 1360px;
    margin-top: 6px;
    color: #0f172a;
  }}
  .cf-pane {{
    border: 1px solid var(--cf-border);
    border-radius: 12px;
    background: linear-gradient(180deg, #ffffff 0%, #f8fafc 100%);
    padding: 13px;
    box-sizing: border-box;
    box-shadow: 0 4px 16px rgba(2, 132, 199, 0.06);
  }}
  .cf-sec-hdr {{
    font-size: 11px;
    font-weight: 800;
    color: var(--cf-blue-dark);
    letter-spacing: 0.7px;
    margin-bottom: 9px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    border-bottom: 1px solid #e0f2fe;
    padding-bottom: 6px;
  }}
  .cf-model-bar {{
    background: linear-gradient(135deg, #f0f9ff 0%, #f0fdfa 100%);
    border: 1px solid #7dd3fc;
    border-radius: 9px;
    padding: 9px 11px;
    margin-bottom: 10px;
  }}
  .cf-model-row {{
    display: grid;
    grid-template-columns: 1fr 195px;
    gap: 8px;
    align-items: center;
  }}
  .cf-model-select {{
    width: 100%;
    padding: 5px 8px;
    font-size: 11.5px;
    font-weight: 600;
    color: #0f172a;
    border: 1px solid #38bdf8;
    border-radius: 6px;
    background: #fff;
  }}
  .cf-model-summary {{
    font-size: 11px;
    color: #334155;
    margin-top: 6px;
    line-height: 1.35;
    display: flex;
    justify-content: space-between;
    align-items: center;
    gap: 8px;
  }}
  .cf-input-row {{ display: flex; gap: 6px; margin-bottom: 6px; align-items: center; }}
  .cf-input {{
    flex: 1;
    padding: 6px 9px;
    font-size: 12px;
    border: 1px solid #93c5fd;
    border-radius: 6px;
    background: #fff;
    color: #0f172a;
    transition: all 0.18s ease;
  }}
  .cf-input:focus {{
    outline: none;
    border-color: var(--cf-teal-bright);
    box-shadow: 0 0 0 3px var(--cf-cyan-glow);
  }}
  .cf-btn {{
    background: var(--cf-grad-primary);
    color: #fff;
    border: 1px solid rgba(255, 255, 255, 0.15);
    padding: 6px 12px;
    font-size: 12px;
    font-weight: 600;
    border-radius: 6px;
    cursor: pointer;
    white-space: nowrap;
    box-shadow: 0 2px 6px rgba(2, 132, 199, 0.25);
    transition: transform 0.12s ease, box-shadow 0.15s ease, filter 0.15s ease;
  }}
  .cf-btn:hover {{
    filter: brightness(1.08);
    box-shadow: 0 4px 12px rgba(13, 148, 136, 0.35);
    transform: translateY(-1px);
  }}
  .cf-btn:active {{ transform: translateY(0); }}
  .cf-btn-green {{ background: var(--cf-grad-action); box-shadow: 0 2px 8px rgba(20, 184, 166, 0.32); }}
  .cf-btn-amber {{ background: var(--cf-grad-lock); box-shadow: 0 2px 8px rgba(3, 105, 161, 0.28); }}
  .cf-chip {{
    font-size: 11px;
    padding: 3px 9px;
    border-radius: 12px;
    background: linear-gradient(135deg, #e0f2fe 0%, #ccfbf1 100%);
    color: #0369a1;
    border: 1px solid #7dd3fc;
    font-weight: 600;
    cursor: pointer;
    transition: all 0.15s ease;
  }}
  .cf-chip:hover {{ background: var(--cf-grad-primary); color: #fff; border-color: transparent; }}
  .cf-desc-grid {{
    display: grid;
    grid-template-columns: repeat(4, 1fr);
    gap: 6px;
    background: linear-gradient(135deg, #f0f9ff 0%, #f0fdfa 100%);
    border: 1px solid #bae6fd;
    padding: 8px;
    border-radius: 8px;
    margin-top: 8px;
    font-size: 11px;
  }}
  .cf-desc-item b {{ display: block; color: #0369a1; font-size: 9px; letter-spacing: 0.4px; }}
  .cf-tabs {{ display: flex; gap: 6px; border-bottom: 1px solid #bae6fd; padding-bottom: 8px; margin-bottom: 10px; flex-wrap: wrap; }}
  .cf-tab {{
    padding: 6px 12px;
    font-size: 12px;
    font-weight: 600;
    border: 1px solid #bae6fd;
    border-radius: 7px;
    background: #f0f9ff;
    color: #0369a1;
    cursor: pointer;
    transition: all 0.15s ease;
  }}
  .cf-tab:hover {{ background: #e0f2fe; }}
  .cf-tab.active {{
    background: var(--cf-grad-primary);
    color: #fff;
    border-color: transparent;
    box-shadow: 0 2px 8px rgba(2, 132, 199, 0.28);
  }}
  .cf-mode-pill {{
    padding: 4px 11px;
    font-size: 11px;
    border-radius: 14px;
    border: 1px solid #7dd3fc;
    background: #f0f9ff;
    color: #0369a1;
    cursor: pointer;
    transition: all 0.15s ease;
  }}
  .cf-mode-pill:hover {{ background: #e0f2fe; }}
  .cf-mode-pill.active {{
    background: var(--cf-grad-action);
    color: #fff;
    border-color: transparent;
    font-weight: 700;
    box-shadow: 0 2px 6px rgba(13, 148, 136, 0.28);
  }}
  .cf-cfg-grid {{
    display: grid;
    grid-template-columns: repeat(3, 1fr);
    gap: 8px;
    background: #f0f9ff;
    padding: 10px;
    border: 1px solid #bae6fd;
    border-radius: 8px;
    margin-bottom: 10px;
    font-size: 11px;
  }}
  .cf-cfg-item label {{ display: block; font-weight: 600; color: #0369a1; margin-bottom: 3px; }}
  .cf-cfg-item select, .cf-cfg-item input {{
    width: 100%;
    padding: 4px 6px;
    font-size: 11px;
    border: 1px solid #7dd3fc;
    border-radius: 5px;
    box-sizing: border-box;
    background: #fff;
  }}
  .cf-strat-box {{
    background: linear-gradient(135deg, #f0f9ff 0%, #f0fdfa 100%);
    border-left: 4px solid var(--cf-teal);
    border-top: 1px solid #bae6fd;
    border-right: 1px solid #bae6fd;
    border-bottom: 1px solid #bae6fd;
    padding: 10px 12px;
    border-radius: 8px;
    margin-bottom: 12px;
  }}
  .cf-route-card {{
    border: 1px solid #bae6fd;
    border-radius: 10px;
    padding: 12px;
    margin-bottom: 12px;
    background: #fff;
    box-shadow: 0 2px 8px rgba(2, 132, 199, 0.05);
  }}
  .cf-badge-green {{ background: #ccfbf1; color: #0f766e; font-size: 10px; font-weight: 700; padding: 2px 7px; border-radius: 5px; }}
  .cf-score-pill {{ background: var(--cf-grad-primary); color: #fff; font-size: 12px; font-weight: 700; padding: 3px 10px; border-radius: 12px; }}
  .cf-pill-grid {{ display: flex; flex-wrap: wrap; gap: 5px; margin: 8px 0; }}
  .cf-metric-pill {{ background: #f0f9ff; color: #0369a1; border: 1px solid #bae6fd; font-size: 10px; padding: 3px 7px; border-radius: 5px; }}
  .cf-fail-details {{ background: #f0fdfa; border: 1px solid #99f6e4; border-radius: 6px; padding: 6px 10px; margin: 8px 0; font-size: 12px; }}
  .cf-fail-details summary {{ cursor: pointer; font-weight: 700; color: #0f766e; }}
  .cf-step-box {{ background: #f8fafc; border: 1px solid #e2e8f0; border-radius: 6px; padding: 8px; margin-top: 6px; overflow-x: auto; }}
  .cf-sm-badge {{ font-size: 11px; padding: 2px 7px; border-radius: 4px; }}
  .cf-sm-buy {{ background: #ccfbf1; color: #0f766e; border: 1px solid #5eead4; }}
  .cf-sm-custom {{ background: #ffe4e6; color: #be123c; border: 1px solid #fda4af; }}
  .cf-eln-table {{ width: 100%; border-collapse: collapse; font-size: 11px; text-align: left; background: #ffffff; border: 1px solid #bae6fd; border-radius: 6px; overflow: hidden; }}
  .cf-eln-table th {{ background: #e0f2fe; color: #0369a1; font-weight: 700; padding: 6px 8px; border-bottom: 1px solid #bae6fd; }}
  .cf-eln-table td {{ padding: 5px 8px; border-bottom: 1px solid #e2e8f0; color: #1e293b; }}
  .cf-progress-card {{ background: linear-gradient(135deg, #0f172a 0%, #1e293b 100%); color: #f8fafc; border: 1px solid #0ea5e9; border-radius: 10px; padding: 14px 16px; margin: 8px 0 12px 0; }}
  .cf-progress-track {{ width: 100%; height: 12px; background: rgba(15, 23, 42, 0.85); border: 1px solid rgba(56, 189, 248, 0.35); border-radius: 999px; overflow: hidden; margin: 9px 0; }}
  .cf-progress-fill {{ height: 100%; width: 2%; background: linear-gradient(90deg, #0284c7 0%, #06b6d4 50%, #14b8a6 100%); background-size: 200% 100%; animation: cfShimmer 2s linear infinite; border-radius: 999px; transition: width 0.4s ease; }}
  @keyframes cfShimmer {{ 0% {{ background-position: 100% 0; }} 100% {{ background-position: -100% 0; }} }}
  .cf-agent-log {{ height: 420px; overflow-y: auto; border: 1px solid #bae6fd; border-radius: 8px; padding: 10px; background: #f8fafc; margin-bottom: 8px; display: flex; flex-direction: column; gap: 8px; }}
  .cf-msg-user {{ align-self: flex-end; max-width: 82%; background: var(--cf-grad-primary); color: #fff; padding: 8px 12px; border-radius: 12px 12px 2px 12px; font-size: 12px; }}
  .cf-msg-bot {{ align-self: flex-start; max-width: 92%; background: #ffffff; color: #0f172a; border: 1px solid #bae6fd; padding: 9px 12px; border-radius: 12px 12px 12px 2px; font-size: 12px; box-shadow: 0 2px 6px rgba(2, 132, 199, 0.05); }}
</style>

<div class="cf-workbench">
  <!-- LEFT PANE: 01 KETCHER STRUCTURE & CONSTRAINTS -->
  <div class="cf-pane">
    <div class="cf-sec-hdr">
      <span>01 · STRUCTURE / REACTION (EPAM KETCHER)</span>
      <span id="cf-status" style="color:#0d9488;font-weight:700;">⏳ Loading Ketcher...</span>
    </div>
    <div class="cf-input-row">
      <input id="cf-name-a" class="cf-input" placeholder="Search Drug Name, CAS, CID, ChEMBL, or SMILES (e.g. Febuxostat, DB04868)..." />
      <button class="cf-btn" onclick="cfLoadSingle()">Load</button>
    </div>
    <div class="cf-input-row">
      <input id="cf-react-a" class="cf-input" placeholder="Reactant A: name / SMILES" />
      <input id="cf-react-b" class="cf-input" placeholder="Reactant B: name / SMILES" />
      <button class="cf-btn" onclick="cfLoadPair()">Load A + B</button>
    </div>
    <div style="display:flex;gap:5px;flex-wrap:wrap;margin-bottom:6px;font-size:11px;color:#0369a1;align-items:center;">
      <b>Examples:</b>
      <button class="cf-chip" onclick="cfQuick('Nilotinib')">Nilotinib</button>
      <button class="cf-chip" onclick="cfQuick('Febuxostat')">Febuxostat</button>
      <button class="cf-chip" onclick="cfQuick('Celecoxib')">Celecoxib</button>
      <button class="cf-chip" onclick="cfQuick('Imatinib')">Imatinib</button>
    </div>
    <iframe id="cf-ketcher-frame" style="width:100%;height:430px;border:1px solid #7dd3fc;border-radius:8px;background:#fff;"></iframe>
    <div class="cf-input-row" style="margin-top:6px;">
      <input id="cf-smiles-box" class="cf-input" style="font-family:monospace;font-size:11px;" value="{default_smi}" />
      <button class="cf-btn cf-btn-green" onclick="cfSyncCanvas(false)">🔄 Sync Canvas</button>
      <button class="cf-btn cf-btn-amber" onclick="cfSyncCanvas(true)">🔒 Lock Selected</button>
    </div>
    <div id="cf-desc-bar" class="cf-desc-grid"></div>
    <details style="margin-top:8px;font-size:11px;border:1px solid #bae6fd;border-radius:6px;padding:6px 10px;background:#f0f9ff;">
      <summary style="cursor:pointer;font-weight:700;color:#0369a1;">🔒 Atom-Indexed Preview, Substructure Lock &amp; Lab Inventory Stock</summary>
      <div id="cf-atom-svg" style="background:#fff;border:1px solid #bae6fd;border-radius:6px;margin:6px 0;text-align:center;"></div>
      <div class="cf-input-row">
        <input id="cf-lock-smarts" class="cf-input" style="font-family:monospace;" placeholder="Frozen Substructure SMARTS (auto-filled when locking atoms or type e.g. c1ncsc1)" />
      </div>
      <textarea id="cf-stock-input" rows="2" class="cf-input" style="width:100%;box-sizing:border-box;font-family:monospace;margin-top:4px;" placeholder="Paste in-house Lab Stock SMILES (one per line)..."></textarea>
      <div style="display:flex;justify-content:space-between;align-items:center;margin-top:4px;">
        <span id="cf-stock-msg" style="color:#0d9488;font-weight:600;"></span>
        <button class="cf-btn" style="padding:4px 10px;font-size:11px;" onclick="cfRegisterStock()">📦 Register Lab Stock</button>
      </div>
    </details>
  </div>

  <!-- RIGHT PANE: 02 CONFIGURE, ANALYZE & ORGANIC CHEMIST AI AGENT -->
  <div class="cf-pane">
    <div class="cf-sec-hdr">
      <span>02 · CONFIGURE, ONLINE ORD VALIDATION &amp; AI AGENT</span>
      <span style="display:flex;align-items:center;gap:6px;">
        <span id="cf-ord-badge" style="font-size:10px;background:#ccfbf1;color:#0f766e;padding:2px 7px;border-radius:10px;font-weight:700;">ORD Index: {init_ord_cnt} rxns</span>
        <span id="cf-ram-badge" style="font-size:10px;background:#e0f2fe;color:#0369a1;padding:2px 7px;border-radius:10px;font-weight:700;">RAM: {init_ram_mb} MB</span>
        <button class="cf-chip" style="padding:2px 7px;font-size:10px;" onclick="cfFreeMemory()" title="Purge search tree, trim C++ heap &amp; run GC">🧹 Free RAM</button>
      </span>
    </div>

    <!-- GLOBAL GOOGLE GENAI MODEL & THINKING LEVEL SELECTOR -->
    <div class="cf-model-bar">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:5px;font-size:10.5px;font-weight:800;color:#0369a1;">
        <span>🧠 GLOBAL GOOGLE GENAI MODEL &amp; THINKING LEVEL (APPLIES TO ALL TABS)</span>
        <span id="cf-active-model-pill" style="background:#ccfbf1;color:#0f766e;padding:1px 7px;border-radius:10px;font-size:10px;">gemini-3.8-flash · medium</span>
      </div>
      <div class="cf-model-row">
        <div>
          <select id="cf-global-model" class="cf-model-select" onchange="cfOnModelChange(true)">
            {model_options_html}
          </select>
        </div>
        <div>
          <select id="cf-global-thinking" class="cf-model-select" onchange="cfOnThinkingChange()">
            <option value="low">Thinking: Low (Fast)</option>
            <option value="medium" selected>Thinking: Medium (Balanced)</option>
            <option value="high">Thinking: High (Deep)</option>
          </select>
        </div>
      </div>
      <div class="cf-model-summary">
        <span id="cf-model-summary-text">ℹ️ {default_summary}</span>
      </div>
    </div>

    <div class="cf-tabs">
      <button id="cf-tab-btn-synthesis" class="cf-tab active" onclick="cfSwitchTab('synthesis', this)">Synthesis</button>
      <button id="cf-tab-btn-agent" class="cf-tab" onclick="cfSwitchTab('agent', this)">🤖 Chemist AI Agent</button>
      <button id="cf-tab-btn-reaction" class="cf-tab" onclick="cfSwitchTab('reaction', this)">Reaction</button>
      <button id="cf-tab-btn-qa" class="cf-tab" onclick="cfSwitchTab('qa', this)">Structure Q&amp;A</button>
      <button id="cf-tab-btn-analytical" class="cf-tab" onclick="cfSwitchTab('analytical', this)">Analytical</button>
    </div>

    <!-- TAB 1: SYNTHESIS + INLINE SEARCH CONFIGURATION -->
    <div id="cf-panel-synthesis">
      <div style="display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:8px;margin-bottom:8px;">
        <div style="display:flex;gap:5px;align-items:center;font-size:11px;flex-wrap:wrap;">
          <b style="color:#0369a1;">RANK BY:</b>
          <button class="cf-mode-pill active" onclick="cfSetRankMode('cro', this)">CRO-Practical</button>
          <button class="cf-mode-pill" onclick="cfSetRankMode('discovery', this)">Discovery (Fast)</button>
          <button class="cf-mode-pill" onclick="cfSetRankMode('evidence', this)" title="Weights Route-Level ORD Empirical Confidence Score (S_ORD) highest">Evidence-Backed (ORD)</button>
          <button class="cf-mode-pill" onclick="cfSetRankMode('green', this)">Green / Safe</button>
        </div>
        <div style="display:flex;gap:6px;flex-wrap:wrap;">
          <button id="cf-plan-btn" class="cf-btn cf-btn-green" style="padding:7px 14px;font-size:12.5px;" onclick="cfRunSynthesis()">🚀 Plan Synthesis Route</button>
          <button id="cf-reclass-btn" class="cf-btn cf-btn-amber" style="padding:7px 12px;font-size:12px;" onclick="cfReclassifyRoutes()" title="Re-validate ORD precedents, re-classify existing routes &amp; rewrite experimental procedures using the selected model without re-running AiZynthFinder">🧠 Re-classify &amp; Re-score (ORD + AI)</button>
        </div>
      </div>
      <details style="margin-bottom:10px;">
        <summary style="cursor:pointer;font-size:11px;font-weight:700;color:#0284c7;margin-bottom:6px;">⚙️ Search, ORD Parquet Shard Sync &amp; Scoring Settings</summary>
        <div class="cf-cfg-grid">
          <div class="cf-cfg-item">
            <label>Search Algorithm</label>
            <select id="cfg-algo">
              <option value="retrostar" selected>Retro* (AND-OR Best-First)</option>
              <option value="mcts">MCTS (Monte Carlo Tree Search)</option>
            </select>
          </div>
          <div class="cf-cfg-item">
            <label>Max Synthetic Steps</label>
            <select id="cfg-steps">
              <option value="3">3 steps</option><option value="4">4 steps</option><option value="5">5 steps</option>
              <option value="6" selected>6 steps</option><option value="8">8 steps</option><option value="10">10 steps</option>
            </select>
          </div>
          <div class="cf-cfg-item">
            <label>Search Time Limit (sec)</label>
            <select id="cfg-time">
              <option value="30">30s</option><option value="60">60s</option><option value="120" selected>120s</option><option value="300">300s</option>
            </select>
          </div>
          <div class="cf-cfg-item">
            <label>Iteration Limit</label>
            <select id="cfg-iter">
              <option value="50">50</option><option value="100" selected>100</option><option value="200">200</option><option value="500">500</option>
            </select>
          </div>
          <div class="cf-cfg-item">
            <label>Template Cutoff</label>
            <select id="cfg-cutoff">
              <option value="0.90">0.90 (Conservative)</option><option value="0.95">0.95 (Standard)</option><option value="0.995" selected>0.995 (Broad)</option>
            </select>
          </div>
          <div class="cf-cfg-item">
            <label>Target Scale (g) &amp; Max Routes</label>
            <div style="display:flex;gap:4px;">
              <input id="cfg-scale" type="number" step="0.5" value="1.0" />
              <input id="cfg-max-routes" type="number" value="8" />
            </div>
          </div>
          <div class="cf-cfg-item">
            <label>Exclude Reaction Types (comma-separated)</label>
            <input id="cfg-exclude" placeholder="e.g. Stille, Azide, Ozonolysis, Grignard" />
          </div>
          <div class="cf-cfg-item">
            <label>ORD Parquet Shards (`ord-data` HF)</label>
            <div style="display:flex;gap:4px;align-items:center;">
              <button id="cf-ord-sync-btn" class="cf-btn" style="padding:4px 8px;font-size:10.5px;width:100%;" onclick="cfSyncOrdShards(1)" title="Stream &amp; index an ORD .parquet shard from Hugging Face into /dev/shm DuckDB">📥 Sync +1 HF ORD Parquet Shard</button>
            </div>
          </div>
          <div class="cf-cfg-item" style="display:flex;flex-direction:column;justify-content:center;gap:4px;">
            <label style="display:flex;align-items:center;gap:4px;cursor:pointer;"><input type="checkbox" id="cfg-first" style="width:auto;" /> Stop at First Solved</label>
            <label style="display:flex;align-items:center;gap:4px;cursor:pointer;"><input type="checkbox" id="cfg-gemini" checked style="width:auto;" /> Gemini + ORD Grounding</label>
          </div>
        </div>
      </details>
      <div id="cf-synthesis-out" style="font-size:12px;color:#475569;">
        Draw or load your target molecule on the left, adjust any search settings above, and click <b>🚀 Plan Synthesis Route</b>. Every unique step is prefetched in parallel against the live Open Reaction Database API and validated in DuckDB for reaction-center similarity, HTE negative-data failure rates, grounded forward conditions, and a Route-Level <b>ORD Empirical Confidence Score</b>.
      </div>
    </div>

    <!-- TAB 2: ORGANIC CHEMIST AI AGENT -->
    <div id="cf-panel-agent" style="display:none;">
      <div style="display:flex;justify-content:space-between;align-items:center;background:#f0f9ff;border:1px solid #bae6fd;border-radius:7px;padding:7px 10px;margin-bottom:8px;font-size:11px;">
        <span id="cf-agent-ctx-bar" style="color:#0369a1;font-weight:700;">🤖 Session: Ready — run synthesis or ask a question below</span>
        <button class="cf-chip" style="padding:2px 8px;font-size:10px;" onclick="cfResetAgentChat()">🗑️ Clear Chat</button>
      </div>
      <div style="display:flex;gap:5px;flex-wrap:wrap;margin-bottom:8px;">
        <button class="cf-chip" onclick="cfAskAgentQuick('Compare the top retrosynthesis routes by ORD Empirical Confidence Score, bottleneck steps, and HTE failure risks.')">🔬 Compare by ORD Confidence</button>
        <button class="cf-chip" onclick="cfAskAgentQuick('Generate a 5.0 g scaled ELN stoichiometry table for Route 1.')">📋 5g ELN Table (Route A)</button>
        <button class="cf-chip" onclick="cfAskAgentQuick('Which steps have ORD negative-data alerts or competing side-products, and how can we mitigate them?')">⚠️ Audit ORD Negative Data</button>
        <button class="cf-chip" onclick="cfAskAgentQuick('Re-run the retrosynthesis in balanced (Evidence-Backed) mode with max 5 steps.')">🔄 Re-run (Evidence Mode, ≤5 Steps)</button>
      </div>
      <div id="cf-agent-log" class="cf-agent-log">
        <div class="cf-msg-bot">
          <b>👨‍🔬 Organic Chemist AI Agent:</b> Welcome! Once you run a synthesis (or right now), ask me to compare routes by their <b>ORD Empirical Confidence Score</b>, audit HTE negative-data failure rates, generate scaled ELN stoichiometry tables, or re-run AiZynthFinder with new reaction constraints.
        </div>
      </div>
      <div class="cf-input-row">
        <input id="cf-agent-input" class="cf-input" placeholder="Chat with the Organic Chemist AI Agent about ORD precedents, routes, conditions, ELN scale-up, or re-running..." onkeydown="if(event.key==='Enter') cfSendAgentMsg();" />
        <button id="cf-agent-send-btn" class="cf-btn cf-btn-green" onclick="cfSendAgentMsg()">Send</button>
      </div>
    </div>

    <!-- TAB 3: REACTION ANALYSIS -->
    <div id="cf-panel-reaction" style="display:none;">
      <div style="margin-bottom:10px;display:flex;justify-content:space-between;align-items:center;">
        <span style="font-size:12px;color:#475569;">Draw a full reaction (<code>Reactants &gt;&gt; Product</code>) in Ketcher or paste in the SMILES box.</span>
        <button class="cf-btn" onclick="cfRunReaction()">Analyze Reaction + ORD Precedent</button>
      </div>
      <div id="cf-reaction-out"></div>
    </div>

    <!-- TAB 4: STRUCTURE Q&A & LITERATURE CHAT -->
    <div id="cf-panel-qa" style="display:none;">
      <div class="cf-input-row">
        <input id="cf-qa-input" class="cf-input" placeholder="Ask about route comparison, ORD yields, literature conditions, or side-products..." />
        <button class="cf-btn" onclick="cfRunQA()">Ask Gemini</button>
      </div>
      <div id="cf-qa-out"></div>
    </div>

    <!-- TAB 5: ANALYTICAL LC-MS & REPORT READER -->
    <div id="cf-panel-analytical" style="display:none;">
      <textarea id="cf-analytical-input" rows="3" class="cf-input" style="width:100%;box-sizing:border-box;margin-bottom:6px;font-family:monospace;" placeholder="Optional: Paste raw LC-MS / NMR / HPLC peak text here, or leave blank to compute expected monoisotopic adduct ions..."></textarea>
      <button class="cf-btn" onclick="cfRunAnalytical()">Compute Ions / Read Report</button>
      <div id="cf-analytical-out" style="margin-top:8px;"></div>
    </div>
  </div>
</div>

<script>
window.cfRankMode = 'cro';
window.cfModelCatalog = {catalog_json};

const THINKING_LABELS = {{
  minimal: "Thinking: Minimal (Fastest)",
  low: "Thinking: Low (Fast)",
  medium: "Thinking: Medium (Balanced)",
  high: "Thinking: High (Deep Reasoning)",
  xhigh: "Thinking: XHigh (Max Frontier)"
}};

async function cfOnModelChange(resetToDefaultThinking) {{
  const mSel = document.getElementById("cf-global-model");
  const tSel = document.getElementById("cf-global-thinking");
  const sumEl = document.getElementById("cf-model-summary-text");
  const pillEl = document.getElementById("cf-active-model-pill");
  const modelId = mSel.value;
  const spec = window.cfModelCatalog[modelId] || window.cfModelCatalog["gemini-3.8-flash"];
  const prevThinking = tSel.value;
  const supported = spec.supported_thinking || ["low", "medium", "high"];
  const targetThinking = resetToDefaultThinking
    ? spec.default_thinking
    : (supported.includes(prevThinking) ? prevThinking : spec.default_thinking);

  tSel.innerHTML = supported.map(lvl => {{
    const isDef = (lvl === spec.default_thinking) ? " · Default" : "";
    const selAttr = (lvl === targetThinking) ? "selected" : "";
    return `<option value="${{lvl}}" ${{selAttr}}>${{THINKING_LABELS[lvl] || lvl}}${{isDef}}</option>`;
  }}).join("");

  if (sumEl) sumEl.innerText = "ℹ️ " + spec.summary;
  if (pillEl) pillEl.innerText = `${{modelId}} · ${{tSel.value}}`;
  try {{
    await google.colab.kernel.invokeFunction("chemforge.set_model", [modelId, tSel.value], {{}});
  }} catch (e) {{}}
}}

async function cfOnThinkingChange() {{
  const mSel = document.getElementById("cf-global-model");
  const tSel = document.getElementById("cf-global-thinking");
  const pillEl = document.getElementById("cf-active-model-pill");
  if (pillEl) pillEl.innerText = `${{mSel.value}} · ${{tSel.value}}`;
  try {{
    await google.colab.kernel.invokeFunction("chemforge.set_model", [mSel.value, tSel.value], {{}});
  }} catch (e) {{}}
}}

(async function initKetcherFrame() {{
  const frame = document.getElementById("cf-ketcher-frame");
  const statusEl = document.getElementById("cf-status");
  let b64 = "{ketcher_b64_html}";
  try {{
    const bin = atob(b64);
    const bytes = new Uint8Array(bin.length);
    for (let i = 0; i < bin.length; i++) bytes[i] = bin.charCodeAt(i);
    frame.srcdoc = new TextDecoder("utf-8").decode(bytes);
    b64 = null;
  }} catch (err) {{
    console.error("Ketcher srcdoc init error:", err);
  }}
  let attempts = 0;
  const timer = setInterval(async () => {{
    attempts++;
    try {{
      const cw = frame.contentWindow;
      if (cw && cw.ketcher && typeof cw.ketcher.setMolecule === "function") {{
        clearInterval(timer);
        window.cfKetcher = cw.ketcher;
        statusEl.innerText = "✅ Ketcher Ready";
        const initSmi = document.getElementById("cf-smiles-box").value.trim();
        if (initSmi) {{
          await cw.ketcher.setMolecule(initSmi);
          cfSyncCanvas(false);
        }}
        return;
      }}
    }} catch (e) {{}}
    if (attempts >= 60) {{
      clearInterval(timer);
    }}
  }}, 350);
  cfOnModelChange(true);
  cfSyncCanvas(false);
}})();

function cfUpdateRamBadge(ramMb) {{
  if (!ramMb) return;
  const el = document.getElementById("cf-ram-badge");
  if (el) el.innerText = `RAM: ${{ramMb}} MB`;
}}

function cfUpdateOrdBadge(ordCount, prefetch) {{
  if (!ordCount) return;
  const el = document.getElementById("cf-ord-badge");
  const statusEl = document.getElementById("cf-status");
  if (prefetch && prefetch.queried_steps > 0) {{
    const addCnt = prefetch.added_records || 0;
    const sec = Number(prefetch.elapsed_sec || 0).toFixed(2);
    if (el) {{
      el.innerText = `ORD Index: ${{ordCount}} rxns (+${{addCnt}} live · ${{sec}}s)`;
      el.title = `Queried ${{prefetch.queried_steps}} unique steps against live open-reaction-database.org API in ${{sec}}s`;
    }}
    if (statusEl) {{
      statusEl.innerText = `🌐 +${{addCnt}} live ORD hits (${{prefetch.queried_steps}} steps in ${{sec}}s)`;
    }}
  }} else {{
    if (el) el.innerText = `ORD Index: ${{ordCount}} rxns`;
  }}
}}

async function cfSyncOrdShards(nShards) {{
  const btn = document.getElementById("cf-ord-sync-btn");
  if (btn) {{
    btn.disabled = true;
    btn.innerText = "⏳ Streaming HF Parquet Shard...";
  }}
  try {{
    const res = await google.colab.kernel.invokeFunction("chemforge.sync_ord_shards", [nShards || 1], {{}});
    const data = res.data["application/json"];
    cfUpdateOrdBadge(data.indexed_count);
    cfUpdateRamBadge(data.ram_mb);
    if (btn) {{
      btn.innerText = data.error ? "⚠️ Sync Error (Retry)" : `✅ Indexed (${{data.indexed_count}} rxns) · +1 Shard`;
    }}
  }} catch (err) {{
    if (btn) btn.innerText = "⚠️ Sync Failed";
  }} finally {{
    if (btn) btn.disabled = false;
  }}
}}

async function cfFreeMemory() {{
  const res = await google.colab.kernel.invokeFunction("chemforge.free_memory", [], {{}});
  const data = res.data["application/json"];
  cfUpdateRamBadge(data.ram_mb);
  document.getElementById("cf-status").innerText = data.message;
}}

function cfRenderDescriptors(d) {{
  const el = document.getElementById("cf-desc-bar");
  if (!d || d.error) {{
    el.innerHTML = `<div style="grid-column:span 4;color:#dc2626;">⚠️ ${{d ? d.error : 'Invalid structure'}}</div>`;
    return;
  }}
  const ro5 = d.ro5_pass
    ? `<span style="color:#0d9488;font-weight:700;">PASS</span>`
    : `<span style="color:#dc2626;font-weight:700;">FAIL (${{d.ro5_flags}})</span>`;
  el.innerHTML = `
    <div class="cf-desc-item"><b>FORMULA</b>${{d.formula}}</div>
    <div class="cf-desc-item"><b>MOL. WEIGHT</b>${{d.mw}}</div>
    <div class="cf-desc-item"><b>EXACT MASS</b>${{d.exact_mass}}</div>
    <div class="cf-desc-item"><b>CLOGP</b>${{d.clogp}}</div>
    <div class="cf-desc-item"><b>TPSA</b>${{d.tpsa}} Å²</div>
    <div class="cf-desc-item"><b>HBD / HBA</b>${{d.hbd}} / ${{d.hba}}</div>
    <div class="cf-desc-item"><b>ROT / RINGS</b>${{d.rotatable}} / ${{d.rings}}</div>
    <div class="cf-desc-item"><b>LIPINSKI RO5</b>${{ro5}}</div>
  `;
}}

async function cfSyncCanvas(lockSelected) {{
  let structStr = document.getElementById("cf-smiles-box").value.trim();
  let selAtoms = [];
  if (window.cfKetcher) {{
    try {{
      const kSmi = await window.cfKetcher.getSmiles();
      if (kSmi && kSmi.trim()) structStr = kSmi.trim();
      if (lockSelected) {{
        structStr = await window.cfKetcher.getMolfile();
        const sel = window.cfKetcher.editor ? window.cfKetcher.editor.selection() : null;
        if (sel && Array.isArray(sel.atoms)) selAtoms = sel.atoms;
      }}
    }} catch (e) {{}}
  }}
  const manualSmarts = lockSelected ? "" : document.getElementById("cf-lock-smarts").value.trim();
  const res = await google.colab.kernel.invokeFunction("chemforge.sync_canvas", [structStr, JSON.stringify(selAtoms), manualSmarts], {{}});
  const data = res.data["application/json"];
  if (!data.error) {{
    document.getElementById("cf-smiles-box").value = data.smiles;
    document.getElementById("cf-lock-smarts").value = data.locked_smarts || "";
    document.getElementById("cf-atom-svg").innerHTML = data.preview_svg || "";
    cfRenderDescriptors(data.descriptors);
    cfUpdateRamBadge(data.ram_mb);
    document.getElementById("cf-status").innerText = data.locked_atoms && data.locked_atoms.length
      ? `🔒 Locked ${{data.locked_atoms.length}} atoms`
      : "✅ Synced with Ketcher";
  }}
}}

async function cfLoadSingle() {{
  const q = document.getElementById("cf-name-a").value.trim();
  if (!q) return;
  document.getElementById("cf-status").innerText = "⏳ Resolving...";
  const res = await google.colab.kernel.invokeFunction("chemforge.load_id", [q, ""], {{}});
  const data = res.data["application/json"];
  if (data.error) {{
    document.getElementById("cf-status").innerText = "⚠️ " + data.error;
  }} else {{
    document.getElementById("cf-smiles-box").value = data.smiles;
    if (window.cfKetcher) await window.cfKetcher.setMolecule(data.smiles);
    cfRenderDescriptors(data.descriptors);
    document.getElementById("cf-status").innerText = "✅ " + data.source;
  }}
}}

function cfQuick(name) {{
  document.getElementById("cf-name-a").value = name;
  cfLoadSingle();
}}

async function cfLoadPair() {{
  const qa = document.getElementById("cf-react-a").value.trim();
  const qb = document.getElementById("cf-react-b").value.trim();
  if (!qa || !qb) return;
  document.getElementById("cf-status").innerText = "⏳ Loading A + B...";
  const res = await google.colab.kernel.invokeFunction("chemforge.load_id", [qa, qb], {{}});
  const data = res.data["application/json"];
  if (!data.error) {{
    document.getElementById("cf-smiles-box").value = data.smiles;
    if (window.cfKetcher) await window.cfKetcher.setMolecule(data.smiles);
    cfRenderDescriptors(data.descriptors);
    document.getElementById("cf-status").innerText = "✅ " + data.source;
  }}
}}

function cfSwitchTab(tab, btn) {{
  ["synthesis", "agent", "reaction", "qa", "analytical"].forEach(t => {{
    document.getElementById("cf-panel-" + t).style.display = (t === tab) ? "block" : "none";
  }});
  btn.parentElement.querySelectorAll(".cf-tab").forEach(b => b.classList.remove("active"));
  btn.classList.add("active");
  if (tab === "agent") cfRefreshAgentStatus();
}}

function cfOpenAgentWithRoutes() {{
  const btn = document.getElementById("cf-tab-btn-agent");
  if (btn) cfSwitchTab("agent", btn);
}}

function cfSetRankMode(mode, btn) {{
  window.cfRankMode = mode;
  btn.parentElement.querySelectorAll(".cf-mode-pill").forEach(b => b.classList.remove("active"));
  btn.classList.add("active");
}}

async function cfRefreshAgentStatus() {{
  const res = await google.colab.kernel.invokeFunction("chemforge.agent_status", [], {{}});
  const d = res.data["application/json"];
  cfUpdateRamBadge(d.ram_mb);
  cfUpdateOrdBadge(d.ord_index_size);
  const bar = document.getElementById("cf-agent-ctx-bar");
  if (bar) {{
    bar.innerText = d.route_count > 0
      ? `✅ Active Session: ${{d.route_count}} ORD-validated routes (${{d.ranking_mode}}) · Model: ${{d.model}} (${{d.thinking_level}}) | Target: ${{d.target_smiles.slice(0, 30)}}...`
      : `🤖 Active Model: ${{d.model}} (${{d.thinking_level}}) | ORD Index: ${{d.ord_index_size}} rxns | Target: ${{d.target_smiles.slice(0, 30)}}`;
  }}
}}

async function cfRunSynthesis() {{
  await cfSyncCanvas(false);
  const smi = document.getElementById("cf-smiles-box").value.trim();
  const timeLimitSec = parseInt(document.getElementById("cfg-time").value, 10) || 120;
  const maxIter = parseInt(document.getElementById("cfg-iter").value, 10) || 100;
  const algoLabel = document.getElementById("cfg-algo").value === "mcts" ? "MCTS" : "Retro*";
  const useGem = document.getElementById("cfg-gemini").checked;
  const selModel = document.getElementById("cf-global-model").value;
  const selThinking = document.getElementById("cf-global-thinking").value;
  const estTotalSec = Math.min(timeLimitSec + (useGem ? 25 : 15), Math.max(30, Math.round(timeLimitSec * 0.65) + (useGem ? 20 : 15)));
  const cfg = {{
    rank_mode: window.cfRankMode,
    algorithm: document.getElementById("cfg-algo").value,
    max_steps: parseInt(document.getElementById("cfg-steps").value, 10),
    time_limit: timeLimitSec,
    iteration_limit: maxIter,
    cutoff: parseFloat(document.getElementById("cfg-cutoff").value),
    scale_grams: parseFloat(document.getElementById("cfg-scale").value || "1.0"),
    max_routes: parseInt(document.getElementById("cfg-max-routes").value || "8", 10),
    exclude_reactions: document.getElementById("cfg-exclude").value,
    return_first: document.getElementById("cfg-first").checked,
    use_gemini: useGem,
    genai_model: selModel,
    genai_thinking: selThinking,
    lock_smarts: document.getElementById("cf-lock-smarts").value.trim()
  }};
  const out = document.getElementById("cf-synthesis-out");
  const planBtn = document.getElementById("cf-plan-btn");
  const reclassBtn = document.getElementById("cf-reclass-btn");
  if (planBtn) planBtn.disabled = true;
  if (reclassBtn) reclassBtn.disabled = true;
  out.innerHTML = `
    <div class="cf-progress-card">
      <div style="display:flex;justify-content:space-between;align-items:center;font-size:12px;font-weight:700;">
        <span id="cf-prog-stage" style="color:#38bdf8;">⚡ Running ${{algoLabel}} Tree Search (USPTO + ZINC Stock + Online ORD Validator)...</span>
        <span id="cf-prog-pct" style="color:#2dd4bf;font-family:monospace;font-size:13px;">0%</span>
      </div>
      <div class="cf-progress-track"><div id="cf-prog-fill" class="cf-progress-fill" style="width:2%;"></div></div>
      <div style="display:flex;justify-content:space-between;align-items:center;font-size:11px;color:#94a3b8;font-family:monospace;">
        <span id="cf-prog-elapsed">Elapsed: 0s</span>
        <span id="cf-prog-eta">Est. remaining: ~${{estTotalSec}}s (Max cap: ${{timeLimitSec}}s)</span>
      </div>
    </div>
  `;
  const startMs = Date.now();
  const progTimer = setInterval(() => {{
    const elapsedSec = Math.floor((Date.now() - startMs) / 1000);
    const fillEl = document.getElementById("cf-prog-fill");
    const pctEl = document.getElementById("cf-prog-pct");
    const elapEl = document.getElementById("cf-prog-elapsed");
    const stageEl = document.getElementById("cf-prog-stage");
    if (!fillEl) return;
    let ratio = elapsedSec / estTotalSec;
    let pct = ratio < 0.85 ? Math.min(85, Math.round(ratio * 100)) : Math.min(96, 85 + Math.round((1 - Math.exp(-(elapsedSec - estTotalSec * 0.85) / 15)) * 11));
    fillEl.style.width = pct + "%";
    pctEl.innerText = pct + "%";
    elapEl.innerText = `Elapsed: ${{elapsedSec}}s`;
    if (elapsedSec < estTotalSec * 0.65) {{
      stageEl.innerText = `⚡ Exploring ${{algoLabel}} disconnection tree (up to ${{maxIter}} iterations)...`;
    }} else if (elapsedSec < estTotalSec * 0.65 + 4) {{
      stageEl.innerText = `🌐 Parallel 16-thread online ORD API prefetch (matching reactants & products across 2M+ rxns)...`;
    }} else {{
      stageEl.innerText = useGem
        ? `🔬 Grounding experimental procedures in live ORD precedents with ${{selModel}}...`
        : `🔬 Computing ORD Empirical Confidence Scores & purging tree memory...`;
    }}
  }}, 500);
  try {{
    const res = await google.colab.kernel.invokeFunction("chemforge.run_synthesis", [smi, JSON.stringify(cfg)], {{}});
    clearInterval(progTimer);
    const data = res.data["application/json"];
    out.innerHTML = data.error ? `<div style="color:#dc2626;font-weight:600;">⚠️ ${{data.error}}</div>` : data.html;
    cfUpdateRamBadge(data.ram_mb);
    cfUpdateOrdBadge(data.ord_index_size, data.ord_prefetch);
    cfRefreshAgentStatus();
  }} catch (err) {{
    clearInterval(progTimer);
    out.innerHTML = `<div style="color:#dc2626;font-weight:600;">⚠️ Kernel error: ${{err}}</div>`;
  }} finally {{
    if (planBtn) planBtn.disabled = false;
    if (reclassBtn) reclassBtn.disabled = false;
  }}
}}

async function cfReclassifyRoutes() {{
  const selModel = document.getElementById("cf-global-model").value;
  const selThinking = document.getElementById("cf-global-thinking").value;
  const out = document.getElementById("cf-synthesis-out");
  const reclassBtn = document.getElementById("cf-reclass-btn");
  const planBtn = document.getElementById("cf-plan-btn");
  if (reclassBtn) reclassBtn.disabled = true;
  if (planBtn) planBtn.disabled = true;
  document.getElementById("cfg-gemini").checked = true;
  const oldBanner = document.getElementById("cf-reclass-banner");
  if (oldBanner) oldBanner.remove();
  out.insertAdjacentHTML("afterbegin", `
    <div id="cf-reclass-banner" class="cf-progress-card" style="margin-bottom:10px;">
      <div style="display:flex;justify-content:space-between;align-items:center;font-size:12px;font-weight:700;">
        <span style="color:#38bdf8;">🌐 Prefetching uncached online ORD steps (16 threads) &amp; rewriting procedures with ${{selModel}} (${{selThinking}})...</span>
        <span style="color:#2dd4bf;font-family:monospace;font-size:11px;">Zero tree-search overhead</span>
      </div>
      <div class="cf-progress-track"><div class="cf-progress-fill" style="width:100%;"></div></div>
    </div>
  `);
  try {{
    const res = await google.colab.kernel.invokeFunction("chemforge.reclassify_routes", [selModel, selThinking, window.cfRankMode], {{}});
    const data = res.data["application/json"];
    const banner = document.getElementById("cf-reclass-banner");
    if (banner) banner.remove();
    if (data.error) {{
      out.insertAdjacentHTML("afterbegin", `<div style="color:#dc2626;font-weight:600;margin-bottom:8px;">⚠️ ${{data.error}}</div>`);
    }} else {{
      out.innerHTML = data.html;
    }}
    cfUpdateRamBadge(data.ram_mb);
    cfUpdateOrdBadge(data.ord_index_size, data.ord_prefetch);
    cfRefreshAgentStatus();
  }} catch (err) {{
    const banner = document.getElementById("cf-reclass-banner");
    if (banner) banner.remove();
    out.insertAdjacentHTML("afterbegin", `<div style="color:#dc2626;font-weight:600;margin-bottom:8px;">⚠️ Kernel error: ${{err}}</div>`);
  }} finally {{
    if (reclassBtn) reclassBtn.disabled = false;
    if (planBtn) planBtn.disabled = false;
  }}
}}

async function cfSendAgentMsg() {{
  const inp = document.getElementById("cf-agent-input");
  const log = document.getElementById("cf-agent-log");
  const sendBtn = document.getElementById("cf-agent-send-btn");
  const msg = inp.value.trim();
  if (!msg) return;
  inp.value = "";
  const smi = document.getElementById("cf-smiles-box").value.trim();
  const selModel = document.getElementById("cf-global-model").value;
  const selThinking = document.getElementById("cf-global-thinking").value;
  log.insertAdjacentHTML("beforeend", `<div class="cf-msg-user">${{msg.replace(/</g, "&lt;")}}</div>`);
  const waitId = "cf-wait-" + Date.now();
  log.insertAdjacentHTML("beforeend", `<div id="${{waitId}}" class="cf-msg-bot">⏳ <b>Chemist AI Agent</b> (<code>${{selModel}}</code> · <code>${{selThinking}}</code>) is analyzing ORD-validated routes &amp; running tools...</div>`);
  log.scrollTop = log.scrollHeight;
  if (sendBtn) sendBtn.disabled = true;
  try {{
    const res = await google.colab.kernel.invokeFunction("chemforge.agent_chat", [msg, smi, selModel, selThinking], {{}});
    const data = res.data["application/json"];
    const waitEl = document.getElementById(waitId);
    if (waitEl) waitEl.outerHTML = `<div class="cf-msg-bot">${{data.reply_html}}</div>`;
    if (data.updated_synthesis_html) {{
      document.getElementById("cf-synthesis-out").innerHTML = data.updated_synthesis_html;
    }}
    cfUpdateRamBadge(data.ram_mb);
    cfRefreshAgentStatus();
  }} catch (err) {{
    const waitEl = document.getElementById(waitId);
    if (waitEl) waitEl.outerHTML = `<div class="cf-msg-bot" style="color:#dc2626;">⚠️ Error: ${{err}}</div>`;
  }} finally {{
    if (sendBtn) sendBtn.disabled = false;
    log.scrollTop = log.scrollHeight;
  }}
}}

function cfAskAgentQuick(promptText) {{
  document.getElementById("cf-agent-input").value = promptText;
  cfSendAgentMsg();
}}

async function cfResetAgentChat() {{
  const res = await google.colab.kernel.invokeFunction("chemforge.agent_reset", [], {{}});
  const data = res.data["application/json"];
  cfUpdateRamBadge(data.ram_mb);
  document.getElementById("cf-agent-log").innerHTML = `<div class="cf-msg-bot">🧹 Chat history and cached objects cleared. Ask a new question about your ORD-validated routes!</div>`;
}}

async function cfRunReaction() {{
  if (window.cfKetcher) {{
    try {{
      const kSmi = await window.cfKetcher.getSmiles();
      if (kSmi) document.getElementById("cf-smiles-box").value = kSmi;
    }} catch (e) {{}}
  }}
  const smi = document.getElementById("cf-smiles-box").value.trim();
  const selModel = document.getElementById("cf-global-model").value;
  const selThinking = document.getElementById("cf-global-thinking").value;
  const out = document.getElementById("cf-reaction-out");
  out.innerHTML = `⏳ <b style='color:#0284c7;'>Prefetching live online ORD precedents &amp; analyzing with ${{selModel}} (${{selThinking}})...</b>`;
  const res = await google.colab.kernel.invokeFunction("chemforge.analyze_reaction", [smi, selModel, selThinking], {{}});
  const data = res.data["application/json"];
  out.innerHTML = data.error ? `<div style="color:#dc2626;">⚠️ ${{data.error}}</div>` : data.html;
}}

async function cfRunQA() {{
  const smi = document.getElementById("cf-smiles-box").value.trim();
  const q = document.getElementById("cf-qa-input").value.trim();
  const selModel = document.getElementById("cf-global-model").value;
  const selThinking = document.getElementById("cf-global-thinking").value;
  const out = document.getElementById("cf-qa-out");
  out.innerHTML = `⏳ <b style='color:#0284c7;'>Querying ${{selModel}} (${{selThinking}}) with active structure &amp; ORD-validated routes...</b>`;
  const res = await google.colab.kernel.invokeFunction("chemforge.structure_qa", [smi, q, selModel, selThinking], {{}});
  out.innerHTML = res.data["application/json"].html;
}}

async function cfRunAnalytical() {{
  const smi = document.getElementById("cf-smiles-box").value.trim();
  const rep = document.getElementById("cf-analytical-input").value.trim();
  const selModel = document.getElementById("cf-global-model").value;
  const selThinking = document.getElementById("cf-global-thinking").value;
  const out = document.getElementById("cf-analytical-out");
  out.innerHTML = `⏳ <b style='color:#0284c7;'>Computing exact monoisotopic LC-MS adducts &amp; running ${{selModel}} (${{selThinking}})...</b>`;
  const res = await google.colab.kernel.invokeFunction("chemforge.analytical", [smi, rep, selModel, selThinking], {{}});
  out.innerHTML = res.data["application/json"].html;
}}

async function cfRegisterStock() {{
  const raw = document.getElementById("cf-stock-input").value;
  const res = await google.colab.kernel.invokeFunction("chemforge.register_stock", [raw], {{}});
  document.getElementById("cf-stock-msg").innerText = res.data["application/json"].message;
}}
</script>
"""
    clear_output(wait=True)
    display(HTML(unified_workbench_html))

    del ketcher_b64_html
    del unified_workbench_html
    cleanup_aizynth_memory(finder, trim_ipython_history=True)


render_unified_chemforge_workbench()

### Cell 6: Agentic AI: Chat with Gemini about the Retrosynthesis Routes
Builds a structured text summary of the current target molecule, active constraints, and top ranked routes (`_build_retrosynthesis_context()`) from `LATEST_RETRO_RESULTS` (or `finder.routes`), and sets up agentic Python tools (such as `tool_rerun_retrosynthesis`) so Gemini can discuss the routes, re-run searches with adjusted constraints, and assist with stoichiometry/ELN and vendor checks.

**What makes Cell 6 agentic:**

* **Tool use (function calling):** Gemini isn't just generating text; it can autonomously invoke Python tools like tool_chem_wizard_bundle (to compute RDKit properties and scaled ELN stoichiometry) and tool_rerun_retrosynthesis (to modify search constraints and re-execute AiZynthFinder).

* **Grounded retrieval:** It combines those local cheminformatics tools with live literature search to ground reaction conditions and references.

* **Goal-directed execution:** Given a high-level prompt (like building a 5.0 g bench-ready plan), it decides which tools to call, ingests their structured outputs, and synthesizes a multi-step experimental protocol.

In [ ]:
# @title 6. Organic Chemist AI Agent: Chat with Gemini about the retrosynthesis routes
import os
import re
import glob
import json
import html
import inspect
import urllib.parse
import requests
import markdown
import IPython
from google.colab import output, userdata
from google import genai
from google.genai import types

# ==============================================================================
# 0. Session Persistence & File Discovery (/content JSON Exports from Cell 5C)
# ==============================================================================
def _discover_retro_session_files(search_dirs=("/content", ".")):
    """Scans /content (and current directory) for saved retrosynthesis JSON exports."""
    candidates = []
    seen_paths = set()
    for d in search_dirs:
        if not os.path.isdir(d):
            continue
        for root, _, files in os.walk(d):
            # Skip hidden/system directories and sample_data
            if any(part.startswith(".") or part in ("sample_data", "data") for part in root.split(os.sep)):
                continue
            for fname in files:
                if not fname.lower().endswith(".json"):
                    continue
                fpath = os.path.abspath(os.path.join(root, fname))
                if fpath in seen_paths:
                    continue
                seen_paths.add(fpath)
                try:
                    # Quick check for valid retrosynthesis export structure
                    with open(fpath, "r", encoding="utf-8") as f:
                        data = json.load(f)
                    if isinstance(data, dict) and ("routes" in data or "target_smiles" in data):
                        mtime = os.path.getmtime(fpath)
                        routes_cnt = len(data.get("routes", []))
                        target_smi = data.get("target_smiles", "Unknown Target")
                        short_smi = (target_smi[:28] + "...") if len(target_smi) > 31 else target_smi
                        candidates.append({
                            "path": fpath,
                            "filename": os.path.basename(fpath),
                            "mtime": mtime,
                            "routes_count": routes_cnt,
                            "target_smiles": target_smi,
                            "label": f"{os.path.basename(fpath)} ({routes_cnt} routes | {short_smi})"
                        })
                except Exception:
                    continue
    candidates.sort(key=lambda x: x["mtime"], reverse=True)
    return candidates


def _sync_session_to_globals(session_data: dict, source_label: str = "Loaded File"):
    """Synchronizes a loaded retrosynthesis dict into LATEST_RETRO_RESULTS and finder (for Cell 7 compatibility)."""
    global LATEST_RETRO_RESULTS
    if not isinstance(session_data, dict):
        return False, "Invalid session payload."
    LATEST_RETRO_RESULTS = session_data
    target_smi = session_data.get("target_smiles")

    # Bridge with AiZynthFinder instance (`finder`) so Cell 7 autonomous loop works seamlessly
    if target_smi and "finder" in globals() and globals()["finder"] is not None:
        try:
            globals()["finder"].target_smiles = target_smi
        except Exception:
            pass
    return True, (
        f"Loaded {len(session_data.get('routes', []))} routes for "
        f"`{target_smi or 'Unknown'}` from {source_label}."
    )


def _ensure_active_retro_session():
    """Ensures LATEST_RETRO_RESULTS is populated in RAM, auto-recovering from disk if needed."""
    global LATEST_RETRO_RESULTS
    if "LATEST_RETRO_RESULTS" in globals() and isinstance(LATEST_RETRO_RESULTS, dict) and LATEST_RETRO_RESULTS.get("routes"):
        # Ensure finder.target_smiles is also synced for Cell 7
        t_smi = LATEST_RETRO_RESULTS.get("target_smiles")
        if t_smi and "finder" in globals() and globals()["finder"] is not None:
            if not getattr(globals()["finder"], "target_smiles", None):
                try:
                    globals()["finder"].target_smiles = t_smi
                except Exception:
                    pass
        return True, "Active RAM session"

    # Fallback: Auto-discover most recent saved JSON session in /content
    discovered = _discover_retro_session_files()
    if discovered:
        latest_file = discovered[0]
        try:
            with open(latest_file["path"], "r", encoding="utf-8") as f:
                data = json.load(f)
            ok, msg = _sync_session_to_globals(data, source_label=latest_file["filename"])
            if ok:
                return True, f"Auto-restored from {latest_file['filename']}"
        except Exception:
            pass
    return False, "No active session in RAM or saved JSON in /content."


def _cb_list_retro_sessions():
    """Colab callback to list available JSON sessions and current RAM status."""
    _ensure_active_retro_session()
    files = _discover_retro_session_files()
    active_smi = ""
    active_routes = 0
    active_mode = "balanced"
    if "LATEST_RETRO_RESULTS" in globals() and isinstance(LATEST_RETRO_RESULTS, dict):
        active_smi = LATEST_RETRO_RESULTS.get("target_smiles", "")
        active_routes = len(LATEST_RETRO_RESULTS.get("routes", []))
        active_mode = LATEST_RETRO_RESULTS.get("ranking_mode", "balanced")
    return IPython.display.JSON({
        "files": files,
        "active_smiles": active_smi,
        "active_routes": active_routes,
        "active_mode": active_mode,
    })


def _cb_load_retro_session(file_path: str):
    """Colab callback to load a selected JSON file from /content into LATEST_RETRO_RESULTS & finder."""
    fpath = (file_path or "").strip()
    if fpath == "__RAM__":
        ok, msg = _ensure_active_retro_session()
        res = globals().get("LATEST_RETRO_RESULTS") or {}
        return IPython.display.JSON({
            "ok": ok,
            "message": msg,
            "target_smiles": res.get("target_smiles", ""),
            "routes_count": len(res.get("routes", [])),
            "ranking_mode": res.get("ranking_mode", "balanced"),
        })
    if not fpath or not os.path.isfile(fpath):
        return IPython.display.JSON({"ok": False, "message": f"File not found: {fpath}"})
    try:
        with open(fpath, "r", encoding="utf-8") as f:
            data = json.load(f)
        ok, msg = _sync_session_to_globals(data, source_label=os.path.basename(fpath))
        return IPython.display.JSON({
            "ok": ok,
            "message": msg,
            "target_smiles": data.get("target_smiles", ""),
            "routes_count": len(data.get("routes", [])),
            "ranking_mode": data.get("ranking_mode", "balanced"),
        })
    except Exception as exc:
        return IPython.display.JSON({"ok": False, "message": f"Failed to load JSON: {exc}"})


# 1. Build an enriched context summary of the current Target & Top 10 AiZynthFinder Routes
def _build_retrosynthesis_context():
    _ensure_active_retro_session()
    if "LATEST_RETRO_RESULTS" in globals() and isinstance(LATEST_RETRO_RESULTS, dict) and LATEST_RETRO_RESULTS.get("routes"):
        res = LATEST_RETRO_RESULTS
        lines = [
            f"Target Molecule SMILES: {res.get('target_smiles', '')}",
            f"Active Ranking Mode: {res.get('ranking_mode', 'balanced')} | Solved: {res.get('is_solved', False)}",
        ]
        if res.get("locked_substructure_smarts"):
            lines.append(f"Locked Substructure SMARTS: {res['locked_substructure_smarts']} (Atoms: {res.get('locked_atom_indices', [])})")
        if res.get("excluded_reactions"):
            lines.append(f"Excluded Reaction Filters: {', '.join(res['excluded_reactions'])}")
        lines.append("")
        for idx, r_entry in enumerate(res["routes"], start=1):
            sm = r_entry.get("starting_materials", {})
            leaf_summary = ", ".join(
                f"{p['smiles']} [{p.get('source', 'Stock')}, {p.get('cost_tier', 'N/A')}]" for p in sm.get("precursors", [])
            ) or "N/A"
            lines.append(
                f"### Route #{idx} (Composite Score: {r_entry.get('composite_score', 0.0):.4f}, "
                f"State Score: {r_entry.get('state_score', 0.0):.4f}, "
                f"Pareto Optimal: {r_entry.get('is_pareto_optimal', False)}, "
                f"Solved: {r_entry.get('is_solved', False)}, "
                f"Safety/Green Avg: {r_entry.get('avg_safety_score', 4.0):.1f}/5, "
                f"Selectivity Risks: {r_entry.get('selectivity_risk_count', 0)}, "
                f"Precursors: {sm.get('overall_cost_tier', 'N/A')} [{sm.get('in_stock_count', 0)}/{sm.get('total_leaves', 0)} in stock])"
            )
            lines.append(f"  - Starting Materials: {leaf_summary}")
            for st in r_entry.get("steps", []):
                s_idx = st.get("step_index", 1)
                name_hint = st.get("reaction_name", "Reaction Step")
                t_id = st.get("template_code", "N/A")
                fwd_smi = st.get("forward_smiles", "")
                cond = st.get("typical_conditions", "")
                safety = st.get("safety_green_score", 4)
                lines.append(
                    f"  - Step {s_idx} ({name_hint}, Template {t_id}, Safety {safety}/5): {fwd_smi} | Conditions: {cond}"
                )
                extras = []
                if st.get("selectivity_warning", "None") != "None":
                    extras.append(f"Selectivity Warning: {st['selectivity_warning']}")
                if st.get("protecting_group_note", "None") != "None":
                    extras.append(f"Protecting Group: {st['protecting_group_note']}")
                if st.get("greener_solvent_swap", "None") != "None":
                    extras.append(f"Greener Swap: {st['greener_solvent_swap']}")
                if st.get("stereochem_note", "Achiral / Racemic") != "Achiral / Racemic":
                    extras.append(f"Stereochem: {st['stereochem_note']}")
                if extras:
                    lines.append("    * " + " | ".join(extras))
        return "\n".join(lines)

    if "finder" not in globals() or not getattr(finder, "target_smiles", None):
        return "No target molecule or routes have been generated in RAM or found in /content JSON exports yet."
    lines = [f"Target Molecule SMILES: {finder.target_smiles}\n"]
    sorted_r = sorted(
        finder.routes,
        key=lambda r: r["score"].get("state score", 0.0),
        reverse=True,
    )[:10]
    for idx, route in enumerate(sorted_r, start=1):
        score_val = route["score"].get("state score", 0.0)
        rxn_tree = route["reaction_tree"]
        lines.append(f"### Route #{idx} (State Score: {score_val:.4f}, Solved: {rxn_tree.is_solved})")
        reactions_forward = list(reversed(list(rxn_tree.reactions())))
        for s_idx, rxn_node in enumerate(reactions_forward, start=1):
            meta = getattr(rxn_node, "metadata", {}) or {}
            t_id = meta.get("template_code", meta.get("template_hash", "N/A"))
            retro_smi = rxn_node.reaction_smiles()
            prod_s, react_s = retro_smi.split(">>", 1) if ">>" in retro_smi else ("", retro_smi)
            fwd_smi = f"{react_s.strip()} >> {prod_s.strip()}"
            step_entry = gemini_lookup.get(fwd_smi) if ("gemini_lookup" in globals() and isinstance(gemini_lookup, dict)) else None
            if isinstance(step_entry, dict):
                name_hint = step_entry.get("reaction_name", "Reaction Step")
            elif hasattr(step_entry, "reaction_name"):
                name_hint = step_entry.reaction_name
            else:
                name_hint = "Step"
            lines.append(f"  - Step {s_idx} ({name_hint}, Template {t_id}): {fwd_smi}")
    return "\n".join(lines)


# 2. Agentic Python Tools for Autonomous Re-Searching, Stoichiometry/ELN & Vendor Checks
def tool_rerun_retrosynthesis(
    ranking_mode: str = "balanced",
    max_transforms: int = 6,
    exclude_reaction_keyword: str = "",
    lock_substructure_smarts: str = "",
) -> str:
    """Re-runs and re-ranks the AiZynthFinder retrosynthesis pipeline with updated search constraints.
    Args:
        ranking_mode: Multi-objective ranking profile ('balanced', 'medchem_fast', 'process_scaleup', or 'green_safe').
        max_transforms: Maximum number of retrosynthetic disconnection steps (e.g. 3 to 8).
        exclude_reaction_keyword: Comma-separated reaction or catalyst names to exclude (e.g. 'Palladium, Stille').
        lock_substructure_smarts: Optional SMARTS substructure to keep intact across all steps (e.g. 'c1ncsc1').
    """
    global LATEST_RETRO_RESULTS
    _ensure_active_retro_session()
    if "run_retrosynthesis_pipeline" not in globals():
        return "Error: run_retrosynthesis_pipeline is not loaded. Please run Cell 5 first."

    # Resolve target SMILES from either finder or restored LATEST_RETRO_RESULTS
    target_smi = None
    if "finder" in globals() and getattr(finder, "target_smiles", None):
        target_smi = finder.target_smiles
    elif "LATEST_RETRO_RESULTS" in globals() and isinstance(LATEST_RETRO_RESULTS, dict):
        target_smi = LATEST_RETRO_RESULTS.get("target_smiles")
        if target_smi and "finder" in globals() and finder is not None:
            try:
                finder.target_smiles = target_smi
            except Exception:
                pass

    if not target_smi:
        return "Error: No target molecule set on finder or in loaded session."

    exclude_list = [kw.strip() for kw in (exclude_reaction_keyword or "").split(",") if kw.strip()] or None
    lock_smarts = (lock_substructure_smarts or "").strip() or None
    kwargs = {
        "target_smiles": target_smi,
        "ranking_mode": (ranking_mode or "balanced").strip(),
        "max_transforms": int(max_transforms),
        "exclude_reaction_names": exclude_list,
        "lock_substructure_smarts": lock_smarts,
        "use_gemini": bool(globals().get("USE_GEMINI_CLASSIFIER", True)),
    }
    sig = inspect.signature(run_retrosynthesis_pipeline)
    if "render_output" in sig.parameters:
        kwargs["render_output"] = False
    updated = run_retrosynthesis_pipeline(**kwargs)
    LATEST_RETRO_RESULTS = updated
    routes = updated.get("routes", [])
    if not routes:
        return (
            f"Re-search completed (mode={ranking_mode}, max_transforms={max_transforms}, "
            f"excluded={exclude_list}, locked={lock_smarts}), but 0 routes satisfied all constraints."
        )
    summary_lines = [
        f"Updated Retrosynthesis Results ({len(routes)} routes | Mode: {updated.get('ranking_mode')} | Solved: {updated.get('is_solved')}):"
    ]
    for idx, r in enumerate(routes[:5], start=1):
        step_names = " -> ".join(s.get("reaction_name", "Step") for s in r.get("steps", []))
        sm_tier = r.get("starting_materials", {}).get("overall_cost_tier", "N/A")
        summary_lines.append(
            f"- Route #{idx}: Composite={r['composite_score']:.4f}, Pareto={r['is_pareto_optimal']}, "
            f"Steps={r['step_count']}, Safety={r['avg_safety_score']:.1f}/5, Precursors={sm_tier} | Sequence: {step_names}"
        )
    return "\n".join(summary_lines)


def tool_generate_scaled_eln_recipe(
    route_number: int,
    target_grams: float = 2.5,
    assumed_yield_per_step: float = 0.80,
) -> str:
    """Generates a complete scaled stoichiometry table (MW, eq, mmol, g, mg) and ELN lab protocol for a route.
    Args:
        route_number: 1-based route number from the active retrosynthesis results (e.g. 1 for Route #1).
        target_grams: Desired mass of final target product in grams (e.g. 2.5 or 5.0).
        assumed_yield_per_step: Fractional yield per step between 0.05 and 1.0 (default 0.80 for 80%).
    """
    _ensure_active_retro_session()
    if "generate_eln_markdown" not in globals():
        return "Error: generate_eln_markdown is not loaded. Please run Cell 5 first."
    return generate_eln_markdown(
        route_idx=int(route_number),
        target_scale_grams=float(target_grams),
        assumed_step_yield=float(assumed_yield_per_step),
    )


def tool_check_precursor_vendors(smiles: str) -> str:
    """Queries local stocks, PubChem PUG-REST/PUG-View, and ChEMBL for commercial vendor availability and synonyms of a SMILES.
    Args:
        smiles: Chemical SMILES string of the starting material or intermediate to look up.
    """
    smi = (smiles or "").strip()
    if not smi:
        return "Error: Empty SMILES provided."
    canon_smi = smi
    inchi_key = ""
    try:
        from rdkit import Chem
        mol = Chem.MolFromSmiles(smi)
        if mol is not None:
            canon_smi = Chem.MolToSmiles(mol)
            inchi_key = Chem.MolToInchiKey(mol)
    except Exception:
        pass
    local_sources = []
    lab_keys = globals().get("CUSTOM_LAB_STOCK_INCHIKEYS", set()) or set()
    lab_smis = globals().get("CUSTOM_LAB_STOCK_SMILES", set()) or set()
    if (inchi_key and inchi_key in lab_keys) or (canon_smi in lab_smis):
        local_sources.append("Custom Lab Inventory ($0 On-Shelf)")
    if "finder" in globals() and finder is not None and hasattr(finder, "stock"):
        try:
            if hasattr(finder.stock, "smiles_in_stock") and finder.stock.smiles_in_stock(canon_smi):
                local_sources.append("Selected AiZynthFinder Stock (ZINC / Lab)")
        except Exception:
            pass
    cid = None
    synonyms = []
    vendors = []
    chembl_id = None
    enc = urllib.parse.quote(canon_smi)
    try:
        r_cid = requests.get(
            f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/smiles/{enc}/cids/JSON",
            timeout=6,
        )
        if r_cid.status_code == 200:
            cids = r_cid.json().get("IdentifierList", {}).get("CID", [])
            if cids:
                cid = cids[0]
        if cid:
            r_syn = requests.get(
                f"https://pubchem.ncbi.nlm.nih.gov/rest/pug/compound/cid/{cid}/synonyms/JSON",
                timeout=6,
            )
            if r_syn.status_code == 200:
                info_list = r_syn.json().get("InformationList", {}).get("Information", [])
                if info_list:
                    synonyms = info_list[0].get("Synonym", [])[:6]
            r_vend = requests.get(
                f"https://pubchem.ncbi.nlm.nih.gov/rest/pug_view/data/compound/{cid}/JSON?heading=Chemical+Vendors",
                timeout=6,
            )
            if r_vend.status_code == 200:
                vendors = list(dict.fromkeys(re.findall(r'"SourceName"\s*:\s*"([^"]+)"', r_vend.text)))[:10]
    except Exception:
        pass
    try:
        if inchi_key:
            r_ch = requests.get(
                f"https://www.ebi.ac.uk/chembl/api/data/molecule/{inchi_key}.json",
                timeout=6,
            )
            if r_ch.status_code == 200:
                chembl_id = r_ch.json().get("molecule_chembl_id")
    except Exception:
        pass
    lines = [
        f"Precursor Vendor & Registry Report for `{canon_smi}`:",
        f"- InChIKey: `{inchi_key or 'N/A'}`",
        f"- Local Stock Status: {', '.join(local_sources) if local_sources else 'Not in local custom/ZINC stock cache'}",
        f"- PubChem CID: `{cid or 'Not found / offline'}` | ChEMBL ID: `{chembl_id or 'N/A'}`",
        f"- Common Synonyms: {', '.join(synonyms) if synonyms else 'None retrieved'}",
        f"- Commercial Vendors ({len(vendors)} sampled): {', '.join(vendors) if vendors else 'No live vendor list retrieved (check ZINC/MolPort/Sigma-Aldrich)'}",
    ]
    return "\n".join(lines)


def _route_step_forward_smiles(route_number=1, step_index=1, custom_forward_smiles=""):
    if custom_forward_smiles:
        return custom_forward_smiles.strip()
    _ensure_active_retro_session()
    routes = (globals().get("LATEST_RETRO_RESULTS") or {}).get("routes", [])
    if not 1 <= int(route_number) <= len(routes):
        raise ValueError("Route number is out of range; run Cell 5B or load a saved JSON session first")
    steps = routes[int(route_number) - 1].get("steps", [])
    if not 1 <= int(step_index) <= len(steps):
        raise ValueError("Step index is out of range")
    return steps[int(step_index) - 1]["forward_smiles"]


def tool_render_chemical_svg(smiles_or_reaction: str, highlight_smarts: str = "", label: str = "") -> str:
    """Render a validated molecule or reaction SMILES as an inline 2D SVG chemistry card."""
    return mol_or_rxn_to_svg_html(smiles_or_reaction, highlight_smarts=highlight_smarts, label=label)


def tool_enumerate_step_byproducts(route_number: int = 1, step_index: int = 1, custom_forward_smiles: str = "") -> str:
    """Enumerate plausible side-products and 2D structures for a route step or custom forward reaction."""
    try:
        candidates = rdkit_enumerate_byproducts(_route_step_forward_smiles(route_number, step_index, custom_forward_smiles))
    except ValueError as exc:
        return f"Error: {exc}"
    return "\n".join(
        f"{p['origin']}: `{p['smiles']}` ({p['formula']}, {p['exact_mw']:.4f} Da) {p['svg_html']}"
        for p in candidates
    ) or "No valid products identified."


def tool_predict_lcms_monitoring_ions(route_number: int = 1, step_index: int = 1, custom_forward_smiles: str = "") -> str:
    """Calculate monoisotopic LC-MS tracking ions for all reactants, products and side-products."""
    try:
        ions = rdkit_predict_lcms_ions(_route_step_forward_smiles(route_number, step_index, custom_forward_smiles))
    except ValueError as exc:
        return f"Error: {exc}"
    return "\n".join(
        f"{p['role']} `{p['smiles']}` {p['formula']} MW {p['exact_mw']:.4f}; "
        + ", ".join(f"{ion} {p[ion]:.4f}" for ion in ("[M+H]+", "[M+Na]+", "[M+NH4]+", "[M+MeCN+H]+", "[M-H]-"))
        + f"; {p['isotope_note']}"
        for p in ions
    ) or "No valid ions identified."


def tool_compute_workup_properties(route_number: int = 1, step_index: int = 1, custom_forward_smiles: str = "") -> str:
    """Compute cLogP/TPSA and phase-partitioning and purification advice for a route step."""
    try:
        data = rdkit_predict_workup_properties(_route_step_forward_smiles(route_number, step_index, custom_forward_smiles))
    except ValueError as exc:
        return f"Error: {exc}"
    if "error" in data:
        return f"Error: {data['error']}"
    p = data["product"]
    return (
        f"Product `{p['smiles']}`: cLogP {p['clogp']:.2f}; TPSA {p['tpsa']:.1f} Å²; HBD/HBA {p['hbd']}/{p['hba']}; "
        f"ΔcLogP {data['delta_clogp']:.2f}, ΔTPSA {data['delta_tpsa']:.1f}. " + " ".join(data["advice"])
    )


def tool_chem_wizard_bundle(
    route_number: int = 1,
    target_grams: float = 5.0,
    assumed_yield_per_step: float = 0.80,
) -> str:
    """Computes the full Chem Wizard RDKit/ELN dataset (scaled stoichiometry, physicochemical workup properties,
    byproducts, and LC-MS ions) for every step of the specified route in a single pass.
    Args:
        route_number: 1-based route number from the active retrosynthesis results (e.g. 1 or 2).
        target_grams: Desired mass of final target product in grams (default 5.0).
        assumed_yield_per_step: Fractional yield per step between 0.05 and 1.0 (default 0.80).
    """
    _ensure_active_retro_session()
    routes = (globals().get("LATEST_RETRO_RESULTS") or {}).get("routes", [])
    if not 1 <= int(route_number) <= len(routes):
        return f"Error: Route #{route_number} is out of range ({len(routes)} routes available). Run Cell 5B or load a saved JSON session first."
    steps = routes[int(route_number) - 1].get("steps", [])
    sections = [
        f"## Scaled ELN & Stoichiometry (Route #{route_number}, {target_grams}g, {int(assumed_yield_per_step * 100)}% yield/step)\n"
        + tool_generate_scaled_eln_recipe(route_number, target_grams, assumed_yield_per_step)
    ]
    for s_idx in range(1, len(steps) + 1):
        sections.append(
            f"### Route #{route_number} Step {s_idx} — RDKit Workup Properties\n"
            + tool_compute_workup_properties(route_number, s_idx)
            + f"\n### Route #{route_number} Step {s_idx} — Enumerated Byproducts\n"
            + tool_enumerate_step_byproducts(route_number, s_idx)
            + f"\n### Route #{route_number} Step {s_idx} — LC-MS Monitoring Ions\n"
            + tool_predict_lcms_monitoring_ions(route_number, s_idx)
        )
    return "\n\n".join(sections)


def tool_modify_and_validate_step(route_number: int, step_index: int, swap_type: str, custom_reactant_smiles: str = "") -> str:
    """Validate and re-score a proposed Suzuki, N-Boc, O-TBS, acid, or custom reactant substitution."""
    import copy
    from rdkit import Chem
    _ensure_active_retro_session()
    if swap_type not in ("stille_to_suzuki", "install_nboc", "install_otbs", "acyl_chloride_to_acid", "custom_reactant"):
        return "Error: Unsupported swap type."
    try:
        old = _route_step_forward_smiles(route_number, step_index)
        react, product = old.split(">>")
        sources = [x.strip() for x in react.split(".") if x.strip()]
        if swap_type == "custom_reactant":
            sources = [x.strip() for x in custom_reactant_smiles.split(".") if x.strip()]
        else:
            if swap_type == "stille_to_suzuki":
                transform = "[c:1]-[Sn]>>[c:1]-B(O)O"
            elif swap_type == "install_nboc":
                transform = "[N;H1,H2;!$(NC=O):1]>>[N:1]C(=O)OC(C)(C)C"
            elif swap_type == "install_otbs":
                transform = "[O;H1;!$([O][C](=O)):1]>>[O:1][Si](C)(C)C(C)(C)C"
            else:
                transform = "[C:1](=[O:2])[Cl]>>[C:1](=[O:2])O"
            from rdkit.Chem import rdChemReactions
            rxn = rdChemReactions.ReactionFromSmarts(transform)
            transformed = False
            for i, smi in enumerate(sources):
                mol = Chem.MolFromSmiles(smi, sanitize=False)
                if mol is None or Chem.SanitizeMol(mol, catchErrors=True) != Chem.SanitizeFlags.SANITIZE_NONE:
                    continue
                for outcome in rxn.RunReactants((mol,), maxProducts=8):
                    candidate = outcome[0]
                    if Chem.SanitizeMol(candidate, catchErrors=True) == Chem.SanitizeFlags.SANITIZE_NONE:
                        sources[i] = Chem.MolToSmiles(candidate)
                        transformed = True
                        break
                if transformed:
                    break
            if not transformed:
                return "Error: No compatible reactant functional group for this swap."
        if not sources or any(_chem_mol(x) is None for x in sources) or any(_chem_mol(x.strip()) is None for x in product.split(".") if x.strip()):
            return "Error: Invalid or unsanitizable proposed reaction SMILES."
        new_forward = ".".join(sources) + " >> " + product.strip()
        updated = copy.deepcopy(LATEST_RETRO_RESULTS)
        step = updated["routes"][int(route_number) - 1]["steps"][int(step_index) - 1]
        inferred_name, inferred_conditions = _match_rdkit_rule(
            [_chem_mol(x) for x in sources],
            [_chem_mol(x.strip()) for x in product.split(".") if x.strip()],
        )
        step.update(analyze_step_heuristics(new_forward, rxn_name=inferred_name, conditions=inferred_conditions))
        if swap_type == "stille_to_suzuki":
            step["typical_conditions"] = inferred_conditions
        step["forward_smiles"] = new_forward
        step["reaction_name"] = "Suzuki-Miyaura Cross-Coupling" if swap_type == "stille_to_suzuki" else step.get("reaction_name", "Reaction Step")
        route = updated["routes"][int(route_number) - 1]
        route["avg_safety_score"] = sum(s.get("safety_green_score", 4) for s in route["steps"]) / len(route["steps"])
        route["selectivity_risk_count"] = sum(s.get("selectivity_warning", "None") != "None" for s in route["steps"])
        route["composite_score"] = compute_composite_route_score(route, updated.get("ranking_mode", "balanced")) if "compute_composite_route_score" in globals() else route.get("composite_score", 0)
        globals()["LATEST_RETRO_RESULTS"] = updated
        return f"Updated Route #{route_number} Step {step_index}: `{new_forward}`; safety {route['avg_safety_score']:.1f}/5, score {route['composite_score']:.4f}\n{mol_or_rxn_to_svg_html(new_forward)}"
    except (ValueError, KeyError, IndexError) as exc:
        return f"Error: {exc}"


AGENT_PYTHON_TOOLS = [
    tool_chem_wizard_bundle,
    tool_rerun_retrosynthesis,
    tool_generate_scaled_eln_recipe,
    tool_check_precursor_vendors,
    tool_render_chemical_svg,
    tool_enumerate_step_byproducts,
    tool_predict_lcms_monitoring_ions,
    tool_compute_workup_properties,
    tool_modify_and_validate_step,
]

_AGENT_TOOL_MAP = {fn.__name__: fn for fn in AGENT_PYTHON_TOOLS}


def _is_chem_wizard_request(user_text: str) -> bool:
    t = (user_text or "").lower()
    return (
        "chem wizard" in t
        or "complete, bench-ready experimental plan" in t
        or ("stoichiometry" in t and "workup" in t and "lc-ms" in t)
    )


def _is_agent_tool_request(user_text: str) -> bool:
    """Determines whether the prompt asks to execute an agentic action/calculation tool rather than pure literature search."""
    t = (user_text or "").lower()
    action_patterns = (
        r"\bre-?run\b",
        r"\bre-?rank\b",
        r"\bprocess[_\s-]scale",
        r"\bmedchem[_\s-]fast\b",
        r"\bgreen[_\s-]safe\b",
        r"\bgreen\s+solvent",
        r"\bexclude\b",
        r"\bwithout\s+(?:using\s+)?[a-z0-9-]+",
        r"\bban\b",
        r"\block\b",
        r"\bfreeze\b",
        r"\bkeep\s+.*\bintact\b",
        r"\bstoichiometr",
        r"\beln\b",
        r"\brecipe\b",
        r"\b\d+(?:\.\d+)?\s*(?:g|grams|mg)\b",
        r"\bmmol\b",
        r"\b(?:byproducts?|regioisomers?|protodeboronation|lc[- ]?ms|m/z|chromatograph|workup|purification|physicochemical|physiochemical|svg|draw|depict|structure|swap)\b",
        r"\bvendor",
        r"\bsupplier",
        r"\bmolport\b",
        r"\bmcule\b",
        r"\bprecursor\s+(?:cost|price|availability)",
    )
    return any(re.search(pat, t) for pat in action_patterns)


def _run_offline_agent_fallback(user_text: str):
    """Executes deterministic local RDKit/pipeline tools when KDD_GEMINI_API_KEY is absent."""
    _ensure_active_retro_session()
    t = (user_text or "").lower()
    route_m = re.search(r"route\s*#?\s*(\d+)", t)
    grams_m = re.search(r"(\d+(?:\.\d+)?)\s*(?:g|grams)\b", t)
    yield_m = re.search(r"(\d+(?:\.\d+)?)\s*%", t)
    r_num = int(route_m.group(1)) if route_m else 1
    g_val = float(grams_m.group(1)) if grams_m else 5.0
    y_val = (float(yield_m.group(1)) / 100.0) if yield_m else 0.80
    if _is_chem_wizard_request(user_text):
        return tool_chem_wizard_bundle(route_number=r_num, target_grams=g_val, assumed_yield_per_step=y_val)
    if any(k in t for k in ("stoichiometr", "eln", "recipe")) or grams_m:
        return tool_generate_scaled_eln_recipe(route_number=r_num, target_grams=g_val, assumed_yield_per_step=y_val)
    if any(k in t for k in ("re-rank", "rerank", "re-run", "rerun", "process_scaleup", "process scale-up", "medchem_fast", "green_safe")):
        mode = "process_scaleup" if "process" in t else ("medchem_fast" if "medchem" in t else ("green_safe" if "green" in t else "balanced"))
        return tool_rerun_retrosynthesis(ranking_mode=mode)
    step_m = re.search(r"step\s*#?\s*(\d+)", t)
    step = int(step_m.group(1)) if step_m else 1
    if "swap" in t or ("stille" in t and "suzuki" in t):
        return tool_modify_and_validate_step(r_num, step, "stille_to_suzuki")
    if "byproduct" in t or "regioisomer" in t:
        return tool_enumerate_step_byproducts(r_num, step) + "\n" + tool_predict_lcms_monitoring_ions(r_num, step)
    if "workup" in t or "purification" in t or "physicochemical" in t or "physiochemical" in t:
        return tool_compute_workup_properties(r_num, step)
    if "lc-ms" in t or "m/z" in t:
        return tool_predict_lcms_monitoring_ions(r_num, step)
    if "draw" in t or "svg" in t:
        smi = re.search(r"`([^`]+)`", user_text or "")
        return tool_render_chemical_svg(smi.group(1)) if smi else "Supply a backticked molecule SMILES to depict."
    return None


def _render_large_rxn_svg_card(rxn_smi: str) -> str:
    """Renders a large, readable 2D RDKit reaction scheme SVG card."""
    try:
        from rdkit.Chem import AllChem
        from rdkit.Chem.Draw import rdMolDraw2D
        rxn = AllChem.ReactionFromSmarts(rxn_smi, useSmiles=True)
        if rxn is not None and rxn.GetNumReactantTemplates() > 0 and rxn.GetNumProductTemplates() > 0:
            drawer = rdMolDraw2D.MolDraw2DSVG(860, 260)
            opts = drawer.drawOptions()
            opts.bondLineWidth = 2.0
            opts.minFontSize = 14
            opts.padding = 0.08
            drawer.DrawReaction(rxn)
            drawer.FinishDrawing()
            svg = drawer.GetDrawingText()
            card = f'<div class="chem-card chem-rxn-card">{svg}</div>'
            globals().setdefault("_CHEM_SVG_CARDS", set()).add(card)
            return card
    except Exception:
        pass
    return mol_or_rxn_to_svg_html(rxn_smi)


def _render_chemistry_markdown(text, trusted_cards=False):
    """Escape untrusted model/lookup HTML; insert locally generated 2D RDKit SVG cards inline."""
    cards = []

    def _register_rxn_card(smi):
        norm = re.sub(r"\s*->\s*", ">>", smi.strip())
        norm = re.sub(r"\s+\+\s+", ".", norm)
        card = _render_large_rxn_svg_card(norm)
        if card.startswith('<div class="chem-card'):
            cards.append(card)
            return f"\n\nCHEMCARDPLACEHOLDER{len(cards) - 1}END\n\n"
        return None

    if trusted_cards:
        def extract(match):
            card = match.group(0)
            if card not in globals().get("_CHEM_SVG_CARDS", set()):
                return card
            cards.append(card)
            return f"CHEMCARDPLACEHOLDER{len(cards) - 1}END"
        text = re.sub(r'<div class="chem-card[^"]*">.*?</div>', extract, text, flags=re.DOTALL)

    rxn_pattern = re.compile(
        r"`?([A-Za-z0-9@+\-\[\]()/\\#%=.]+(?:\s+\+\s+[A-Za-z0-9@+\-\[\]()/\\#%=.]+)*\s*(?:>>|->)\s*[A-Za-z0-9@+\-\[\]()/\\#%=.]+(?:\s+\+\s+[A-Za-z0-9@+\-\[\]()/\\#%=.]+)*)`?"
    )

    def _replace_rxn(match):
        placeholder = _register_rxn_card(match.group(1))
        return placeholder if placeholder else match.group(0)

    text = rxn_pattern.sub(_replace_rxn, text)

    if not trusted_cards:
        mol_pattern = re.compile(r"`([A-Za-z0-9@+\-\[\]()/\\#%=.:]{5,400})`")
        processed_lines = []
        for line in text.splitlines():
            if not line.strip():
                processed_lines.append("")
                continue
            if line.strip().startswith("|") and line.strip().endswith("|"):
                processed_lines.append(line)
                continue
            line_cards = []

            def _collect_mol(match):
                smi = match.group(1)
                card = mol_or_rxn_to_svg_html(smi)
                if card.startswith('<div class="chem-card'):
                    cards.append(card)
                    line_cards.append(f"CHEMCARDPLACEHOLDER{len(cards) - 1}END")
                    if line.strip() == match.group(0):
                        return ""
                return match.group(0)

            new_line = mol_pattern.sub(_collect_mol, line)
            if new_line.strip():
                processed_lines.append(new_line)
            for token in line_cards:
                processed_lines.append(f"\n{token}\n")
        text = "\n".join(processed_lines)

    table_fixed_lines = []
    raw_lines = text.splitlines()
    for idx, line in enumerate(raw_lines):
        is_tbl = line.strip().startswith("|") and "|" in line.strip()[1:]
        prev_is_tbl = idx > 0 and raw_lines[idx - 1].strip().startswith("|") and "|" in raw_lines[idx - 1].strip()[1:]
        if is_tbl and not prev_is_tbl and table_fixed_lines and table_fixed_lines[-1].strip() != "":
            table_fixed_lines.append("")
        table_fixed_lines.append(line)
        next_is_tbl = (idx + 1 < len(raw_lines)) and raw_lines[idx + 1].strip().startswith("|") and "|" in raw_lines[idx + 1].strip()[1:]
        if is_tbl and not next_is_tbl:
            table_fixed_lines.append("")
    text = "\n".join(table_fixed_lines)

    text = re.sub(r"(?m)^ {0,3}\[[^]\n]+\]:\s*(?!https?://)\S+.*$", "", text, flags=re.IGNORECASE)
    text = re.sub(
        r"!?\[([^]\n]*)\]\(([^)\n]*)\)",
        lambda m: m.group(1) if m.group(0).startswith("!") or
        urllib.parse.urlsplit(m.group(2).strip()).scheme.lower() not in ("http", "https")
        else m.group(0),
        text,
    )
    rendered = markdown.markdown(html.escape(text), extensions=["tables", "fenced_code"])
    for i, card in enumerate(cards):
        if card not in globals().get("_CHEM_SVG_CARDS", set()):
            continue
        token = f"CHEMCARDPLACEHOLDER{i}END"
        if token in rendered:
            rendered = rendered.replace(f"<p>{token}</p>", card).replace(token, card)
        else:
            rendered += card
    return rendered


# 3. Persistent Multi-Turn Chat State
_CHAT_HISTORY = []


def _get_system_instruction():
    return f"""You are an autonomous AI-Agent Organic Chemist integrated into an AiZynthFinder + RDKit retrosynthesis notebook.
Your job is to help the user evaluate, re-run, re-rank, and scale up retrosynthetic routes, verify peer-reviewed/patent reaction conditions, check precursor vendors, and generate lab-ready stoichiometry protocols.

CRITICAL INSTRUCTIONS:
1. When asked to re-run/re-rank routes, exclude catalysts/reactions, lock substructures, check precursor vendors, calculate scaled stoichiometry/ELN tables, compute RDKit physicochemical workup properties, or predict byproducts/LC-MS ions, invoke the provided Python agent tools directly (prefer tool_chem_wizard_bundle for full-route plans).
2. When asked for literature precedents, journal references (ACS journals, J. Med. Chem., Org. Process Res. Dev. (OPRD), J. Org. Chem., Angew. Chem. Int. Ed.), or patent procedures, use Google Search grounding and cite full references inline and at the end of each step.
3. Provide specific stoichiometric equivalents, catalysts, greener solvent swaps, protecting group choreography, and stereochemical control notes.
4. Flag any chemoselectivity, regioselectivity, safety, or functional group compatibility risks, and propose concrete troubleshooting/mitigation solutions.
5. Output chemical formulas as plain text (e.g., Pd(PPh3)4, Na2CO3). DO NOT use LaTeX or MathJax anywhere in your response.
6. For every reaction transformation, output the exact forward reaction SMILES on its own line in standard `reactant1.reactant2>>product` format (no extra text on that line) so the UI renders it as a large 2D RDKit reaction scheme.
7. Whenever you discuss an anticipated byproduct, selectivity isomer (e.g. Delta-2 or anti-oxime), or alternative reagent (e.g. EDC·HCl instead of DCC), include its valid SMILES in backticks (e.g. `SMILES`) on the line immediately underneath that explanation so the UI renders its 2D RDKit structure directly below the text.
8. When presenting multi-step experimental plans, ALWAYS organize the response STEP-BY-STEP (i.e. Step 1: Scheme, Stoichiometry Table, Procedure & CAS, Workup/Purification, LC-MS & Byproducts, Anticipated Failures, and Step 1 References; then Step 2 with all of its subsections, etc.). Ensure every Markdown table has a blank line above and below it and each row is on its own line.

CURRENT AIZYNTHFINDER ROUTES IN NOTEBOOK:
{_build_retrosynthesis_context()}
"""


def _invoke_with_model_fallback(client, contents, config, selected_model):
    candidate_models = [selected_model, "gemini-3.8-flash", "gemini-3-flash-preview", "gemini-2.5-flash"]
    last_err = None
    for m in dict.fromkeys(candidate_models):
        if not m:
            continue
        try:
            resp = client.models.generate_content(model=m, contents=contents, config=config)
            return resp, m, None
        except Exception as e:
            last_err = str(e)
    return None, selected_model, last_err


def _send_chat_message(user_text, selected_model, use_search):
    global _CHAT_HISTORY
    _ensure_active_retro_session()
    user_text = (user_text or "").strip()
    if not user_text:
        return IPython.display.JSON({"error": "Empty message"})
    api_key = None
    try:
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")
    use_agent_tools = _is_agent_tool_request(user_text)
    if not api_key:
        if use_agent_tools:
            offline_md = _run_offline_agent_fallback(user_text)
            if offline_md:
                badge = (
                    "<div style='font-size:11px; color:#137333; margin-bottom:6px;'>"
                    "🛠️ Executed deterministic local RDKit/AiZynthFinder tool (offline mode — no KDD_GEMINI_API_KEY required).</div>"
                )
                rendered_off = _render_chemistry_markdown(offline_md, trusted_cards=True)
                return IPython.display.JSON({"html": badge + rendered_off, "model": "local-rdkit-agent"})
        return IPython.display.JSON({
            "error": "Missing KDD_GEMINI_API_KEY. Please add KDD_GEMINI_API_KEY to Google Colab Secrets (🔑 icon on the left sidebar)."
        })

    client = genai.Client(api_key=api_key)
    _CHAT_HISTORY.append(types.Content(role="user", parts=[types.Part.from_text(text=user_text)]))
    tool_badges_html = ""
    response = None
    used_model = selected_model
    last_err = None
    fcalls = None
    tool_outputs = []

    # Fast-Path: Direct 1-Pass Chem Wizard Shortcut
    if _is_chem_wizard_request(user_text):
        t = user_text.lower()
        route_m = re.search(r"route\s*#?\s*(\d+)", t)
        grams_m = re.search(r"(\d+(?:\.\d+)?)\s*(?:g|grams)\b", t)
        yield_m = re.search(r"(\d+(?:\.\d+)?)\s*%", t)
        r_num = int(route_m.group(1)) if route_m else 1
        g_val = float(grams_m.group(1)) if grams_m else 5.0
        y_val = (float(yield_m.group(1)) / 100.0) if yield_m else 0.80
        bundle_out = tool_chem_wizard_bundle(route_number=r_num, target_grams=g_val, assumed_yield_per_step=y_val)
        tool_badges_html = (
            "<div style='font-size:11px; background:#e6f4ea; color:#137333; padding:4px 8px; "
            "border-radius:6px; margin-bottom:8px;'>⚡ <b>1-Pass Chem Wizard Bundle Executed:</b> "
            f"tool_chem_wizard_bundle(route_number={r_num}, target_grams={g_val}, assumed_yield_per_step={y_val})</div>"
        )
        followup_contents = list(_CHAT_HISTORY) + [
            types.Content(
                role="user",
                parts=[types.Part.from_text(
                    text=(
                        "The local RDKit/ELN Chem Wizard bundle returned the following live calculations:\n\n"
                        + bundle_out
                        + "\n\nCRITICAL FORMATTING: Organize the entire response STEP-BY-STEP (Step 1 complete with all subsections, "
                        "then Step 2 complete with all subsections, etc.). Under each step include: "
                        "(a) Forward reaction SMILES on its own line (`reactants>>product`), "
                        "(b) Clean multi-line Markdown stoichiometry table with a blank line above and below, "
                        "(c) Step-by-step experimental procedure with CAS numbers and IPC, "
                        "(d) RDKit physicochemical workup & purification plan, "
                        "(e) Product & byproduct LC-MS monitoring ions (with backticked SMILES on the line below each byproduct), "
                        "(f) Anticipated failures & troubleshooting (with backticked SMILES on the line below each isomer/alternative reagent), and "
                        "(g) Grounded scientific literature references (ACS, OPRD, Angew. Chem.) for that step."
                    )
                )],
            )
        ]
        synth_tools = [types.Tool(google_search=types.GoogleSearch())] if use_search else []
        synth_config = types.GenerateContentConfig(
            system_instruction=_get_system_instruction(),
            tools=synth_tools,
            temperature=0.2,
        )
        response, used_model, last_err = _invoke_with_model_fallback(
            client, followup_contents, synth_config, selected_model
        )

    # Route 1: Standard Python Agent Tools
    elif use_agent_tools:
        cfg_kwargs = {
            "system_instruction": _get_system_instruction(),
            "tools": AGENT_PYTHON_TOOLS,
            "temperature": 0.1,
        }
        if hasattr(types, "AutomaticFunctionCallingConfig"):
            cfg_kwargs["automatic_function_calling"] = types.AutomaticFunctionCallingConfig(disable=True)
        tool_config = types.GenerateContentConfig(**cfg_kwargs)
        response, used_model, last_err = _invoke_with_model_fallback(
            client, _CHAT_HISTORY, tool_config, selected_model
        )
        fcalls = getattr(response, "function_calls", None) if response is not None else None
        if fcalls:
            executed_names = []
            for fc in fcalls:
                fn_name = getattr(fc, "name", "")
                raw_args = dict(getattr(fc, "args", {}) or {})
                fn_obj = _AGENT_TOOL_MAP.get(fn_name)
                if fn_obj is not None:
                    try:
                        out_str = str(fn_obj(**raw_args))
                    except Exception as tool_exc:
                        out_str = f"Tool execution error in {fn_name}: {tool_exc}"
                    executed_names.append(f"{fn_name}({', '.join(f'{k}={v!r}' for k, v in raw_args.items())})")
                    tool_outputs.append(f"### Tool Output `{fn_name}`\n{out_str}")
            if executed_names:
                tool_badges_html = (
                    "<div style='font-size:11px; background:#e6f4ea; color:#137333; padding:4px 8px; "
                    "border-radius:6px; margin-bottom:8px;'>🛠️ <b>Agent Tool Executed:</b> "
                    + html.escape("; ".join(executed_names))
                    + "</div>"
                )
            combined_tool_report = "\n\n".join(tool_outputs)
            followup_contents = list(_CHAT_HISTORY) + [
                types.Content(
                    role="user",
                    parts=[types.Part.from_text(
                        text=(
                            "The requested Python chemistry tool(s) returned the following live results:\n\n"
                            + combined_tool_report
                            + "\n\nPresent and interpret these results clearly for the chemist, preserving any stoichiometry Markdown tables "
                            "(each row on its own line with blank lines above/below), and include backticked SMILES directly under any "
                            "byproduct, isomer, or alternative reagent discussed."
                        )
                    )],
                )
            ]
            synth_tools = [types.Tool(google_search=types.GoogleSearch())] if use_search else []
            synth_config = types.GenerateContentConfig(
                system_instruction=_get_system_instruction(),
                tools=synth_tools,
                temperature=0.2,
            )
            synth_resp, synth_model, _ = _invoke_with_model_fallback(
                client, followup_contents, synth_config, used_model
            )
            if synth_resp is not None and getattr(synth_resp, "text", None):
                response = synth_resp
                used_model = synth_model
            else:
                _CHAT_HISTORY.append(types.Content(role="model", parts=[types.Part.from_text(text=combined_tool_report)]))
                rendered_md = _render_chemistry_markdown(combined_tool_report, trusted_cards=True)
                return IPython.display.JSON({
                    "html": tool_badges_html + rendered_md,
                    "model": used_model,
                })

    # Route 2: Google Search Grounding for Live Literature & Patent Verification
    if response is None and not use_agent_tools:
        search_tools = [types.Tool(google_search=types.GoogleSearch())] if use_search else []
        search_config = types.GenerateContentConfig(
            system_instruction=_get_system_instruction(),
            tools=search_tools,
            temperature=0.2,
        )
        response, used_model, last_err = _invoke_with_model_fallback(
            client, _CHAT_HISTORY, search_config, selected_model
        )

    if response is None:
        _CHAT_HISTORY.pop()
        return IPython.display.JSON({"error": f"API Error: {last_err}"})

    reply_text = response.text or "(No text returned)"
    _CHAT_HISTORY.append(types.Content(role="model", parts=[types.Part.from_text(text=reply_text)]))

    sources_html = ""
    try:
        cand = response.candidates[0]
        gm = getattr(cand, "grounding_metadata", None)
        chunks = getattr(gm, "grounding_chunks", None) if gm else None
        if chunks:
            links = []
            for c in chunks:
                web = getattr(c, "web", None)
                if web and getattr(web, "uri", None):
                    title = html.escape(getattr(web, "title", None) or web.uri)
                    if urllib.parse.urlparse(web.uri).scheme in ("https", "http"):
                        links.append(
                            f"<li><a href='{html.escape(web.uri, quote=True)}' target='_blank' rel='noopener noreferrer' style='color:#1a73e8;'>{title}</a></li>"
                        )
            if links:
                sources_html = (
                    "<div style='margin-top:10px; padding-top:8px; border-top:1px solid #e0e0e0; font-size:12px;'>"
                    "<b>📚 Grounded Literature & Web Sources:</b><ul style='margin:4px 0 0 18px; padding:0;'>"
                    + "".join(dict.fromkeys(links))
                    + "</ul></div>"
                )
    except Exception:
        pass

    note = ""
    if used_model != selected_model:
        note = f"<div style='font-size:11px; color:#b06000; margin-bottom:4px;'>ℹ️ Note: '{html.escape(selected_model)}' routed to active endpoint '{html.escape(used_model)}'.</div>"

    rendered_md = _render_chemistry_markdown(reply_text)
    return IPython.display.JSON({
        "html": note + tool_badges_html + rendered_md + sources_html,
        "model": used_model,
    })


def _reset_chat_history():
    global _CHAT_HISTORY
    _CHAT_HISTORY = []
    return IPython.display.JSON({"status": "cleared"})


output.register_callback("notebook.gemini_retro_chat", _send_chat_message)
output.register_callback("notebook.gemini_retro_reset", _reset_chat_history)
output.register_callback("notebook.gemini_retro_list_sessions", _cb_list_retro_sessions)
output.register_callback("notebook.gemini_retro_load_session", _cb_load_retro_session)

# Trigger initial auto-check on cell execution so Cell 7 works immediately if a JSON exists in /content
_init_ok, _init_msg = _ensure_active_retro_session()

# 4. Render Interactive Chat UI inside the Colab Cell Output (with Session Persistence Bar)
chat_ui_html = r"""
<style>
  .retro-chat-box {
    font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, sans-serif;
    border: 1px solid #dadce0;
    border-radius: 10px;
    background: #ffffff;
    max-width: 1040px;
    box-shadow: 0 2px 6px rgba(0,0,0,0.06);
    overflow: hidden;
  }
  .retro-chat-header {
    background: #f8f9fa;
    padding: 12px 16px;
    border-bottom: 1px solid #dadce0;
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    justify-content: space-between;
    gap: 10px;
  }
  .retro-session-bar {
    background: #eef4ff;
    padding: 8px 16px;
    border-bottom: 1px solid #d2e3fc;
    display: flex;
    flex-wrap: wrap;
    align-items: center;
    justify-content: space-between;
    gap: 8px;
    font-size: 12px;
  }
  .retro-session-status {
    color: #174ea6;
    font-weight: 500;
  }
  .retro-chat-messages {
    height: 520px;
    overflow-y: auto;
    padding: 16px;
    background: #fcfcfc;
    display: flex;
    flex-direction: column;
    gap: 12px;
  }
  .msg-user {
    align-self: flex-end;
    flex-shrink: 0;
    background: #e8f0fe;
    color: #174ea6;
    padding: 10px 14px;
    border-radius: 14px 14px 2px 14px;
    max-width: 78%;
    font-size: 14px;
    line-height: 1.45;
  }
  .msg-model {
    align-self: flex-start;
    flex-shrink: 0;
    background: #ffffff;
    color: #202124;
    border: 1px solid #e0e0e0;
    padding: 14px 18px;
    border-radius: 14px 14px 14px 2px;
    max-width: 96%;
    width: 96%;
    box-sizing: border-box;
    font-size: 14px;
    line-height: 1.55;
    overflow-x: auto;
  }
  .msg-model table {
    border-collapse: collapse;
    margin: 10px 0 14px 0;
    font-size: 13px;
    width: 100%;
    background: #fff;
  }
  .msg-model th {
    background: #f1f3f4;
    color: #202124;
    font-weight: 600;
    text-align: left;
  }
  .msg-model th, .msg-model td {
    border: 1px solid #dadce0;
    padding: 7px 10px;
    white-space: nowrap;
  }
  .msg-model tr:nth-child(even) td {
    background: #fafafa;
  }
  .msg-model .chem-card {
    margin: 8px 0 12px 0;
    padding: 8px;
    background: #ffffff;
    border: 1px solid #e8eaed;
    border-radius: 8px;
    display: inline-block;
    max-width: 100%;
    overflow-x: auto;
  }
  .msg-model .chem-rxn-card {
    display: block;
    width: 100%;
    box-sizing: border-box;
    text-align: center;
  }
  .msg-model .chem-rxn-card svg {
    width: 100% !important;
    min-width: 640px;
    max-width: 880px;
    height: auto !important;
  }
  .retro-chat-input-area {
    padding: 12px 16px;
    border-top: 1px solid #dadce0;
    background: #f8f9fa;
    display: flex;
    gap: 10px;
    align-items: flex-end;
  }
  .retro-quick-btns {
    padding: 8px 16px;
    background: #f1f3f4;
    border-top: 1px solid #e8eaed;
    display: flex;
    gap: 8px;
    flex-wrap: wrap;
    font-size: 12px;
  }
  .quick-chip {
    background: #fff;
    border: 1px solid #dadce0;
    border-radius: 16px;
    padding: 4px 10px;
    cursor: pointer;
    color: #3c4043;
  }
  .quick-chip:hover { background: #e8f0fe; border-color: #1a73e8; color: #1a73e8; }
  .chem-wizard-btn {
    background: linear-gradient(135deg, #673ab7, #1a73e8);
    color: #ffffff;
    border: none;
    border-radius: 16px;
    padding: 5px 13px;
    font-weight: 600;
    cursor: pointer;
    box-shadow: 0 1px 4px rgba(103, 58, 183, 0.3);
  }
  .chem-wizard-btn:hover { opacity: 0.92; }
</style>

<div class="retro-chat-box">
  <div class="retro-chat-header">
    <div>
      <strong style="font-size:15px; color:#1a73e8;">🔬 Gemini Retrosynthesis Literature &amp; Conditions Assistant</strong>
      <div style="font-size:12px; color:#5f6368;">Context loaded with your Target SMILES &amp; Top 10 AiZynthFinder Routes (RAM + /content Session Persistence)</div>
    </div>
    <div style="display:flex; align-items:center; gap:10px;">
      <label style="font-size:12px; font-weight:600; color:#3c4043;">Model:</label>
      <select id="gemini-model-select" style="padding:5px 8px; border-radius:5px; border:1px solid #ccc; font-size:13px;">
        <option value="gemini-3.8-flash" selected>gemini-3.8-flash (Default)</option>
        <option value="argon">Argon</option>
        <option value="gemini-3-flash-preview">gemini-3-flash-preview</option>
        <option value="gemini-3.1-pro-preview">gemini-3.1-pro-preview</option>
        <option value="gemini-2.5-flash">gemini-2.5-flash</option>
        <option value="gemini-2.5-pro">gemini-2.5-pro</option>
      </select>
      <label style="font-size:12px; display:flex; align-items:center; gap:4px; cursor:pointer; color:#3c4043;">
        <input type="checkbox" id="gemini-search-toggle" checked /> 📚 Literature Search
      </label>
      <button id="gemini-clear-btn" style="padding:5px 10px; border:1px solid #ccc; background:#fff; border-radius:5px; cursor:pointer; font-size:12px;">
        🗑️ Clear Chat
      </button>
    </div>
  </div>

  <!-- Session Persistence & File Discovery Bar -->
  <div class="retro-session-bar">
    <div style="display:flex; align-items:center; gap:8px; flex-wrap:wrap;">
      <strong style="color:#174ea6;">📂 Session Source:</strong>
      <select id="retro-session-select" style="padding:4px 8px; border-radius:5px; border:1px solid #aecbfa; font-size:12px; min-width:280px; background:#fff;">
        <option value="__RAM__">⚡ Active RAM Session (LATEST_RETRO_RESULTS)</option>
      </select>
      <button id="retro-load-session-btn" style="padding:4px 10px; background:#1a73e8; color:#fff; border:none; border-radius:5px; cursor:pointer; font-size:12px; font-weight:600;">
        Load Session
      </button>
      <button id="retro-refresh-sessions-btn" style="padding:4px 8px; background:#fff; color:#1a73e8; border:1px solid #aecbfa; border-radius:5px; cursor:pointer; font-size:12px;">
        🔄 Scan /content
      </button>
    </div>
    <div id="retro-session-status" class="retro-session-status">Checking active session...</div>
  </div>

  <div id="retro-chat-messages" class="retro-chat-messages">
    <div class="msg-model">
      👋 <b>Ready!</b> I have your active <b>AiZynthFinder routes</b> loaded (and can restore saved JSON exports from <code>/content</code> via the dropdown above, keeping <b>Cell 7's autonomous loop</b> in sync).
      Ask me to search the literature for exact reaction conditions, catalysts, yields, or patent procedures—or click <b>🧙‍♂️ Chem Wizard</b> to build a complete, step-by-step experimental plan.
    </div>
  </div>

  <div class="retro-quick-btns">
    <span style="color:#5f6368; font-weight:600; align-self:center;">Quick prompts:</span>
    <button class="chem-wizard-btn" onclick="fillChemWizardPrompt()">🧙‍♂️ Chem Wizard (Full 5.0g Experiment Plan)</button>
    <button class="quick-chip" onclick="fillPrompt('Perform a literature search for the exact reaction conditions, reagents, yields, and references for each step of Route #1.')">Route #1 Literature Conditions</button>
    <button class="quick-chip" onclick="fillPrompt('Perform a literature search for the exact reaction conditions and references for Route #6 (Suzuki coupling, SNAr, and ester hydrolysis).')">Route #6 Conditions &amp; Refs</button>
    <button class="quick-chip" onclick="fillPrompt('Compare Route #1 (Suzuki coupling) vs Route #2 (Hantzsch thiazole cyclization) in the published literature and patents for this compound. Which has higher yield and better scalability?')">Compare Route #1 vs Route #2</button>
    <button class="quick-chip" onclick="fillPrompt('Re-run and re-rank the retrosynthesis routes in process_scaleup mode prioritizing green solvents and safety.')">⚡ Re-rank for Process Scale-Up &amp; Green Solvents</button>
    <button class="quick-chip" onclick="fillPrompt('Generate a 5.0g Stoichiometry & ELN Table for Route #1 assuming 80% step yield.')">⚖️ Generate 5.0g Stoichiometry &amp; ELN Table for Route #1</button>
    <button class="quick-chip" onclick="fillPrompt('Predict byproducts and LC-MS ions for Route #1 Step 1.')">🧪 Predict Byproducts &amp; LC-MS Ions (Route #1 Step 1)</button>
    <button class="quick-chip" onclick="fillPrompt('Compute physicochemical workup and purification plan for Route #1 Step 1.')">💧 Physicochemical Workup &amp; Purification Plan (Route #1)</button>
    <button class="quick-chip" onclick="fillPrompt('Swap Stille hazardous Step 1 of Route #1 to Suzuki and re-score.')">🔄 Swap Stille/Hazardous Step to Suzuki &amp; Re-Score</button>
  </div>

  <div class="retro-chat-input-area">
    <textarea id="retro-chat-input" rows="3" placeholder="Ask about reaction conditions, catalysts, literature precedents, or click Chem Wizard for a full lab plan (Press Enter to send, Shift+Enter for newline)..."
              style="flex:1; padding:10px; border:1px solid #ccc; border-radius:6px; font-family:inherit; font-size:14px; resize:vertical;"></textarea>
    <button id="retro-send-btn" style="background:#1a73e8; color:white; border:none; padding:10px 18px; border-radius:6px; font-weight:bold; cursor:pointer; height:42px;">
      Send ➤
    </button>
  </div>
</div>

<script>
function fillPrompt(text) {
  const input = document.getElementById("retro-chat-input");
  input.value = text;
  input.focus();
}

function fillChemWizardPrompt() {
  const prompt = [
    "Act as a senior process & medicinal chemist and build a complete, bench-ready experimental plan for Route #1 at a 5.0g target scale (assuming 80% yield per step). Use the RDKit/ELN bundle for every step of Route #1 and ground all procedures in peer-reviewed literature.",
    "IMPORTANT STRUCTURE: Organize the entire plan STEP-BY-STEP. For each step (Step 1, Step 2, Step 3, etc.), group ALL of the following subsections together under that step so the chemist can read each step from start to finish without scrolling:",
    "  a) Reaction Scheme: Output the exact forward reaction SMILES on its own line (`reactants>>product`).",
    "  b) 5.0g Scaled Stoichiometry & ELN Table: Clean Markdown table (MW, CAS RN, eq, mmol, g/mg, molarity, solvent volumes) with a blank line above and below.",
    "  c) Step-by-Step Experimental Procedure: Detailed lab procedure with exact reactant/reagent CAS numbers, addition order, temperature profile, reaction time, and IPC checkpoints.",
    "  d) Physicochemical Workup & Purification Plan: Use the RDKit properties (cLogP, TPSA, HBD/HBA, ΔcLogP, ΔTPSA) to design the quench, extraction pH/solvent system, and purification strategy.",
    "  e) Product & Byproduct LC-MS Monitoring: Exact monoisotopic LC-MS tracking ions ([M+H]+, [M+Na]+, [M-H]-, isotope patterns) for reactants, product, and plausible byproducts, placing each byproduct's backticked SMILES directly underneath its description.",
    "  f) Anticipated Failures & Troubleshooting Solutions: Highest-risk failure modes (chemoselectivity, isomerization, hydrolysis, stalling, safety/exotherm) with preventative controls and fallback reagents (place backticked SMILES directly underneath any isomer or alternative reagent).",
    "  g) Grounded Scientific References for This Step: Peer-reviewed references (ACS journals such as J. Org. Chem., J. Med. Chem., OPRD, Org. Lett., or Angew. Chem. Int. Ed.)."
  ].join("\n");
  fillPrompt(prompt);
}

async function refreshSessionDropdown() {
  const selectEl = document.getElementById("retro-session-select");
  const statusEl = document.getElementById("retro-session-status");
  try {
    const result = await google.colab.kernel.invokeFunction("notebook.gemini_retro_list_sessions", [], {});
    const data = result.data["application/json"];
    selectEl.innerHTML = "";
    const ramOpt = document.createElement("option");
    ramOpt.value = "__RAM__";
    ramOpt.textContent = data.active_routes > 0
      ? `⚡ Active RAM (${data.active_routes} routes | ${data.active_smiles.slice(0, 28)})`
      : "⚡ Active RAM Session (Empty)";
    selectEl.appendChild(ramOpt);

    (data.files || []).forEach(f => {
      const opt = document.createElement("option");
      opt.value = f.path;
      opt.textContent = "📄 " + f.label;
      selectEl.appendChild(opt);
    });

    if (data.active_routes > 0) {
      statusEl.textContent = `✅ Active: ${data.active_routes} routes (${data.active_mode}) | Target: ${data.active_smiles.slice(0, 32)}`;
    } else if ((data.files || []).length > 0) {
      statusEl.textContent = `📁 Found ${data.files.length} saved session(s) in /content. Click Load Session.`;
    } else {
      statusEl.textContent = "⚠️ No active routes in RAM or /content JSON files found.";
    }
  } catch (err) {
    statusEl.textContent = "⚠️ Could not scan /content sessions.";
  }
}

async function loadSelectedSession() {
  const selectEl = document.getElementById("retro-session-select");
  const statusEl = document.getElementById("retro-session-status");
  const msgContainer = document.getElementById("retro-chat-messages");
  const chosen = selectEl.value;
  statusEl.textContent = "⏳ Loading session...";
  try {
    const result = await google.colab.kernel.invokeFunction("notebook.gemini_retro_load_session", [chosen], {});
    const data = result.data["application/json"];
    if (data.ok) {
      statusEl.textContent = `✅ Loaded ${data.routes_count} routes (${data.ranking_mode}) | Target: ${(data.target_smiles || "").slice(0, 32)}`;
      const noteDiv = document.createElement("div");
      noteDiv.className = "msg-model";
      noteDiv.innerHTML = `📂 <b>Session Restored:</b> ${data.message} <br><span style="font-size:12px; color:#5f6368;"><code>LATEST_RETRO_RESULTS</code> and <code>finder.target_smiles</code> are now synchronized for both interactive chat and <b>Cell 7</b> autonomous optimization.</span>`;
      msgContainer.appendChild(noteDiv);
      msgContainer.scrollTop = msgContainer.scrollHeight;
    } else {
      statusEl.textContent = "⚠️ " + data.message;
    }
  } catch (err) {
    statusEl.textContent = "⚠️ Error loading session: " + err;
  }
}

async function sendRetroMessage() {
  const inputEl = document.getElementById("retro-chat-input");
  const msgContainer = document.getElementById("retro-chat-messages");
  const modelSelect = document.getElementById("gemini-model-select").value;
  const useSearch = document.getElementById("gemini-search-toggle").checked;
  const sendBtn = document.getElementById("retro-send-btn");
  const text = inputEl.value.trim();
  if (!text) return;

  const userDiv = document.createElement("div");
  userDiv.className = "msg-user";
  userDiv.innerText = text;
  msgContainer.appendChild(userDiv);
  inputEl.value = "";

  const loadingDiv = document.createElement("div");
  loadingDiv.className = "msg-model";
  loadingDiv.innerHTML = "<i>🔎 Running RDKit tools, searching chemical literature &amp; generating response with <b>" + modelSelect + "</b>...</i>";
  msgContainer.appendChild(loadingDiv);
  msgContainer.scrollTop = msgContainer.scrollHeight;
  sendBtn.disabled = true;

  try {
    const result = await google.colab.kernel.invokeFunction(
      "notebook.gemini_retro_chat",
      [text, modelSelect, useSearch],
      {}
    );
    const data = result.data["application/json"];
    if (data.error) {
      loadingDiv.textContent = "⚠️ " + data.error;
    } else {
      loadingDiv.innerHTML = data.html;
    }
    refreshSessionDropdown();
  } catch (err) {
    loadingDiv.textContent = "⚠️ Error communicating with Colab kernel: " + err;
  } finally {
    sendBtn.disabled = false;
    msgContainer.scrollTop = msgContainer.scrollHeight;
  }
}

document.getElementById("retro-send-btn").onclick = sendRetroMessage;
document.getElementById("retro-load-session-btn").onclick = loadSelectedSession;
document.getElementById("retro-refresh-sessions-btn").onclick = refreshSessionDropdown;
document.getElementById("retro-chat-input").addEventListener("keydown", function(e) {
  if (e.key === "Enter" && !e.shiftKey) {
    e.preventDefault();
    sendRetroMessage();
  }
});
document.getElementById("gemini-clear-btn").onclick = async function() {
  await google.colab.kernel.invokeFunction("notebook.gemini_retro_reset", [], {});
  document.getElementById("retro-chat-messages").innerHTML =
    '<div class="msg-model">🧹 <b>Chat history cleared.</b> Ask a new question about any of your retrosynthetic routes!</div>';
};

refreshSessionDropdown();
</script>
"""

IPython.display.display(IPython.display.HTML(chat_ui_html))

# Autonomous Multi-Agent Retrosynthesis

We can transition this notebook from a human-in-the-loop, tool-assisted chat interface into a fully autonomous multi-agent retrosynthesis system.

Rather than waiting for manual prompts, an autonomous multi-agent architecture introduces specialized roles (e.g., T**he Retrosynthesis Planner, The Reaction Critic/Safety Auditor,** and **The Parameter Optimizer**) that run in an automated loop. They iteratively generate routes, critique them for safety or yield bottlenecks, re-run searches with adjusted constraints, and converge on an optimized plan without human intervention.

## Cell 7: Autonomous Multi-Agent Retrosynthesis Loop
This cell instantiates a closed-loop multi-agent workflow:
1. **Agent 1 (The Planner):** Evaluates target retrosynthesis routes, selects candidate routes, and proposes strategic modifications (e.g., swapping toxic reagents or adjusting max steps).
2. **Agent 2 (The Safety & Process Critic):** Audits each step for chemoselectivity risks, harsh conditions, or low green/safety scores, issuing specific failure hypotheses.
3. **Agent 3 (The Optimization Controller):** Programmatically invokes `tool_rerun_retrosynthesis` or step-swapping tools, feeding feedback back into AiZynthFinder until a target composite score and safety threshold are satisfied.

## What Key Architectural Shifts this into True AI Autonomy?

1. **Closed Feedback Loop:** Instead of waiting for a user to type a follow-up in Cell 6, the loop in Cell 7 programmatically inspects metric thresholds (avg_safety_score and composite_score).

2. **Autonomous Tool Dispatching:** The agent parses its own critique into structured parameters and directly updates the global search configuration via Python functions without manual button clicking.

3. **Multi-Role Prompting:** Dividing the system instruction into explicit sub-agents (Planner, Safety Critic, and Optimizer) prevents hallucinated routes by forcing validation against RDKit property filters and AiZynthFinder score bounds.

### 🤖 How to Track Progress & Underlying AI Model

#### 1. How to Track the Agent's Progress
In **Cell 7**, the multi-agent loop prints real-time telemetry directly to the cell's output stream as it executes. You can track its progress by monitoring:
- **Iteration Counter:** Displays current loop progress (e.g., `--- 🔄 Multi-Agent Iteration 1 of 3 ---`).
- **Route Metric Evaluations:** Prints live scoring updates for the top route, including composite performance and average safety scores (`Current Best Route Score: 0.6241 | Safety Average: 4.0/5`).
- **Agent Reasoning & Critique Logs:** Outputs raw reasoning text from the Critic and Optimization agents explaining why a route was approved or flagged.
- **Automated Tool Triggers:** Displays logs whenever the controller programmatically invokes `tool_rerun_retrosynthesis` with adjusted parameters (e.g., `process_scaleup` or `green_safe`).

#### 2. What AI Agent Framework / Model Is Being Used?
This notebook does **not** use a standalone pre-packaged framework like "Gemini Antigravity." Instead, it is built using:
- **Google GenAI SDK (`genai.Client`):** Directly instantiated using the `google-genai` Python library.
- **Model Architecture:** Powered by Google's native multimodal models—specifically **`gemini-3.8-flash`** (or configured Gemini preview/pro endpoints) with structured Pydantic outputs and custom Python function calling tools.
- **Custom Agent Logic:** The multi-role agent behavior (Planner, Safety Critic, and Optimization Controller) is orchestrated via **custom system prompts and structured JSON parsing** defined directly in Python inside **Cell 7**.

In [ ]:
# @title 7. Autonomous Multi-Agent Synthesis Controller
import time
import json

def run_autonomous_retrosynthesis_loop(
    max_agent_iterations: int = 3,
    min_target_safety_score: float = 3.5,
    min_composite_score: float = 0.50
):
    """
    Executes a fully autonomous multi-agent optimization loop:
    Planner -> Safety/Process Critic -> Optimization Controller -> AiZynthFinder Re-run.
    """
    # Tell Python to use the global LATEST_RETRO_RESULTS variable
    global LATEST_RETRO_RESULTS

    if "finder" not in globals() or not getattr(finder, "target_smiles", None):
        return "Error: No target molecule set. Please run Cell 4 and Cell 5 first."

    print(f"🤖 [Multi-Agent System] Initializing autonomous loop for target: {finder.target_smiles}")

    # 1. Initial Search if LATEST_RETRO_RESULTS doesn't exist globally
    if "LATEST_RETRO_RESULTS" not in globals() or not LATEST_RETRO_RESULTS or not LATEST_RETRO_RESULTS.get("routes"):
        print("⚡ [Planner Agent] No initial routes found. Triggering baseline search...")
        LATEST_RETRO_RESULTS = run_retrosynthesis_pipeline(
            target_smiles=finder.target_smiles,
            ranking_mode="balanced",
            max_transforms=6,
            use_gemini=True
        )

    client = None
    try:
        api_key = userdata.get("KDD_GEMINI_API_KEY")
    except Exception:
        api_key = os.environ.get("KDD_GEMINI_API_KEY")

    if not api_key:
        return "Error: KDD_GEMINI_API_KEY required for autonomous agent reasoning loops."

    agent_client = genai.Client(api_key=api_key)

    for iteration in range(1, max_agent_iterations + 1):
        print(f"\n--- 🔄 Multi-Agent Iteration {iteration} of {max_agent_iterations} ---")

        # Build current state context
        current_context = _build_retrosynthesis_context()
        top_route = LATEST_RETRO_RESULTS.get("routes", [{}])[0]
        avg_safety = top_route.get("avg_safety_score", 4.0)
        composite = top_route.get("composite_score", 0.0)

        print(f"   📊 Current Best Route Score: {composite:.4f} | Safety Average: {avg_safety:.1f}/5")

        # Check convergence criteria
        if avg_safety >= min_target_safety_score and composite >= min_composite_score:
            print("✨ [Multi-Agent System] Convergence criteria met! Routes are optimized and safe.")
            break

        # Agent Prompt: Critic & Optimizer evaluation
        prompt = (
            f"You are the Lead Process Chemistry Multi-Agent Controller.\n"
            f"Current Target State:\n{current_context}\n\n"
            f"Evaluate the top route. If safety score is below {min_target_safety_score} or "
            f"if hazardous reagents are present, decide whether to re-run the search using "
            f"`tool_rerun_retrosynthesis` with `ranking_mode='process_scaleup'` or `green_safe`.\n"
            f"Output a JSON decision block with keys: `action` ('rerun' or 'approve'), "
            f"`ranking_mode` (string), `exclude_keyword` (string), and `reasoning` (string)."
        )

        response = agent_client.models.generate_content(
            model="gemini-3.8-flash",
            contents=prompt,
            config=types.GenerateContentConfig(temperature=0.1)
        )

        print(f"🧠 [Critic Agent Reasoning]:\n{response.text}")

        if "rerun" in response.text.lower() or "process_scaleup" in response.text.lower():
            mode = "process_scaleup" if "process_scaleup" in response.text.lower() else "green_safe"
            exclude = "Stille, Palladium" if "stille" in response.text.lower() else ""
            print(f"⚙️ [Optimization Agent] Executing automated parameter adjustment: mode={mode}, exclude={exclude}")

            tool_rerun_retrosynthesis(
                ranking_mode=mode,
                max_transforms=6,
                exclude_reaction_keyword=exclude
            )
            time.sleep(1)
        else:
            print("✅ [Multi-Agent System] Agent accepted current optimal path.")
            break

    return "Autonomous multi-agent optimization cycle completed successfully."

print("Autonomous multi-agent controller registered. Call run_autonomous_retrosynthesis_loop() to initiate.")

In [ ]:
# @title
run_autonomous_retrosynthesis_loop(
    max_agent_iterations=3,
    min_target_safety_score=3.5,
    min_composite_score=0.50
)